# When Transparency Works: Service Shoppability, Contracting Depth, and the Price Effects of Hospital Disclosure

**Replication code -- control and instrument construction stage**

Danny Sierra
Department of Economics, Florida State University
Ds22c@fsu.edu

---

## What this notebook does

This is stage 2 of a three-stage pipeline. Stage 1 (Snowflake SQL,
`HPT_Phase1_Rebuilt.sql` and `HPT_Phase2to4_Rebuilt.sql`) parses hospital
machine-readable files into a cleaned price panel and builds the clinical
codebook. This notebook, stage 2, constructs the treatment, the instruments,
and every control variable, then writes the analysis-ready panels. Stage 3
(`HPT_Analysis_Pipeline.R`) does all estimation.

Nothing here repeats work done in SQL. Raw price cleaning, payer-cell
construction, exact-code trimming, and concept/family aggregation all happen
upstream. This notebook starts from the finished Snowflake exports.

## Inputs

Snowflake exports, read from `Data/data_clean/Snowflake_Exports`:

| Folder | Contents |
|---|---|
| `Data/exact_standalone` | Standalone billing codes, exact-code level |
| `Data/exact_component` | Component billing codes, exact-code level |
| `Data/concept` | Prices aggregated to the clinical concept |
| `Data/family` | Prices aggregated to the clinical family |
| `QA/` | Three phase QA summaries, the codebook, the hospital directory |

External control files, located by search under `Data/data_raw/External_Controls`:
IPUMS ACS extract, hospital ownership counts, and CMS enforcement actions.

Expected row counts are read from each phase's QA table at load time rather
than hardcoded, so a rerun of the SQL pipeline cannot silently invalidate a
stale constant.

## Design decisions

**Four data exports, not six.** Billing codes are either STANDALONE or
COMPONENT. There is no MAIN / ROBUSTNESS_ONLY / COMPONENT_ONLY three-way
split, and no separate STRICT-tier table -- `STRICT_TIER_FLAG` is a column on
the exact-code rows, so that subset is a filter rather than a parallel table.

**Shoppability schemes are built here, not in SQL.** Snowflake exports the
raw, objective ingredients a shoppability definition needs on
`HPT_CODEBOOK.csv`: OPPS status indicator, ASC eligibility, CMS-70 membership,
DRG acuity tag, contrast variant, add-on flag. Schemes are constructed
explicitly in Section 11 as functions of those columns. This is deliberate --
it means competing definitions can be tested against each other without
rebuilding the SQL pipeline, which is what the shoppability robustness in the
paper requires.

**Posting time is a disclosure cohort, not a panel month.** Hospitals
generally disclose once. `POST_MONTH` is therefore an observed disclosure
cohort, and no observation is carried forward. Strictly prior posters are
hospitals whose relevant service entry occurred before the focal cohort;
same-month co-posters are counted separately. This is the structural fact that
rules out an event study at the estimation stage.

**Execution is disk-backed and low-memory.** The wide exact-code, concept, and
family panels are never held in memory simultaneously where that can be
avoided. Section 12 caches each panel to Parquet, builds narrow lookups,
releases the wide panel, then enriches one billing-code domain at a time. The
temporary cache is written to `~/Library/Caches/HPT_Python_Working_Panels` so
that multi-gigabyte intermediates are not synced to cloud storage.

## Outputs

Written under `Data/data_final`:

| Folder | Contents |
|---|---|
| `01_R_Analysis_Panels` | Analysis-ready panels consumed by the R script |
| `02_Treatments_and_Instruments` | Prior-poster measures and system instruments |
| `03_Controls` | ACS, ownership, and enforcement controls |
| `04_Coverage` | County and population coverage |
| `05_QA_and_Dictionaries` | QA results and variable dictionaries |

The four primary panels are `HPT_R_MAIN_PRIMARY_COUNTY_*`, at outpatient
exact-code, inpatient DRG, and concept levels. City and CBSA geographic
variants are written alongside them for robustness.

## How to run

Set `PROJECT_ROOT` in Section 0, then run the cells in order. The optional
blocks in Section 0 (`BUILD_ACS_CONTROLS`, `BUILD_OWNERSHIP_CONTROLS`,
`BUILD_ENFORCEMENT_CONTROLS`, `BUILD_COVERAGE_MAP`,
`BUILD_SERVICE_SPECIFIC_SYSTEM_IV`, `WRITE_ALL_R_GEOGRAPHY_DATASETS`) toggle
the expensive sections.

Section 15 runs a final QA pass over every written panel. It must report zero
FAIL rows before the R stage is run.

## Requirements

Python 3.11 or later, with `pandas`, `numpy`, and `pyarrow`. The coverage map
in Section 14 additionally needs `geopandas` and `matplotlib`, and is skipped
when `BUILD_COVERAGE_MAP` is off.

## Note on paths

`PROJECT_ROOT` in Section 0 is an absolute local path and must be edited
before this notebook will run anywhere else. Outputs have been cleared from
this copy so the committed file stays diff-friendly and carries no local
filesystem paths.


In [ ]:
from __future__ import annotations

import gc
import json
import math
import re
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Mapping, Sequence

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.options.mode.copy_on_write = True

In [ ]:
# =============================================================================
# 0. USER CONFIGURATION
# =============================================================================
#
# Everything configurable lives in this cell: paths, the study window, the
# optional-block switches, and the export registry. Nothing below hard-codes a
# directory or a filename, so relocating the project should only require
# editing PROJECT_ROOT.

PROJECT_ROOT = Path(
    "/Users/danielsierra/Library/CloudStorage/OneDrive-FloridaStateUniversity/"
    "Hospital Price Transparency Paper"
)
DATA_DIR = PROJECT_ROOT / "Data"

# Folder created after downloading HPT_PY_EXPORT_STAGE from Snowflake.
SNOWFLAKE_ROOT = DATA_DIR / "data_clean" / "Snowflake_Exports"
EXPORT_QA_DIR = SNOWFLAKE_ROOT / "QA"
EXPORT_DATA_DIR = SNOWFLAKE_ROOT / "Data"

# Use a new output folder so any previous Python pipeline output is untouched.
OUTPUT_DIR = DATA_DIR / "data_final"

R_PANEL_DIR = OUTPUT_DIR / "01_R_Analysis_Panels"
INSTRUMENT_DIR = OUTPUT_DIR / "02_Treatments_and_Instruments"
CONTROL_DIR = OUTPUT_DIR / "03_Controls"
COVERAGE_DIR = OUTPUT_DIR / "04_Coverage"
QA_DIR = OUTPUT_DIR / "05_QA_and_Dictionaries"
SHAPEFILE_DIR = OUTPUT_DIR / "_Cached_Shapefiles"

for folder in [
    OUTPUT_DIR, R_PANEL_DIR, INSTRUMENT_DIR, CONTROL_DIR,
    COVERAGE_DIR, QA_DIR, SHAPEFILE_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

# Search roots for the external controls files (ACS, ownership, enforcement).
RAW_DIR = DATA_DIR / "data_raw" / "External_Controls"
SEARCH_DIRS_RAW = [RAW_DIR, DATA_DIR, PROJECT_ROOT]

# Effective observed posting window (matches HPT_P1_STUDY_WINDOW in Snowflake).
STUDY_START = pd.Timestamp("2024-07-01")
STUDY_END = pd.Timestamp("2025-10-01")

# Optional blocks -- unchanged from the original notebook.
BUILD_SERVICE_SPECIFIC_SYSTEM_IV = False
WRITE_ALL_R_GEOGRAPHY_DATASETS = True
BUILD_ACS_CONTROLS = True
BUILD_OWNERSHIP_CONTROLS = True
BUILD_ENFORCEMENT_CONTROLS = True
BUILD_COVERAGE_MAP = True

# Canonical system-instrument timing -- unchanged.
CANONICAL_LOOKBACK_MONTHS = 9
SUPPORTING_WINDOWS = (3, 6, 9, 12)

# Four data exports. Billing codes are either STANDALONE or COMPONENT, so
# there is no MAIN/ROBUSTNESS_ONLY split, and there is no separate STRICT-tier
# table: STRICT_TIER_FLAG is a column on the exact-code rows, which makes that
# subset a filter rather than a parallel table to load.
DATASET_FOLDER_CANDIDATES = {
    "exact_standalone": ["exact_standalone", "Exact_Standalone"],
    "exact_component": ["exact_component", "Exact_Component"],
    "concept": ["concept", "Concept"],
    "family": ["family", "Family"],
}

# QA files: three small Snowflake QA summaries, the codebook, and the hospital
# directory. There is no shoppability-scheme file among them. Schemes are
# constructed in Section 11 from the raw, objective attributes carried on
# HPT_CODEBOOK.csv -- OPPS status indicator, ASC eligibility, CMS-70
# membership, DRG acuity tag, contrast variant -- rather than joined from a
# pre-baked table. That is what allows competing shoppability definitions to be
# tested against each other without rebuilding the SQL pipeline.
QA_FILES = {
    "phase2_qa": "HPT_P2_QA_SUMMARY.csv",
    "phase3_qa": "HPT_P3_QA_SUMMARY.csv",
    "phase4_qa": "HPT_P4_QA_SUMMARY.csv",
    "codebook": "HPT_CODEBOOK.csv",
    "hospital_directory": "HPT_HOSPITAL_DIRECTORY.csv",
}

RAW_FILES = {
    "acs": "usa_00004.csv",
    "puma_cbsa": "puma_cbsa_crosswalk.csv",
    "ownership": "hospitalownership_raw.csv",
    "enforcement_cbsa": "hospitalenforcementwithcbsa.csv",
    "enforcement_county": "Hospital_Enforcement_CMS_with_county.csv",
}


In [ ]:
# =============================================================================
# 1. SHARED CONSTANTS AND HELPERS
# =============================================================================

from typing import Any, Iterable, Sequence

FIPS_TO_ABBREV = {
    "01": "AL", "02": "AK", "04": "AZ", "05": "AR", "06": "CA",
    "08": "CO", "09": "CT", "10": "DE", "11": "DC", "12": "FL",
    "13": "GA", "15": "HI", "16": "ID", "17": "IL", "18": "IN",
    "19": "IA", "20": "KS", "21": "KY", "22": "LA", "23": "ME",
    "24": "MD", "25": "MA", "26": "MI", "27": "MN", "28": "MS",
    "29": "MO", "30": "MT", "31": "NE", "32": "NV", "33": "NH",
    "34": "NJ", "35": "NM", "36": "NY", "37": "NC", "38": "ND",
    "39": "OH", "40": "OK", "41": "OR", "42": "PA", "44": "RI",
    "45": "SC", "46": "SD", "47": "TN", "48": "TX", "49": "UT",
    "50": "VT", "51": "VA", "53": "WA", "54": "WV", "55": "WI",
    "56": "WY",
}

ABBREV_TO_FIPS = {
    abbreviation: fips
    for fips, abbreviation in FIPS_TO_ABBREV.items()
}

VALID_STATE_FIPS = set(FIPS_TO_ABBREV)

DEMO_COLS = [
    "uninsured_rate",
    "medicaid_share",
    "employer_ins_share",
    "poverty_rate",
    "median_income",
    "log_median_income",
    "age65plus_share",
    "college_share",
    "black_share",
    "hispanic_share",
    "employment_rate",
    "homeowner_rate",
    "population",
]

OWNERSHIP_COLS = [
    "For-Profit",
    "Government",
    "Non-Profit",
    "Total_Hospitals",
    "Share_ForProfit",
    "Share_Government",
    "Share_NonProfit",
]

ENFORCEMENT_BASE_COLS = [
    "closure_notice",
    "fine",
    "warning",
]

ENFORCEMENT_ROLL_COLS = [
    "closure_notice",
    "fine",
    "warning",
    "any_enforcement",
    "any_fine_or_warning",
    "any_fine_or_closure",
    "any_warning_or_closure",
]


def msg(text: str) -> None:
    """Print a progress message immediately."""

    print(text, flush=True)


def require_columns(
    df: pd.DataFrame,
    columns: Iterable[str],
    label: str,
) -> None:
    """Raise an informative error when required columns are absent."""

    missing = [
        column
        for column in columns
        if column not in df.columns
    ]

    if missing:
        raise KeyError(
            f"{label} is missing required columns: {missing}"
        )


def normalize_column_names(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """Convert column names to uppercase snake-case identifiers."""

    out = df.copy()

    out.columns = [
        re.sub(
            r"[^A-Z0-9]+",
            "_",
            str(column).strip().upper(),
        ).strip("_")
        for column in out.columns
    ]

    return out


def normalize_text(
    value: object,
) -> str | None:
    """
    Normalize a scalar text value.

    Returns
    -------
    str | None
        Uppercase normalized text, or None when the value is missing.
    """

    if value is None or pd.isna(value):
        return None

    text = re.sub(
        r"\s+",
        " ",
        str(value).strip().upper(),
    )

    return text if text else None


def normalize_name_key(
    value: object,
) -> str | None:
    """
    Create a compact alphanumeric matching key from a name.
    """

    text = normalize_text(value)

    if text is None:
        return None

    key = re.sub(
        r"[^A-Z0-9]+",
        "",
        text,
    )

    return key if key else None


def normalize_county_name(
    value: object,
) -> str | None:
    """
    Normalize county-equivalent names for matching.

    Removes common suffixes such as COUNTY, PARISH, BOROUGH,
    CENSUS AREA, and MUNICIPALITY.
    """

    text = normalize_text(value)

    if text is None:
        return None

    text = re.sub(
        r"\bCITY AND BOROUGH\b$",
        "",
        text,
    ).strip()

    text = re.sub(
        r"\bCENSUS AREA\b$",
        "",
        text,
    ).strip()

    text = re.sub(
        r"\bMUNICIPALITY\b$",
        "",
        text,
    ).strip()

    text = re.sub(
        r"\bCOUNTY\b$",
        "",
        text,
    ).strip()

    text = re.sub(
        r"\bPARISH\b$",
        "",
        text,
    ).strip()

    text = re.sub(
        r"\bBOROUGH\b$",
        "",
        text,
    ).strip()

    text = re.sub(
        r"[^A-Z0-9 ]+",
        "",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text if text else None


def normalize_county_state_key(
    county_state: object,
    county: object,
    state: object,
) -> str | None:
    """
    Construct a standardized STATE|COUNTY geographic key.

    Accepts either:
    - an existing STATE|COUNTY value;
    - a COUNTY, STATE value;
    - or separate county and state values.
    """

    state_text = normalize_text(state)
    county_text = normalize_county_name(county)

    if county_state is not None and not pd.isna(county_state):
        raw = str(county_state).strip()

        if "|" in raw:
            left, right = raw.split("|", 1)

            state_text = normalize_text(left)
            county_text = normalize_county_name(right)

        elif "," in raw:
            left, right = raw.rsplit(",", 1)

            county_text = normalize_county_name(left)
            state_text = normalize_text(right)

    if state_text is None or county_text is None:
        return None

    return f"{state_text}|{county_text}"


def normalize_cbsa(
    value: object,
) -> str | None:
    """
    Normalize a CBSA code as a five-character string.

    Returns None for missing values and the placeholder code 99999.
    """

    if value is None or pd.isna(value):
        return None

    # Values imported from CSV may appear as 35620.0. Converting through
    # float and then int prevents the decimal zero from becoming part
    # of the code.
    try:
        numeric_value = int(float(str(value).strip()))
    except (TypeError, ValueError, OverflowError):
        return None

    if numeric_value < 0:
        return None

    normalized = str(numeric_value).zfill(5)

    if normalized == "99999":
        return None

    if len(normalized) != 5:
        return None

    return normalized


def first_nonmissing(
    series: pd.Series,
) -> Any | None:
    """Return the first nonmissing value in a Series."""

    nonmissing = series.dropna()

    if nonmissing.empty:
        return None

    return nonmissing.iloc[0]


def mode_nonmissing(
    series: pd.Series,
) -> Any | None:
    """Return the modal nonmissing value in a Series."""

    nonmissing = series.dropna()

    if nonmissing.empty:
        return None

    mode_values = nonmissing.mode(dropna=True)

    if not mode_values.empty:
        return mode_values.iloc[0]

    return nonmissing.iloc[0]


def weighted_mean(
    values: pd.Series,
    weights: pd.Series,
) -> float:
    """Calculate a weighted mean after excluding invalid observations."""

    value_array = pd.to_numeric(
        values,
        errors="coerce",
    ).to_numpy(dtype=float)

    weight_array = pd.to_numeric(
        weights,
        errors="coerce",
    ).to_numpy(dtype=float)

    keep = (
        np.isfinite(value_array)
        & np.isfinite(weight_array)
        & (weight_array > 0)
    )

    if not keep.any():
        return np.nan

    return float(
        np.average(
            value_array[keep],
            weights=weight_array[keep],
        )
    )


def weighted_median(
    values: pd.Series,
    weights: pd.Series,
) -> float:
    """
    Calculate a memory-safe weighted median.

    This does not repeat observations according to PERWT.
    """

    value_array = pd.to_numeric(
        values,
        errors="coerce",
    ).to_numpy(dtype=float)

    weight_array = pd.to_numeric(
        weights,
        errors="coerce",
    ).to_numpy(dtype=float)

    keep = (
        np.isfinite(value_array)
        & np.isfinite(weight_array)
        & (weight_array > 0)
    )

    if not keep.any():
        return np.nan

    value_array = value_array[keep]
    weight_array = weight_array[keep]

    order = np.argsort(
        value_array,
        kind="mergesort",
    )

    value_array = value_array[order]
    weight_array = weight_array[order]

    cumulative_weight = np.cumsum(weight_array)
    cutoff = 0.5 * weight_array.sum()

    median_index = np.searchsorted(
        cumulative_weight,
        cutoff,
        side="left",
    )

    return float(value_array[median_index])


def find_file(
    filename: str,
    preferred_dirs: Sequence[Path],
) -> Path:
    """
    Find a file recursively within one or more preferred directories.
    """

    candidates: list[Path] = []

    for directory in preferred_dirs:
        if not directory.exists():
            continue

        exact_path = directory / filename

        if exact_path.exists():
            candidates.append(exact_path)

        candidates.extend(
            directory.rglob(filename)
        )

    unique_paths = sorted(
        {
            path.resolve()
            for path in candidates
        }
    )

    if not unique_paths:
        searched = ", ".join(
            str(path)
            for path in preferred_dirs
        )

        raise FileNotFoundError(
            f"Could not locate {filename!r}. "
            f"Searched: {searched}"
        )

    if len(unique_paths) > 1:
        msg(
            f"WARNING: found multiple copies of {filename}; "
            f"using {unique_paths[0]}"
        )

    return unique_paths[0]



def find_latest_downloaded_file(
    filename: str,
    preferred_dirs: Sequence[Path],
) -> Path:
    """
    Find the newest matching downloaded file.

    macOS and browsers commonly append suffixes such as `` (1)`` or `` (2)``
    when a file is downloaded more than once. This resolver treats those names
    as versions of the requested file and selects the most recently modified
    copy.

    Examples
    --------
    HPT_PHASE4_QA_SUMMARY.csv
    HPT_PHASE4_QA_SUMMARY (1).csv
    HPT_PHASE4_QA_SUMMARY (2).csv
    """

    requested = Path(filename)
    requested_suffixes = "".join(requested.suffixes).lower()
    requested_stem = requested.name[
        : -len(requested_suffixes)
    ] if requested_suffixes else requested.name

    duplicate_pattern = re.compile(
        rf"^{re.escape(requested_stem)}(?: \(\d+\))?"
        rf"{re.escape(requested_suffixes)}$",
        flags=re.IGNORECASE,
    )

    candidates: list[Path] = []

    for directory in preferred_dirs:
        if not directory.exists():
            continue

        for path in directory.rglob("*"):
            if (
                path.is_file()
                and duplicate_pattern.match(path.name)
            ):
                candidates.append(path.resolve())

    unique_paths = sorted(set(candidates))

    if not unique_paths:
        searched = ", ".join(str(path) for path in preferred_dirs)
        raise FileNotFoundError(
            f"Could not locate a current copy of {filename!r}. "
            f"Searched: {searched}"
        )

    selected = max(
        unique_paths,
        key=lambda path: (
            path.stat().st_mtime_ns,
            path.stat().st_size,
            str(path),
        ),
    )

    if len(unique_paths) > 1:
        msg(
            f"Found {len(unique_paths)} versions of {filename}; "
            f"using newest: {selected.name}"
        )

    return selected


def dataframe_scalar_if_present(
    df: pd.DataFrame,
    column: str,
) -> object | None:
    """Return the first nonmissing scalar from a column when it exists."""

    if column not in df.columns or df.empty:
        return None

    values = df[column].dropna()

    if values.empty:
        return None

    return values.iloc[0]


def inventory_row_count(
    inventory: pd.DataFrame,
    table_names: Sequence[str],
) -> int | None:
    """Return N_ROWS for the first matching final-table inventory entry."""

    required = {"TABLE_NAME", "N_ROWS"}

    if not required.issubset(inventory.columns):
        return None

    names = {
        str(name).strip().upper()
        for name in table_names
    }

    matches = inventory.loc[
        inventory["TABLE_NAME"]
        .astype("string")
        .str.strip()
        .str.upper()
        .isin(names)
    ]

    if matches.empty:
        return None

    value = pd.to_numeric(
        matches.iloc[0]["N_ROWS"],
        errors="coerce",
    )

    if pd.isna(value):
        return None

    return int(value)


def resolve_expected_row_count(
    *,
    dataset: str,
    phase4_qa: pd.DataFrame,
    phase4_inventory: pd.DataFrame,
    qa_column: str,
    inventory_table_names: Sequence[str],
    validated_fallback: int,
) -> tuple[int, str]:
    """
    Resolve an expected row count from the strongest available source.

    Priority
    --------
    1. Current Phase 4 QA column.
    2. Current final-table inventory.
    3. Validated counts from the final PASS Phase 4 run.
    """

    qa_value = dataframe_scalar_if_present(
        phase4_qa,
        qa_column,
    )

    if qa_value is not None:
        numeric = pd.to_numeric(
            pd.Series([qa_value]),
            errors="coerce",
        ).iloc[0]

        if not pd.isna(numeric):
            return int(numeric), f"Phase 4 QA column {qa_column}"

    inventory_value = inventory_row_count(
        phase4_inventory,
        inventory_table_names,
    )

    if inventory_value is not None:
        return (
            inventory_value,
            "Phase 4 final-table inventory",
        )

    return (
        int(validated_fallback),
        "validated final PASS Phase 4 count",
    )

def optional_file(
    filename: str,
    preferred_dirs: Sequence[Path],
) -> Path | None:
    """Find an optional file, returning None when it is absent."""

    try:
        return find_file(
            filename,
            preferred_dirs,
        )

    except FileNotFoundError:
        return None


def read_csv_file(
    path: Path,
    *,
    usecols: Sequence[str] | None = None,
) -> pd.DataFrame:
    """Read a CSV or compressed CSV and normalize its column names."""

    msg(f"Reading {path.name} ...")

    df = pd.read_csv(
        path,
        low_memory=False,
        usecols=usecols,
    )

    df = normalize_column_names(df)

    msg(
        f"  {len(df):,} rows × "
        f"{df.shape[1]} columns"
    )

    return df


def write_parquet(
    df: pd.DataFrame,
    path: Path,
) -> None:
    """Write a DataFrame as a compressed Parquet file."""

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    try:
        df.to_parquet(
            path,
            index=False,
            compression="zstd",
        )

    except ImportError as exc:
        raise ImportError(
            "Install pyarrow before writing Parquet: "
            "pip install pyarrow"
        ) from exc

    msg(
        f"Saved {path.name}: "
        f"{len(df):,} rows × "
        f"{df.shape[1]} columns"
    )


def month_start(
    series: pd.Series,
) -> pd.Series:
    """Convert values to the first day of their calendar month."""

    return (
        pd.to_datetime(
            series,
            errors="coerce",
        )
        .dt.to_period("M")
        .dt.to_timestamp()
    )


def unique_key_qa(
    df: pd.DataFrame,
    keys: Sequence[str],
    label: str,
) -> dict[str, object]:
    """Create a QA result for a proposed unique key."""

    duplicate_rows = int(
        df.duplicated(
            list(keys),
            keep=False,
        ).sum()
    )

    return {
        "check": f"{label}_unique_key",
        "status": (
            "PASS"
            if duplicate_rows == 0
            else "FAIL"
        ),
        "value": duplicate_rows,
        "expected": 0,
        "details": " | ".join(keys),
    }


def safe_left_merge(
    left: pd.DataFrame,
    right: pd.DataFrame,
    on: Sequence[str],
    label: str,
    validate: str = "m:1",
) -> pd.DataFrame:
    """
    Perform a left merge and verify that the row count does not change.
    """

    n_before = len(left)

    out = left.merge(
        right,
        on=list(on),
        how="left",
        validate=validate,
    )

    if len(out) != n_before:
        raise RuntimeError(
            f"{label} changed row count: "
            f"{n_before:,} -> {len(out):,}"
        )

    return out

# =============================================================================
# PHASE 4 SHARD AND SCHEMA HELPERS
# =============================================================================

def _path_key(value: str | Path) -> str:
    """Normalize a path or folder name for tolerant matching."""

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower(),
    )


def resolve_dataset_directory(
    root: Path,
    candidates: Sequence[str],
    label: str,
) -> Path:
    """
    Resolve a downloaded Snowflake dataset folder.

    Exact candidate paths are preferred. If none exists, a recursive normalized
    folder-name match is used.
    """

    for candidate in candidates:
        path = root / candidate
        if path.exists() and path.is_dir():
            return path

    wanted = {
        _path_key(Path(candidate).name)
        for candidate in candidates
    }

    matches = [
        path
        for path in root.rglob("*")
        if path.is_dir()
        and _path_key(path.name) in wanted
    ]

    if not matches:
        raise FileNotFoundError(
            f"Could not locate the {label} shard folder under {root}. "
            f"Tried: {list(candidates)}"
        )

    matches = sorted(
        {path.resolve() for path in matches},
        key=lambda p: (len(p.parts), str(p)),
    )

    if len(matches) > 1:
        msg(
            f"WARNING: multiple folders matched {label}; "
            f"using {matches[0]}"
        )

    return matches[0]


def list_csv_shards(folder: Path) -> list[Path]:
    """Return all CSV or CSV.GZ shards in deterministic order."""

    files = sorted(
        {
            *folder.glob("*.csv"),
            *folder.glob("*.csv.gz"),
            *folder.glob("*.CSV"),
            *folder.glob("*.CSV.GZ"),
        }
    )

    if not files:
        # Snowflake downloads may retain an extra nested directory.
        files = sorted(
            {
                *folder.rglob("*.csv"),
                *folder.rglob("*.csv.gz"),
                *folder.rglob("*.CSV"),
                *folder.rglob("*.CSV.GZ"),
            }
        )

    if not files:
        raise FileNotFoundError(
            f"No CSV or CSV.GZ shards found under {folder}"
        )

    return files


def read_sharded_csv(
    folder: Path,
    label: str,
    *,
    usecols: Sequence[str] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Read and concatenate every Snowflake CSV shard in a folder.

    Returns
    -------
    data
        Concatenated DataFrame.

    manifest
        One row per physical shard with row and file-size metadata.
    """

    shard_paths = list_csv_shards(folder)
    pieces: list[pd.DataFrame] = []
    manifest_rows: list[dict[str, object]] = []
    reference_columns: list[str] | None = None

    msg(
        f"\nReading {label}: "
        f"{len(shard_paths)} shard(s) from {folder}"
    )

    for shard_number, path in enumerate(shard_paths, start=1):
        piece = read_csv_file(
            path,
            usecols=usecols,
        )

        columns = list(piece.columns)

        if reference_columns is None:
            reference_columns = columns
        elif columns != reference_columns:
            missing = sorted(set(reference_columns) - set(columns))
            extra = sorted(set(columns) - set(reference_columns))
            raise RuntimeError(
                f"{label} shard schemas differ. "
                f"File={path.name}; missing={missing}; extra={extra}"
            )

        manifest_rows.append({
            "dataset": label,
            "shard_number": shard_number,
            "file_name": path.name,
            "full_path": str(path),
            "size_bytes": path.stat().st_size,
            "n_rows": len(piece),
            "n_columns": piece.shape[1],
        })

        pieces.append(piece)

    data = pd.concat(
        pieces,
        ignore_index=True,
        sort=False,
        copy=False,
    )

    manifest = pd.DataFrame(manifest_rows)

    msg(
        f"Combined {label}: "
        f"{len(data):,} rows × {data.shape[1]} columns"
    )

    return data, manifest


def normalize_phase4_frame(
    df: pd.DataFrame,
    label: str,
) -> pd.DataFrame:
    """Normalize common identifiers and date fields in a Phase 4 export."""

    out = normalize_column_names(df)

    for column in [
        "POST_MONTH",
        "HOSPITAL_FIRST_POST_MONTH",
        "HOSPITAL_LAST_POST_MONTH",
        "COUNTY_FIRST_POST_MONTH",
        "CBSA_FIRST_POST_MONTH",
        "INGESTED_ON",
        "LAST_LOADED_ON",
        "FINAL_DATA_CREATED_AT",
    ]:
        if column in out.columns:
            out[column] = pd.to_datetime(
                out[column],
                errors="coerce",
            )

    for column in [
        "HOSPITAL_ID",
        "PROVIDER_NPI",
        "HEALTH_SYSTEM_ID",
        "BILLING_CODE_TYPE",
        "BILLING_CODE",
        "ANALYSIS_SUPERFAMILY_ID",
        "ANALYSIS_FAMILY_ID",
        "ANALYSIS_CONCEPT_ID",
        "SOURCE_ANALYSIS_CONCEPT_ID",
        "CMS70_SERVICE_ID",
    ]:
        if column in out.columns:
            out[column] = (
                out[column]
                .astype("string")
                .str.strip()
            )

    if "BILLING_CODE" in out.columns:
        out["BILLING_CODE"] = (
            out["BILLING_CODE"]
            .str.upper()
        )

    if "PROVIDER_STATE" in out.columns:
        out["PROVIDER_STATE"] = (
            out["PROVIDER_STATE"]
            .astype("string")
            .str.upper()
            .str.strip()
        )

    if "CBSA_CODE" in out.columns:
        out["CBSA_CODE"] = (
            out["CBSA_CODE"]
            .map(normalize_cbsa)
            .astype("string")
        )

    if "PROVIDER_CITY" in out.columns and "PROVIDER_STATE" in out.columns:
        out["CITY_STATE_KEY"] = (
            out["PROVIDER_STATE"].fillna("")
            + "|"
            + out["PROVIDER_CITY"]
                .map(normalize_text)
                .astype("string")
                .fillna("")
        ).replace({"|": pd.NA})

    if {
        "COUNTY_STATE",
        "COUNTY",
        "PROVIDER_STATE",
    }.issubset(out.columns):
        out["COUNTY_STATE_KEY"] = [
            normalize_county_state_key(
                county_state,
                county,
                state,
            )
            for county_state, county, state in zip(
                out["COUNTY_STATE"],
                out["COUNTY"],
                out["PROVIDER_STATE"],
            )
        ]

    msg(
        f"Normalized {label}: "
        f"{len(out):,} rows"
    )

    return out


def add_legacy_analysis_aliases(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add backward-compatible FINAL_* names used by the existing R code.

    The canonical fields remain ANALYSIS_*.
    """

    out = df.copy()

    alias_pairs = {
        "FINAL_SUPERFAMILY_ID": "ANALYSIS_SUPERFAMILY_ID",
        "FINAL_FAMILY_ID": "ANALYSIS_FAMILY_ID",
        "FINAL_FAMILY_NAME": "ANALYSIS_FAMILY_NAME",
        "FINAL_CONCEPT_ID": "ANALYSIS_CONCEPT_ID",
        "FINAL_CONCEPT_NAME": "ANALYSIS_CONCEPT_NAME",
    }

    for alias, source in alias_pairs.items():
        if source in out.columns:
            out[alias] = out[source]

    return out


def phase4_concept_key_columns() -> list[str]:
    """Complete Phase 4 concept identifier, excluding hospital and month."""

    return [
        "BILLING_CODE_TYPE",
        "FINAL_SUPERFAMILY_ID",
        "FINAL_FAMILY_ID",
        "FINAL_CONCEPT_ID",
    ]


def phase4_family_key_columns() -> list[str]:
    """Complete Phase 4 family identifier, excluding hospital and month."""

    return [
        "BILLING_CODE_TYPE",
        "FINAL_SUPERFAMILY_ID",
        "FINAL_FAMILY_ID",
    ]


def build_composite_service_id(
    df: pd.DataFrame,
    columns: Sequence[str],
) -> pd.Series:
    """Build a stable composite service key without collapsing hierarchy levels."""

    missing = [
        column
        for column in columns
        if column not in df.columns
    ]

    if missing:
        raise KeyError(
            "Cannot build composite service ID; "
            f"missing columns: {missing}"
        )

    result = df[columns[0]].astype("string")

    for column in columns[1:]:
        result = (
            result
            + "::"
            + df[column].astype("string")
        )

    return result


def qa_expected_value(
    qa: pd.DataFrame,
    column: str,
) -> int | float | str | None:
    """Return a scalar from the one-row Phase 4 QA table."""

    if qa.empty or column not in qa.columns:
        return None

    value = qa.iloc[0][column]

    if pd.isna(value):
        return None

    return value


# Override the earlier CSV reader so identifier fields retain leading zeros.
CSV_IDENTIFIER_COLUMNS = {
    "HOSPITAL_ID",
    "PROVIDER_NPI",
    "HEALTH_SYSTEM_ID",
    "FILE_ID",
    "RAW_FILE_KEY",
    "BILLING_CODE",
    "CBSA_CODE",
    "COUNTY_FIPS",
    "ANALYSIS_SUPERFAMILY_ID",
    "ANALYSIS_FAMILY_ID",
    "ANALYSIS_CONCEPT_ID",
    "SOURCE_ANALYSIS_CONCEPT_ID",
    "CMS70_SERVICE_ID",
    "COMPONENT_BILLING_CODE",
    "DIRECT_PARENT_BILLING_CODE",
    "PARENT_BILLING_CODE",
    "SCHEME_ID",
}


def read_csv_file(
    path: Path,
    *,
    usecols: Sequence[str] | None = None,
) -> pd.DataFrame:
    """
    Read a CSV or CSV.GZ while preserving identifier leading zeros.

    Snowflake outputs include numeric-looking CPT/HCPCS and MS-DRG identifiers.
    Reading those columns as numbers would turn values such as 001 into 1.
    """

    msg(f"Reading {path.name} ...")

    header = pd.read_csv(
        path,
        nrows=0,
    )

    header_columns = [
        re.sub(
            r"[^A-Z0-9]+",
            "_",
            str(column).strip().upper(),
        ).strip("_")
        for column in header.columns
    ]

    dtype_map = {
        original: "string"
        for original, normalized in zip(
            header.columns,
            header_columns,
        )
        if normalized in CSV_IDENTIFIER_COLUMNS
    }

    df = pd.read_csv(
        path,
        low_memory=False,
        usecols=usecols,
        dtype=dtype_map,
    )

    df = normalize_column_names(df)

    msg(
        f"  {len(df):,} rows × "
        f"{df.shape[1]} columns"
    )

    return df


In [ ]:
# =============================================================================
# 2. LOAD, RECONCILE, AND NORMALIZE FINAL SNOWFLAKE EXPORTS
# =============================================================================
#
# Loads the four data folders and the QA files, attaches the hospital
# directory, and reconciles row counts against the source.
#
# Expected row counts are read directly from each phase's QA table rather than
# stored as constants here. A hardcoded count dictionary goes stale the moment
# the SQL pipeline is rerun, and it fails silently when it does -- reading the
# QA table means a mismatch is always a real mismatch.

if not SNOWFLAKE_ROOT.exists():
    raise FileNotFoundError(
        "The Snowflake export folder was not found:\n"
        f"  {SNOWFLAKE_ROOT}\n"
        "Confirm HPT_PY_EXPORT_STAGE was downloaded there (QA/ and Data/ subfolders)."
    )

# -----------------------------------------------------------------------------
# 2A. QA files and codebook
# -----------------------------------------------------------------------------

qa_paths = {
    name: find_latest_downloaded_file(filename, [EXPORT_QA_DIR, SNOWFLAKE_ROOT])
    for name, filename in QA_FILES.items()
}

msg("\nSelected Snowflake QA exports")
for name, path in qa_paths.items():
    msg(f"  {name:<12}: {path}")

phase2_qa = read_csv_file(qa_paths["phase2_qa"])
phase3_qa = read_csv_file(qa_paths["phase3_qa"])
phase4_qa = read_csv_file(qa_paths["phase4_qa"])
codebook = read_csv_file(qa_paths["codebook"])
hospital_directory = read_csv_file(qa_paths["hospital_directory"])

for qa_frame, status_column, label in [
    (phase2_qa, "PHASE2_QA_STATUS", "Phase 2"),
    (phase3_qa, "PHASE3_QA_STATUS", "Phase 3"),
    (phase4_qa, "PHASE4_QA_STATUS", "Phase 4"),
]:
    require_columns(qa_frame, [status_column], f"{label} QA")
    status = str(qa_frame.iloc[0][status_column]).upper()
    if status != "PASS":
        raise RuntimeError(
            f"{status_column} is {status}, not PASS. "
            f"Do not build Python analysis panels from a non-passing {label} export."
        )

msg("All three Snowflake QA gates read PASS.")

codebook = normalize_column_names(codebook)
hospital_directory = normalize_phase4_frame(hospital_directory, "hospital directory")
require_columns(
    hospital_directory,
    ["HOSPITAL_ID", "PROVIDER_NAME", "PROVIDER_CITY", "PROVIDER_STATE",
     "COUNTY", "COUNTY_STATE", "CBSA_CODE", "HOSPITAL_TYPE",
     "HEALTH_SYSTEM_ID", "HEALTH_SYSTEM_NAME", "TOTAL_BEDS"],
    "hospital directory",
)
qa_rows_directory_check = unique_key_qa(hospital_directory, ["HOSPITAL_ID"], "hospital_directory")

# -----------------------------------------------------------------------------
# 2B. Resolve and read the four shard folders
# -----------------------------------------------------------------------------

dataset_dirs = {
    name: resolve_dataset_directory(EXPORT_DATA_DIR, candidates, label=name)
    for name, candidates in DATASET_FOLDER_CANDIDATES.items()
}

exact_standalone, manifest_exact_standalone = read_sharded_csv(
    dataset_dirs["exact_standalone"], "exact_standalone",
)
exact_component, manifest_exact_component = read_sharded_csv(
    dataset_dirs["exact_component"], "exact_component",
)
concept_prices, manifest_concept = read_sharded_csv(
    dataset_dirs["concept"], "concept",
)
family_prices, manifest_family = read_sharded_csv(
    dataset_dirs["family"], "family",
)

snowflake_shard_manifest = pd.concat(
    [manifest_exact_standalone, manifest_exact_component, manifest_concept, manifest_family],
    ignore_index=True, sort=False,
)
snowflake_shard_manifest.to_csv(
    QA_DIR / "HPT_DOWNLOADED_SHARD_MANIFEST.csv", index=False,
)

# -----------------------------------------------------------------------------
# 2C. Normalize schemas
# -----------------------------------------------------------------------------

exact_standalone = normalize_phase4_frame(exact_standalone, "standalone exact-code data")
exact_component = normalize_phase4_frame(exact_component, "component exact-code data")
concept_prices = normalize_phase4_frame(concept_prices, "concept data")
family_prices = normalize_phase4_frame(family_prices, "family data")

# -----------------------------------------------------------------------------
# 2C-2. Attach the hospital directory to every frame
#
# Each of the four tables carries a different, inconsistent subset of
# hospital-level attributes depending on what its Snowflake aggregation
# happened to select. Rather than patch each one individually as gaps turn
# up, every frame gets the complete, authoritative directory joined on
# HOSPITAL_ID -- any column already present is dropped first and replaced
# with the directory's version, since the directory is computed from every
# payer cell for that hospital (MIN/MEDIAN across the hospital's full row
# set), which is at least as reliable as a partial aggregate computed only
# from the rows relevant to one specific code, concept, or family.
# -----------------------------------------------------------------------------

def attach_hospital_directory(frame: pd.DataFrame, label: str) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()

    overlap = [
        column for column in hospital_directory.columns
        if column != "HOSPITAL_ID" and column in frame.columns
    ]
    base = frame.drop(columns=overlap, errors="ignore")

    return safe_left_merge(
        base, hospital_directory, on=["HOSPITAL_ID"], label=f"{label} hospital directory",
    )

exact_standalone = attach_hospital_directory(exact_standalone, "exact_standalone")
exact_component = attach_hospital_directory(exact_component, "exact_component")
concept_prices = attach_hospital_directory(concept_prices, "concept")
family_prices = attach_hospital_directory(family_prices, "family")

# Rebuild CITY_STATE_KEY / COUNTY_STATE_KEY now that PROVIDER_CITY / COUNTY /
# COUNTY_STATE are freshly available from the directory join -- these derived
# keys were already computed once inside normalize_phase4_frame above, but
# before the directory join, so they need recomputing with the join in place.
exact_standalone = normalize_phase4_frame(exact_standalone, "standalone exact-code data (post-directory)")
exact_component = normalize_phase4_frame(exact_component, "component exact-code data (post-directory)")
concept_prices = normalize_phase4_frame(concept_prices, "concept data (post-directory)")
family_prices = normalize_phase4_frame(family_prices, "family data (post-directory)")

msg("Hospital directory attached to all four frames.")

# -----------------------------------------------------------------------------
# 2D. Required fields and domain splits
# -----------------------------------------------------------------------------
# Column names below match the Snowflake schema exactly: ANALYSIS_* throughout,
# with no parallel FINAL_* naming convention.

require_columns(
    exact_standalone,
    [
        "HOSPITAL_CODE_MONTH_ID", "HOSPITAL_ID", "POST_MONTH",
        "BILLING_CODE_TYPE", "BILLING_CODE",
        "ANALYSIS_SUPERFAMILY_ID", "ANALYSIS_FAMILY_ID", "ANALYSIS_CONCEPT_ID",
        "CONTRAST_VARIANT", "MEDIAN_PRICE", "MIN_PRICE", "P25_PRICE",
        "P75_PRICE", "MAX_PRICE", "N_PAYER_CELLS", "N_DISTINCT_PAYERS",
    ],
    "standalone exact-code data",
)

require_columns(
    concept_prices,
    [
        "HOSPITAL_ID", "POST_MONTH", "BILLING_CODE_TYPE",
        "ANALYSIS_SUPERFAMILY_ID", "ANALYSIS_FAMILY_ID", "ANALYSIS_CONCEPT_ID",
        "N_CODES_IN_CONCEPT", "MEDIAN_PRICE",
    ],
    "concept data",
)

require_columns(
    family_prices,
    [
        "HOSPITAL_ID", "POST_MONTH", "BILLING_CODE_TYPE",
        "ANALYSIS_SUPERFAMILY_ID", "ANALYSIS_FAMILY_ID", "MEDIAN_PRICE",
    ],
    "family data",
)

outpatient_exact = exact_standalone.loc[
    exact_standalone["BILLING_CODE_TYPE"] == "CPT_HCPCS"
].copy()

inpatient_exact = exact_standalone.loc[
    exact_standalone["BILLING_CODE_TYPE"] == "MS_DRG"
].copy()

# Backward-compatible FINAL_* names for the parts of the notebook (e.g.
# phase4_concept_key_columns / phase4_family_key_columns, used in Sections 4
# and 11) that group on the old naming convention. The canonical columns
# remain ANALYSIS_* everywhere; these are aliases, not replacements. Note:
# ANALYSIS_CONCEPT_NAME does not exist in this pipeline's codebook (only
# ANALYSIS_FAMILY_NAME does), so FINAL_CONCEPT_NAME will not be created --
# harmless, since nothing in the concept/family key functions needs it.
outpatient_exact = add_legacy_analysis_aliases(outpatient_exact)
inpatient_exact = add_legacy_analysis_aliases(inpatient_exact)
exact_component = add_legacy_analysis_aliases(exact_component)
concept_prices = add_legacy_analysis_aliases(concept_prices)
family_prices = add_legacy_analysis_aliases(family_prices)

# -----------------------------------------------------------------------------
# 2E. Reconcile downloaded rows to Snowflake QA
#
# Expected counts come straight from each phase's QA table -- no hardcoded
# dictionary of "last known good" counts to keep in sync by hand.
# -----------------------------------------------------------------------------

expected_counts = {
    "exact_standalone": int(phase4_qa.iloc[0]["N_EXACT_CODE_ROWS"]),
    "concept": int(phase4_qa.iloc[0]["N_CONCEPT_ROWS"]),
    "family": int(phase4_qa.iloc[0]["N_FAMILY_ROWS"]),
    "exact_component": int(phase4_qa.iloc[0]["N_COMPONENT_ROWS"]),
}

actual_counts = {
    "exact_standalone": len(exact_standalone),
    "exact_component": len(exact_component),
    "concept": len(concept_prices),
    "family": len(family_prices),
}

qa_rows: list[dict[str, object]] = []
qa_rows.append(qa_rows_directory_check)

for dataset, expected in expected_counts.items():
    actual = actual_counts[dataset]
    qa_rows.append({
        "check": f"{dataset}_downloaded_row_count",
        "status": "PASS" if actual == expected else "FAIL",
        "value": actual,
        "expected": expected,
        "details": "Combined row count across every downloaded CSV.GZ shard",
    })
    if actual != expected:
        raise RuntimeError(
            f"{dataset} row count does not match Snowflake QA: "
            f"{actual:,} versus {expected:,}"
        )

qa_rows.append(
    unique_key_qa(exact_standalone, ["HOSPITAL_CODE_MONTH_ID"], "exact_standalone")
)

for frame, label in [
    (exact_standalone, "exact_standalone"),
    (concept_prices, "concept"),
    (family_prices, "family"),
]:
    # exact_standalone carries MIN/MAX (from Phase 3); concept and family
    # were built leaner (median/mean/P25/P75 only) -- check whatever price
    # columns each frame actually has, in ascending order, rather than
    # assuming all three frames share the same five columns.
    available_price_cols = [
        c for c in ["MIN_PRICE", "P25_PRICE", "MEDIAN_PRICE", "P75_PRICE", "MAX_PRICE"]
        if c in frame.columns
    ]
    require_columns(frame, available_price_cols, label)

    invalid_order = pd.Series(False, index=frame.index)
    for lo, hi in zip(available_price_cols[:-1], available_price_cols[1:]):
        invalid_order = invalid_order | (frame[lo] > frame[hi])

    qa_rows.append({
        "check": f"{label}_price_order",
        "status": "PASS" if int(invalid_order.sum()) == 0 else "FAIL",
        "value": int(invalid_order.sum()),
        "expected": 0,
        "details": f"Ascending order check across: {' <= '.join(available_price_cols)}",
    })

msg("\nFinal source summary")
for label, frame in [
    ("Outpatient exact (standalone)", outpatient_exact),
    ("Inpatient exact (standalone, DRG)", inpatient_exact),
    ("Component exact", exact_component),
    ("Concept", concept_prices),
    ("Family", family_prices),
]:
    if frame.empty:
        continue
    msg(f"  {label:<34}: {len(frame):>10,} rows; {frame['HOSPITAL_ID'].nunique():>5,} hospitals")

del exact_standalone
gc.collect()


In [ ]:
# =============================================================================
# 3. CENSUS COUNTY GEOGRAPHY FROM HOSPITAL COORDINATES
# =============================================================================

TIGER_COUNTY_URL = (
    "https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/"
    "tl_2023_us_county.zip"
)
TIGER_STATE_URL = (
    "https://www2.census.gov/geo/tiger/TIGER2023/STATE/"
    "tl_2023_us_state.zip"
)

COUNTY_ZIP = SHAPEFILE_DIR / "tl_2023_us_county.zip"
STATE_ZIP = SHAPEFILE_DIR / "tl_2023_us_state.zip"


def download_if_missing(
    url: str,
    output: Path,
) -> None:
    if output.exists():
        return

    try:
        import certifi
        import requests
    except ImportError as exc:
        raise ImportError(
            "Install requests and certifi to download Census shapefiles."
        ) from exc

    msg(f"Downloading {output.name} ...")

    response = requests.get(
        url,
        timeout=180,
        verify=certifi.where(),
    )
    response.raise_for_status()
    output.write_bytes(response.content)


def load_census_geographies():
    try:
        import geopandas as gpd
    except ImportError as exc:
        raise ImportError(
            "Install geopandas and shapely: "
            "pip install geopandas shapely"
        ) from exc

    download_if_missing(
        TIGER_COUNTY_URL,
        COUNTY_ZIP,
    )
    download_if_missing(
        TIGER_STATE_URL,
        STATE_ZIP,
    )

    counties = gpd.read_file(COUNTY_ZIP)
    states = gpd.read_file(STATE_ZIP)

    counties["STATE_ABBREV"] = (
        counties["STATEFP"]
        .map(FIPS_TO_ABBREV)
    )
    counties["COUNTY_NAME_KEY"] = (
        counties["NAME"]
        .map(normalize_county_name)
    )
    counties["COUNTY_STATE_KEY"] = (
        counties["STATE_ABBREV"]
        + "|"
        + counties["COUNTY_NAME_KEY"].astype("string")
    )
    counties["COUNTY_FIPS"] = (
        counties["GEOID"]
        .astype("string")
        .str.zfill(5)
    )

    return counties, states


counties_gdf, states_gdf = load_census_geographies()

# Use all exact-code roles so hospitals that only ever posted a component
# (add-on) code, and never a standalone code, are still retained.
location_frames = [
    outpatient_exact,
    inpatient_exact,
]

if not exact_component.empty:
    location_frames.append(exact_component)

# PROVIDER_CITY, LATITUDE, and LONGITUDE are not present on the exported
# exact-code tables. PROVIDER_CITY exists at the payer-cell stage but is not
# carried into the final price table, and coordinates are not pulled from the
# raw source at all -- both are upstream gaps rather than missing files here.
#
# The coordinate-based spatial join is therefore skipped and county assignment
# relies on name-based COUNTY_STATE_KEY matching, which is the same fallback
# path this cell already uses when a spatial join fails. Match rate is reported
# below and should be checked before proceeding.
location_columns = [
    "HOSPITAL_ID",
    "PROVIDER_NAME",
    "PROVIDER_CITY",
    "PROVIDER_STATE",
    "COUNTY_STATE_KEY",
    "CBSA_CODE",
]

location_source = pd.concat(
    [
        frame[location_columns]
        for frame in location_frames
    ],
    ignore_index=True,
    sort=False,
)

hospital_locations = (
    location_source
    .groupby("HOSPITAL_ID", as_index=False)
    .agg({
        "PROVIDER_NAME": mode_nonmissing,
        "PROVIDER_CITY": mode_nonmissing,
        "PROVIDER_STATE": mode_nonmissing,
        "COUNTY_STATE_KEY": mode_nonmissing,
        "CBSA_CODE": mode_nonmissing,
    })
)

# Coordinate spatial join skipped -- see note above. Name-match-only path.
spatial = pd.DataFrame(
    columns=[
        "HOSPITAL_ID",
        "COUNTY_FIPS",
        "COUNTY_STATE_KEY_CENSUS",
    ]
)

name_crosswalk = (
    counties_gdf[
        [
            "COUNTY_FIPS",
            "COUNTY_STATE_KEY",
        ]
    ]
    .drop_duplicates("COUNTY_STATE_KEY")
    .rename(
        columns={
            "COUNTY_FIPS":
                "COUNTY_FIPS_NAME",
            "COUNTY_STATE_KEY":
                "COUNTY_STATE_KEY_RAW",
        }
    )
)

hospital_locations = hospital_locations.merge(
    spatial,
    on="HOSPITAL_ID",
    how="left",
)

hospital_locations = hospital_locations.merge(
    name_crosswalk,
    left_on="COUNTY_STATE_KEY",
    right_on="COUNTY_STATE_KEY_RAW",
    how="left",
    suffixes=("", "_NAME"),
)

hospital_locations["COUNTY_FIPS"] = (
    hospital_locations["COUNTY_FIPS"]
    .fillna(
        hospital_locations["COUNTY_FIPS_NAME"]
    )
)

hospital_locations["COUNTY_GEOGRAPHY_METHOD"] = (
    np.select(
        [
            hospital_locations["COUNTY_FIPS"].notna()
            & hospital_locations[
                "COUNTY_STATE_KEY_CENSUS"
            ].notna(),

            hospital_locations["COUNTY_FIPS"].notna(),
        ],
        [
            "SPATIAL_COORDINATE",
            "NAME_MATCH",
        ],
        default="UNMATCHED",
    )
)

hospital_locations = hospital_locations.drop(
    columns=[
        "COUNTY_STATE_KEY_RAW",
        "COUNTY_FIPS_NAME",
    ],
    errors="ignore",
)

hospital_locations.to_csv(
    OUTPUT_DIR / "HPT_HOSPITAL_GEOGRAPHY_CROSSWALK.csv",
    index=False,
)


def attach_hospital_geography(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()

    base = frame.drop(
        columns=[
            "COUNTY_FIPS",
            "COUNTY_GEOGRAPHY_METHOD",
        ],
        errors="ignore",
    )

    return safe_left_merge(
        base,
        hospital_locations[
            [
                "HOSPITAL_ID",
                "COUNTY_FIPS",
                "COUNTY_GEOGRAPHY_METHOD",
            ]
        ],
        on=["HOSPITAL_ID"],
        label="hospital geography merge",
    )


for frame_name in [
    "outpatient_exact",
    "inpatient_exact",
    "concept_prices",
    "family_prices",
    "exact_component",
]:
    frame = globals().get(frame_name)

    if isinstance(frame, pd.DataFrame):
        globals()[frame_name] = (
            attach_hospital_geography(frame)
        )

matched_share = (
    hospital_locations["COUNTY_FIPS"]
    .notna()
    .mean()
)

qa_rows.append({
    "check": "hospital_county_fips_match",
    "status": (
        "PASS"
        if matched_share >= 0.98
        else "REVIEW"
    ),
    "value": float(matched_share),
    "expected": ">= 0.98",
    "details": (
        "Coordinate spatial join with name fallback; "
        "all exact-code roles (standalone and component) included"
    ),
})

msg(
    "Hospital county-FIPS match rate: "
    f"{matched_share:.2%}"
)


In [ ]:
# =============================================================================
# 4. HOSPITAL DISCLOSURE EVENTS AND SERVICE ENTRY FLAGS
# =============================================================================

# Two notes on available fields.
#
# LATITUDE and LONGITUDE are unavailable. The raw source carries only
# HQ_LATITUDE and HQ_LONGITUDE, which are headquarters rather than facility
# coordinates and are deliberately not used -- see the note in Section 3.
# PROVIDER_CITY, CITY_STATE_KEY, and HEALTH_SYSTEM_NAME are available on every
# frame through the hospital-directory join in Section 2.
#
# FILE_ID and INGESTED_ON are Phase 2 file-submission bookkeeping fields, used
# internally for canonical-file selection and deduplication. They are not
# hospital-level attributes and do not belong in the hospital directory.
# POST_MONTH already captures the disclosure timing the analysis needs, so they
# are dropped rather than carried forward.
EVENT_COLS = [
    "HOSPITAL_ID",
    "POST_MONTH",
    "PROVIDER_NPI",
    "PROVIDER_NAME",
    "PROVIDER_CITY",
    "PROVIDER_STATE",
    "CITY_STATE_KEY",
    "COUNTY_STATE_KEY",
    "COUNTY_FIPS",
    "CBSA_CODE",
    "HOSPITAL_TYPE",
    "HEALTH_SYSTEM_ID",
    "HEALTH_SYSTEM_NAME",
    "TOTAL_BEDS",
]


def create_system_key(
    df: pd.DataFrame,
) -> pd.Series:
    system_id = (
        df["HEALTH_SYSTEM_ID"]
        .map(normalize_text)
        .astype("string")
    )

    system_name = (
        df["HEALTH_SYSTEM_NAME"]
        .map(normalize_name_key)
        .astype("string")
    )

    return pd.Series(
        np.where(
            system_id.notna(),
            "ID|" + system_id.fillna(""),
            np.where(
                system_name.notna(),
                "NAME|" + system_name.fillna(""),
                pd.NA,
            ),
        ),
        index=df.index,
        dtype="string",
    )


# Add system keys to every available table.
for frame_name in [
    "outpatient_exact",
    "inpatient_exact",
    "concept_prices",
    "family_prices",
    "exact_component",
]:
    frame = globals().get(frame_name)

    if isinstance(frame, pd.DataFrame):
        frame["SYSTEM_KEY"] = create_system_key(frame)
        globals()[frame_name] = frame

# Build hospital disclosure events from all exact-code roles, retaining any
# hospital that only ever posted a component (add-on) code and never a
# standalone code.
event_frames = [
    outpatient_exact,
    inpatient_exact,
]

if not exact_component.empty:
    event_frames.append(exact_component)

event_source = pd.concat(
    [
        frame[EVENT_COLS + ["SYSTEM_KEY"]]
        for frame in event_frames
    ],
    ignore_index=True,
    sort=False,
)

require_columns(
    event_source,
    EVENT_COLS + ["SYSTEM_KEY"],
    "all-role exact-code event source",
)

hospital_events = (
    event_source
    .groupby(
        ["HOSPITAL_ID", "POST_MONTH"],
        as_index=False,
    )
    .agg({
        "PROVIDER_NPI": mode_nonmissing,
        "PROVIDER_NAME": mode_nonmissing,
        "PROVIDER_CITY": mode_nonmissing,
        "PROVIDER_STATE": mode_nonmissing,
        "CITY_STATE_KEY": mode_nonmissing,
        "COUNTY_STATE_KEY": mode_nonmissing,
        "COUNTY_FIPS": mode_nonmissing,
        "CBSA_CODE": mode_nonmissing,
        "HOSPITAL_TYPE": mode_nonmissing,
        "HEALTH_SYSTEM_ID": mode_nonmissing,
        "HEALTH_SYSTEM_NAME": mode_nonmissing,
        "SYSTEM_KEY": mode_nonmissing,
        "TOTAL_BEDS": "median",
    })
)

hospital_events["HOSPITAL_FIRST_POST_MONTH"] = (
    hospital_events
    .groupby("HOSPITAL_ID")["POST_MONTH"]
    .transform("min")
)

hospital_events["IS_HOSPITAL_FIRST_POST_MONTH"] = (
    hospital_events["POST_MONTH"]
    == hospital_events["HOSPITAL_FIRST_POST_MONTH"]
).astype("int8")

hospital_first_events = hospital_events.loc[
    hospital_events["IS_HOSPITAL_FIRST_POST_MONTH"] == 1
].copy()

qa_rows.append(
    unique_key_qa(
        hospital_first_events,
        ["HOSPITAL_ID"],
        "hospital_first_events",
    )
)

# N_HOSPITALS_WITH_COHORT is the distinct hospital count that the Phase 4
# posting-cohort table reconciles to, and it matches the all-roles hospital
# count in the Phase 3 QA table exactly. The check below is a hard stop: a
# mismatch means the event roster and the source disagree about which hospitals
# are in the sample.
expected_all_hospitals = int(
    phase4_qa.iloc[0]["N_HOSPITALS_WITH_COHORT"]
)

if hospital_first_events["HOSPITAL_ID"].nunique() != expected_all_hospitals:
    raise RuntimeError(
        "Hospital event roster does not reconcile to Phase 4 QA: "
        f"{hospital_first_events['HOSPITAL_ID'].nunique():,} "
        f"versus {expected_all_hospitals:,}"
    )

write_parquet(
    hospital_events,
    INSTRUMENT_DIR / "HPT_HOSPITAL_DISCLOSURE_EVENTS.parquet",
)


def add_service_entry_flags(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add first observed posting month for each hospital-service definition.

    Exact, concept, and family entries are distinct because a hospital may
    disclose different services in different observed posting files.
    """

    out = df.copy()

    exact_keys = [
        "HOSPITAL_ID",
        "BILLING_CODE_TYPE",
        "BILLING_CODE",
    ]

    concept_keys = [
        "HOSPITAL_ID",
        *phase4_concept_key_columns(),
    ]

    family_keys = [
        "HOSPITAL_ID",
        *phase4_family_key_columns(),
    ]

    out["EXACT_CODE_ENTRY_MONTH"] = (
        out.groupby(exact_keys)["POST_MONTH"]
        .transform("min")
    )

    out["CONCEPT_ENTRY_MONTH"] = (
        out.groupby(concept_keys)["POST_MONTH"]
        .transform("min")
    )

    out["FAMILY_ENTRY_MONTH"] = (
        out.groupby(family_keys)["POST_MONTH"]
        .transform("min")
    )

    out["IS_EXACT_CODE_ENTRY_MONTH"] = (
        out["POST_MONTH"]
        == out["EXACT_CODE_ENTRY_MONTH"]
    ).astype("int8")

    out["IS_CONCEPT_ENTRY_MONTH"] = (
        out["POST_MONTH"]
        == out["CONCEPT_ENTRY_MONTH"]
    ).astype("int8")

    out["IS_FAMILY_ENTRY_MONTH"] = (
        out["POST_MONTH"]
        == out["FAMILY_ENTRY_MONTH"]
    ).astype("int8")

    return out


outpatient_exact = add_service_entry_flags(
    outpatient_exact
)
inpatient_exact = add_service_entry_flags(
    inpatient_exact
)

if not exact_component.empty:
    exact_component = add_service_entry_flags(
        exact_component
    )

# HOSPITAL_FIRST_POST_MONTH and IS_HOSPITAL_FIRST_POST_MONTH are computed on
# hospital_events above but are not columns on the Snowflake exact-code export.
# They are merged in here from data already in memory, which avoids another
# export and download round-trip for two derived fields.
_hospital_first_post = (
    hospital_events[["HOSPITAL_ID", "POST_MONTH", "HOSPITAL_FIRST_POST_MONTH", "IS_HOSPITAL_FIRST_POST_MONTH"]]
    .drop_duplicates(["HOSPITAL_ID", "POST_MONTH"])
)

for _name in ["outpatient_exact", "inpatient_exact", "exact_component"]:
    _frame = globals()[_name]
    if not _frame.empty and "IS_HOSPITAL_FIRST_POST_MONTH" not in _frame.columns:
        _frame = safe_left_merge(
            _frame, _hospital_first_post, on=["HOSPITAL_ID", "POST_MONTH"],
            label=f"{_name} first-post flag",
        )
        globals()[_name] = _frame

del _hospital_first_post
msg("IS_HOSPITAL_FIRST_POST_MONTH attached to all three exact-code frames.")



In [ ]:
# =============================================================================
# 5. STRICT PRIOR-POSTER MEASURES: CITY, COUNTY, AND CBSA
# =============================================================================


def build_strict_prior_lookup(
    entries: pd.DataFrame,
    geo_col: str,
    service_cols: Sequence[str],
    prefix: str,
) -> pd.DataFrame:
    """
    Count distinct hospitals entering strictly before the focal month.

    Same-month co-posters are stored separately and never enter the strictly
    prior treatment.
    """

    keys = [
        geo_col,
        *service_cols,
    ]

    source = entries.dropna(
        subset=[
            geo_col,
            "POST_MONTH",
            "HOSPITAL_ID",
        ]
    ).copy()

    monthly = (
        source
        .groupby(
            keys + ["POST_MONTH"],
            as_index=False,
        )["HOSPITAL_ID"]
        .nunique()
        .rename(
            columns={
                "HOSPITAL_ID":
                    f"N_POSTERS_THIS_MONTH_{prefix}"
            }
        )
        .sort_values(
            keys + ["POST_MONTH"]
        )
    )

    monthly[
        f"N_CUMULATIVE_POSTERS_{prefix}"
    ] = (
        monthly
        .groupby(keys, sort=False)[
            f"N_POSTERS_THIS_MONTH_{prefix}"
        ]
        .cumsum()
    )

    monthly[
        f"N_PRIOR_POSTERS_{prefix}"
    ] = (
        monthly[
            f"N_CUMULATIVE_POSTERS_{prefix}"
        ]
        -
        monthly[
            f"N_POSTERS_THIS_MONTH_{prefix}"
        ]
    )

    monthly[
        f"IS_FIRST_WAVE_{prefix}"
    ] = (
        monthly[
            f"N_PRIOR_POSTERS_{prefix}"
        ] == 0
    ).astype("int8")

    return monthly


def attach_prior_measure(
    target: pd.DataFrame,
    entries: pd.DataFrame,
    geo_col: str,
    service_cols: Sequence[str],
    prefix: str,
) -> pd.DataFrame:
    lookup = build_strict_prior_lookup(
        entries,
        geo_col,
        service_cols,
        prefix,
    )

    merge_keys = [
        geo_col,
        *service_cols,
        "POST_MONTH",
    ]

    out = safe_left_merge(
        target,
        lookup,
        on=merge_keys,
        label=f"prior posters {prefix}",
    )

    new_cols = [
        f"N_POSTERS_THIS_MONTH_{prefix}",
        f"N_CUMULATIVE_POSTERS_{prefix}",
        f"N_PRIOR_POSTERS_{prefix}",
        f"IS_FIRST_WAVE_{prefix}",
    ]

    for column in new_cols:
        if column.startswith("IS_"):
            out[column] = (
                out[column]
                .fillna(0)
                .astype("int8")
            )
        else:
            out[column] = (
                out[column]
                .fillna(0)
                .astype("int32")
            )

    return out


def add_all_prior_measures(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """Attach exact, concept, and family prior-poster measures."""

    out = df.copy()

    geo_specs = [
        ("CITY_STATE_KEY", "CITY"),
        ("COUNTY_STATE_KEY", "COUNTY"),
        ("CBSA_CODE", "CBSA"),
    ]

    service_specs = [
        (
            [
                "BILLING_CODE_TYPE",
                "BILLING_CODE",
            ],
            "EXACT",
            "IS_EXACT_CODE_ENTRY_MONTH",
        ),
        (
            phase4_concept_key_columns(),
            "CONCEPT",
            "IS_CONCEPT_ENTRY_MONTH",
        ),
        (
            phase4_family_key_columns(),
            "FAMILY",
            "IS_FAMILY_ENTRY_MONTH",
        ),
    ]

    for geo_col, geo_label in geo_specs:
        for (
            service_cols,
            service_label,
            entry_flag,
        ) in service_specs:

            entries = out.loc[
                out[entry_flag] == 1,
                [
                    "HOSPITAL_ID",
                    "POST_MONTH",
                    geo_col,
                    *service_cols,
                ],
            ].drop_duplicates()

            prefix = (
                f"{geo_label}_{service_label}"
            )

            out = attach_prior_measure(
                out,
                entries,
                geo_col,
                service_cols,
                prefix,
            )

    return out


msg("Building service-specific prior-poster measures ...")

outpatient_exact = add_all_prior_measures(
    outpatient_exact
)
inpatient_exact = add_all_prior_measures(
    inpatient_exact
)

if not exact_component.empty:
    exact_component = add_all_prior_measures(
        exact_component
    )

# Market-wide hospital disclosure counts, independent of service.
for geo_col, geo_label in [
    ("CITY_STATE_KEY", "CITY_ALL"),
    ("COUNTY_STATE_KEY", "COUNTY_ALL"),
    ("CBSA_CODE", "CBSA_ALL"),
]:
    lookup = build_strict_prior_lookup(
        hospital_first_events[
            [
                "HOSPITAL_ID",
                "POST_MONTH",
                geo_col,
            ]
        ].drop_duplicates(),
        geo_col,
        [],
        geo_label,
    )

    for frame_name in [
        "outpatient_exact",
        "inpatient_exact",
        "exact_component",
    ]:
        frame = globals().get(frame_name)

        if not isinstance(frame, pd.DataFrame) or frame.empty:
            continue

        frame = safe_left_merge(
            frame,
            lookup,
            on=[
                geo_col,
                "POST_MONTH",
            ],
            label=(
                "all-service prior posters "
                f"{geo_label}"
            ),
        )

        for column in [
            f"N_POSTERS_THIS_MONTH_{geo_label}",
            f"N_CUMULATIVE_POSTERS_{geo_label}",
            f"N_PRIOR_POSTERS_{geo_label}",
        ]:
            frame[column] = (
                frame[column]
                .fillna(0)
                .astype("int32")
            )

        frame[
            f"IS_FIRST_WAVE_{geo_label}"
        ] = (
            frame[
                f"IS_FIRST_WAVE_{geo_label}"
            ]
            .fillna(0)
            .astype("int8")
        )

        globals()[frame_name] = frame

# Backward-compatible aliases used by earlier R scripts.
for frame_name in [
    "outpatient_exact",
    "inpatient_exact",
]:
    frame = globals()[frame_name]

    frame["N_PRIOR_POSTERS_VERIFIED"] = (
        frame["N_PRIOR_POSTERS_COUNTY_EXACT"]
    )

    frame[
        "N_HOSPITALS_CUMULATIVE_COUNTY"
    ] = (
        frame[
            "N_CUMULATIVE_POSTERS_COUNTY_EXACT"
        ]
    )

    frame[
        "N_HOSPITALS_CUMULATIVE_CITY"
    ] = (
        frame[
            "N_CUMULATIVE_POSTERS_CITY_EXACT"
        ]
    )

    frame[
        "N_HOSPITALS_CUMULATIVE"
    ] = (
        frame[
            "N_CUMULATIVE_POSTERS_CBSA_EXACT"
        ]
    )

    frame["IS_ENTRANT_MONTH_COUNTY"] = (
        frame["IS_EXACT_CODE_ENTRY_MONTH"]
    )
    frame["IS_ENTRANT_MONTH_CITY"] = (
        frame["IS_EXACT_CODE_ENTRY_MONTH"]
    )
    frame["IS_ENTRANT_MONTH"] = (
        frame["IS_EXACT_CODE_ENTRY_MONTH"]
    )

    globals()[frame_name] = frame

# Validate the market-wide county treatment against the Snowflake posting
# context for first-post rows. These should be the same construction.
county_context_differences = 0

for frame in [
    outpatient_exact,
    inpatient_exact,
]:
    if (
        "COUNTY_N_PRIOR_POSTERS" not in frame.columns
        or "N_PRIOR_POSTERS_COUNTY_ALL" not in frame.columns
    ):
        continue

    comparison = frame.loc[
        frame["IS_HOSPITAL_FIRST_POST_MONTH"] == 1,
        [
            "COUNTY_N_PRIOR_POSTERS",
            "N_PRIOR_POSTERS_COUNTY_ALL",
        ],
    ].dropna()

    county_context_differences += int(
        (
            pd.to_numeric(
                comparison["COUNTY_N_PRIOR_POSTERS"],
                errors="coerce",
            )
            !=
            pd.to_numeric(
                comparison["N_PRIOR_POSTERS_COUNTY_ALL"],
                errors="coerce",
            )
        ).sum()
    )

qa_rows.append({
    "check": "snowflake_python_county_prior_context",
    "status": (
        "PASS"
        if county_context_differences == 0
        else "REVIEW"
    ),
    "value": county_context_differences,
    "expected": 0,
    "details": (
        "Snowflake and Python all-service county prior posters "
        "on hospital first-post rows"
    ),
})


In [ ]:
# =============================================================================
# 6. COUNTY-LEVEL HEALTH-SYSTEM INSTRUMENTS
# =============================================================================

# One predetermined provider roster from first observed hospital disclosure.
provider_roster = hospital_first_events[[
    "HOSPITAL_ID", "HOSPITAL_FIRST_POST_MONTH", "COUNTY_STATE_KEY", "CBSA_CODE",
    "CITY_STATE_KEY", "SYSTEM_KEY", "HEALTH_SYSTEM_ID", "HEALTH_SYSTEM_NAME",
]].copy()
provider_roster = provider_roster.rename(columns={"HOSPITAL_FIRST_POST_MONTH": "PEER_POST_MONTH"})

system_sizes = (
    provider_roster.dropna(subset=["SYSTEM_KEY"])
    .groupby("SYSTEM_KEY")["HOSPITAL_ID"]
    .nunique()
    .sort_values(ascending=False)
)
TOP5_SYSTEMS = set(system_sizes.head(5).index)

local_system_roster = (
    provider_roster.dropna(subset=["COUNTY_STATE_KEY", "SYSTEM_KEY"])
    .groupby(["COUNTY_STATE_KEY", "SYSTEM_KEY"], as_index=False)
    .agg(
        LOCAL_SYSTEM_FIRST_POST_MONTH=("PEER_POST_MONTH", "min"),
        LOCAL_SYSTEM_N_HOSPITALS=("HOSPITAL_ID", "nunique"),
    )
    .rename(columns={"COUNTY_STATE_KEY": "FOCAL_COUNTY"})
)

peer_events = provider_roster.dropna(subset=["SYSTEM_KEY", "COUNTY_STATE_KEY", "PEER_POST_MONTH"]).rename(columns={
    "HOSPITAL_ID": "PEER_HOSPITAL_ID",
    "COUNTY_STATE_KEY": "PEER_COUNTY",
    "CBSA_CODE": "PEER_CBSA",
})

county_peer_universe = local_system_roster.merge(peer_events, on="SYSTEM_KEY", how="inner")
county_peer_universe = county_peer_universe.loc[
    county_peer_universe["FOCAL_COUNTY"] != county_peer_universe["PEER_COUNTY"]
].copy()
county_peer_universe["SYSTEM_IS_TOP5"] = county_peer_universe["SYSTEM_KEY"].isin(TOP5_SYSTEMS)

county_month_index = (
    hospital_events[
        [
            "COUNTY_STATE_KEY",
            "CBSA_CODE",
            "POST_MONTH",
        ]
    ]
    .dropna(
        subset=[
            "COUNTY_STATE_KEY",
            "POST_MONTH",
        ]
    )
    .groupby(
        [
            "COUNTY_STATE_KEY",
            "POST_MONTH",
        ],
        as_index=False,
    )
    .agg(
        CBSA_CODE=(
            "CBSA_CODE",
            mode_nonmissing,
        )
    )
    .rename(
        columns={
            "COUNTY_STATE_KEY":
                "FOCAL_COUNTY",
            "CBSA_CODE":
                "FOCAL_CBSA",
        }
    )
)


def _window_mask(dates: pd.Series, focal_month: pd.Timestamp, months: int, include_current: bool) -> pd.Series:
    lower = focal_month - pd.DateOffset(months=months)
    upper_ok = dates <= focal_month if include_current else dates < focal_month
    return (dates >= lower) & upper_ok


def _distinct_count(frame: pd.DataFrame, column: str, mask: pd.Series) -> int:
    return int(frame.loc[mask, column].nunique())


def build_county_month_system_instruments(
    county_months: pd.DataFrame,
    peer_universe: pd.DataFrame,
) -> pd.DataFrame:
    results: list[dict[str, object]] = []
    peer_groups = {k: g.copy() for k, g in peer_universe.groupby("FOCAL_COUNTY", sort=False)}

    for focal_county, months_df in county_months.groupby("FOCAL_COUNTY", sort=False):
        peers = peer_groups.get(focal_county)
        for row in months_df.itertuples(index=False):
            focal_month = pd.Timestamp(row.POST_MONTH)
            focal_cbsa = row.FOCAL_CBSA
            result: dict[str, object] = {
                "COUNTY_STATE_KEY": focal_county,
                "POST_MONTH": focal_month,
            }

            if peers is None or peers.empty:
                for col in [
                    "Z_SYS_ORIGINAL_9M_INCL_CURRENT",
                    "Z_SYS_STRICT_9M_EXCL_CURRENT",
                    "Z_SYS_FIXED_ROSTER_9M_EXCL_CURRENT",
                    "Z_SYS_ACTIVE_SYSTEMS_9M_EXCL_CURRENT",
                    "Z_SYS_OUTSIDE_CBSA_9M_EXCL_CURRENT",
                    "Z_SYS_RECENT_FLOW_3M_EXCL_CURRENT",
                    "Z_SYS_CUMULATIVE_EXTERNAL_HOSPITALS",
                    "Z_SYS_CUMULATIVE_ROLLOUT_SHARE",
                    "Z_SYS_EVER_INITIATED_SYSTEMS",
                    "Z_SYS_STRICT_9M_EXCL_TOP5",
                    "Z_SYS_ELIGIBLE_EXTERNAL_HOSPITALS_FIXED",
                ]:
                    result[col] = 0.0 if "SHARE" in col else 0
                for window in SUPPORTING_WINDOWS:
                    for unit in ["HOSPITALS", "COUNTIES", "SYSTEMS"]:
                        result[f"Z_SYS_PEER_{unit}_{window}M_STRICT"] = 0
                        result[f"Z_SYS_PEER_{unit}_{window}M_INCL_CURRENT"] = 0
                results.append(result)
                continue

            dynamic_ok = peers["LOCAL_SYSTEM_FIRST_POST_MONTH"] <= focal_month
            fixed = pd.Series(True, index=peers.index)
            strict9 = _window_mask(peers["PEER_POST_MONTH"], focal_month, CANONICAL_LOOKBACK_MONTHS, False)
            inclusive9 = _window_mask(peers["PEER_POST_MONTH"], focal_month, CANONICAL_LOOKBACK_MONTHS, True)
            recent3 = _window_mask(peers["PEER_POST_MONTH"], focal_month, 3, False)
            cumulative = peers["PEER_POST_MONTH"] < focal_month

            # 1. Original: dynamic local-system roster, t-9 through t inclusive.
            result["Z_SYS_ORIGINAL_9M_INCL_CURRENT"] = _distinct_count(
                peers, "PEER_HOSPITAL_ID", dynamic_ok & inclusive9
            )
            # 2. Strict same-month exclusion.
            result["Z_SYS_STRICT_9M_EXCL_CURRENT"] = _distinct_count(
                peers, "PEER_HOSPITAL_ID", dynamic_ok & strict9
            )
            # 3. Full-sample fixed local-system roster, strict window.
            result["Z_SYS_FIXED_ROSTER_9M_EXCL_CURRENT"] = _distinct_count(
                peers, "PEER_HOSPITAL_ID", fixed & strict9
            )
            # 4. Distinct active external rollout systems.
            result["Z_SYS_ACTIVE_SYSTEMS_9M_EXCL_CURRENT"] = _distinct_count(
                peers, "SYSTEM_KEY", dynamic_ok & strict9
            )
            # 5. External peers must also be outside the focal CBSA.
            if pd.isna(focal_cbsa):
                result["Z_SYS_OUTSIDE_CBSA_9M_EXCL_CURRENT"] = np.nan
            else:
                outside_cbsa = peers["PEER_CBSA"].isna() | (peers["PEER_CBSA"] != focal_cbsa)
                result["Z_SYS_OUTSIDE_CBSA_9M_EXCL_CURRENT"] = _distinct_count(
                    peers, "PEER_HOSPITAL_ID", dynamic_ok & strict9 & outside_cbsa
                )
            # 7. Recent flow t-3:t-1.
            result["Z_SYS_RECENT_FLOW_3M_EXCL_CURRENT"] = _distinct_count(
                peers, "PEER_HOSPITAL_ID", dynamic_ok & recent3
            )
            # 8. All strictly prior external peer hospitals in fixed roster.
            result["Z_SYS_CUMULATIVE_EXTERNAL_HOSPITALS"] = _distinct_count(
                peers, "PEER_HOSPITAL_ID", fixed & cumulative
            )
            # 9. Cumulative rollout share among eligible external peer hospitals.
            eligible = int(peers["PEER_HOSPITAL_ID"].nunique())
            cumulative_n = int(result["Z_SYS_CUMULATIVE_EXTERNAL_HOSPITALS"])
            result["Z_SYS_ELIGIBLE_EXTERNAL_HOSPITALS_FIXED"] = eligible
            result["Z_SYS_CUMULATIVE_ROLLOUT_SHARE"] = cumulative_n / eligible if eligible > 0 else 0.0
            # 10. Number of local systems that have ever initiated externally.
            result["Z_SYS_EVER_INITIATED_SYSTEMS"] = _distinct_count(
                peers, "SYSTEM_KEY", fixed & cumulative
            )
            # 11. Strict instrument excluding the five largest national systems.
            result["Z_SYS_STRICT_9M_EXCL_TOP5"] = _distinct_count(
                peers, "PEER_HOSPITAL_ID", dynamic_ok & strict9 & ~peers["SYSTEM_IS_TOP5"]
            )

            # Supporting windows: peer hospitals, peer counties, and peer systems.
            for window in SUPPORTING_WINDOWS:
                strict = _window_mask(peers["PEER_POST_MONTH"], focal_month, window, False)
                incl = _window_mask(peers["PEER_POST_MONTH"], focal_month, window, True)
                for unit, column in [
                    ("HOSPITALS", "PEER_HOSPITAL_ID"),
                    ("COUNTIES", "PEER_COUNTY"),
                    ("SYSTEMS", "SYSTEM_KEY"),
                ]:
                    result[f"Z_SYS_PEER_{unit}_{window}M_STRICT"] = _distinct_count(
                        peers, column, dynamic_ok & strict
                    )
                    result[f"Z_SYS_PEER_{unit}_{window}M_INCL_CURRENT"] = _distinct_count(
                        peers, column, dynamic_ok & incl
                    )

            results.append(result)

    return pd.DataFrame(results)


msg("Building county-month system-peer instruments ...")
county_month_iv = build_county_month_system_instruments(county_month_index, county_peer_universe)
qa_rows.append(unique_key_qa(county_month_iv, ["COUNTY_STATE_KEY", "POST_MONTH"], "county_month_iv"))
write_parquet(county_month_iv, INSTRUMENT_DIR / "HPT_COUNTY_MONTH_SYSTEM_INSTRUMENTS.parquet")

# 6. Competitor-only instrument is focal-hospital specific: exclude the focal
# hospital's own system from the dynamic strict peer set.

def build_competitor_instrument(
    events: pd.DataFrame,
    peer_universe: pd.DataFrame,
) -> pd.DataFrame:
    peer_groups = {k: g.copy() for k, g in peer_universe.groupby("FOCAL_COUNTY", sort=False)}
    rows: list[dict[str, object]] = []
    for focal in events.itertuples(index=False):
        county = focal.COUNTY_STATE_KEY
        month = pd.Timestamp(focal.POST_MONTH)
        own_system = focal.SYSTEM_KEY
        focal_cbsa = getattr(focal, "CBSA_CODE", None)
        peers = peer_groups.get(county)
        if peers is None or peers.empty:
            n_hospitals = n_systems = n_counties = 0
            # Outside-CBSA competitor variants: NaN when there are no peers at
            # all (not zero), consistent with how IV05 handles a missing/
            # unusable comparison rather than treating "no peers" as "zero
            # competitors found."
            n_hospitals_cbsa = n_systems_cbsa = n_counties_cbsa = np.nan
        else:
            dynamic_ok = peers["LOCAL_SYSTEM_FIRST_POST_MONTH"] <= month
            strict = _window_mask(peers["PEER_POST_MONTH"], month, CANONICAL_LOOKBACK_MONTHS, False)
            competitor = pd.Series(True, index=peers.index)
            if own_system is not None and not pd.isna(own_system):
                competitor &= peers["SYSTEM_KEY"] != own_system
            mask = dynamic_ok & strict & competitor
            n_hospitals = int(peers.loc[mask, "PEER_HOSPITAL_ID"].nunique())
            n_systems = int(peers.loc[mask, "SYSTEM_KEY"].nunique())
            n_counties = int(peers.loc[mask, "PEER_COUNTY"].nunique())

            # Competitor peers that are ALSO outside the focal hospital's
            # CBSA -- combines IV06 (competitor-only, excludes own system)
            # with IV05's exclusion restriction (peer must be outside the
            # focal CBSA, not just outside the focal county). Same NaN
            # convention as IV05: if the focal hospital has no CBSA, the
            # comparison isn't meaningful, so these are NaN, not 0.
            if pd.isna(focal_cbsa):
                n_hospitals_cbsa = n_systems_cbsa = n_counties_cbsa = np.nan
            else:
                outside_cbsa = peers["PEER_CBSA"].isna() | (peers["PEER_CBSA"] != focal_cbsa)
                mask_cbsa = mask & outside_cbsa
                n_hospitals_cbsa = int(peers.loc[mask_cbsa, "PEER_HOSPITAL_ID"].nunique())
                n_systems_cbsa = int(peers.loc[mask_cbsa, "SYSTEM_KEY"].nunique())
                n_counties_cbsa = int(peers.loc[mask_cbsa, "PEER_COUNTY"].nunique())
        rows.append({
            "HOSPITAL_ID": focal.HOSPITAL_ID,
            "POST_MONTH": month,
            "Z_SYS_COMPETITOR_ONLY_9M_EXCL_CURRENT": n_hospitals,
            "Z_SYS_COMPETITOR_SYSTEMS_9M_EXCL_CURRENT": n_systems,
            "Z_SYS_COMPETITOR_COUNTIES_9M_EXCL_CURRENT": n_counties,
            "Z_SYS_COMPETITOR_OUTSIDE_CBSA_9M_EXCL_CURRENT": n_hospitals_cbsa,
            "Z_SYS_COMPETITOR_SYSTEMS_OUTSIDE_CBSA_9M_EXCL_CURRENT": n_systems_cbsa,
            "Z_SYS_COMPETITOR_COUNTIES_OUTSIDE_CBSA_9M_EXCL_CURRENT": n_counties_cbsa,
        })
    return pd.DataFrame(rows)

competitor_iv = build_competitor_instrument(hospital_events, county_peer_universe)
qa_rows.append(unique_key_qa(competitor_iv, ["HOSPITAL_ID", "POST_MONTH"], "competitor_iv"))


def merge_system_instruments(df: pd.DataFrame) -> pd.DataFrame:
    out = safe_left_merge(
        df,
        county_month_iv,
        on=["COUNTY_STATE_KEY", "POST_MONTH"],
        label="county-month system IV",
    )
    out = safe_left_merge(
        out,
        competitor_iv,
        on=["HOSPITAL_ID", "POST_MONTH"],
        label="competitor system IV",
    )
    iv_cols = [c for c in out.columns if c.startswith("Z_SYS_")]
    # These stay NaN (not 0) when the focal hospital has no CBSA, since "no
    # CBSA to compare against" is not the same claim as "zero competitors" --
    # same convention for all four CBSA-dependent instruments.
    cbsa_dependent_cols = {
        "Z_SYS_OUTSIDE_CBSA_9M_EXCL_CURRENT",
        "Z_SYS_COMPETITOR_OUTSIDE_CBSA_9M_EXCL_CURRENT",
        "Z_SYS_COMPETITOR_SYSTEMS_OUTSIDE_CBSA_9M_EXCL_CURRENT",
        "Z_SYS_COMPETITOR_COUNTIES_OUTSIDE_CBSA_9M_EXCL_CURRENT",
    }
    for col in iv_cols:
        if col in cbsa_dependent_cols:
            continue
        out[col] = out[col].fillna(0)
    return out

outpatient_exact = merge_system_instruments(
    outpatient_exact
)
inpatient_exact = merge_system_instruments(
    inpatient_exact
)

if not exact_component.empty:
    exact_component = merge_system_instruments(
        exact_component
    )

# Canonical aliases for the 11 original variants, plus 3 competitor-outside-
# CBSA hybrids (IV06 x IV05: competitor-only AND outside the focal CBSA)
# added to match the R analysis's NEW_COMPETITOR_INSTRUMENTS menu.
CANONICAL_INSTRUMENTS = {
    "IV01_ORIGINAL_9M_INCL": "Z_SYS_ORIGINAL_9M_INCL_CURRENT",
    "IV02_STRICT_9M_EXCL": "Z_SYS_STRICT_9M_EXCL_CURRENT",
    "IV03_FIXED_ROSTER_9M": "Z_SYS_FIXED_ROSTER_9M_EXCL_CURRENT",
    "IV04_DISTINCT_ACTIVE_SYSTEMS": "Z_SYS_ACTIVE_SYSTEMS_9M_EXCL_CURRENT",
    "IV05_OUTSIDE_CBSA": "Z_SYS_OUTSIDE_CBSA_9M_EXCL_CURRENT",
    "IV06_COMPETITOR_ONLY": "Z_SYS_COMPETITOR_ONLY_9M_EXCL_CURRENT",
    "IV07_RECENT_FLOW_3M": "Z_SYS_RECENT_FLOW_3M_EXCL_CURRENT",
    "IV08_CUMULATIVE_EXTERNAL_HOSPITALS": "Z_SYS_CUMULATIVE_EXTERNAL_HOSPITALS",
    "IV09_CUMULATIVE_ROLLOUT_SHARE": "Z_SYS_CUMULATIVE_ROLLOUT_SHARE",
    "IV10_EVER_INITIATED_SYSTEMS": "Z_SYS_EVER_INITIATED_SYSTEMS",
    "IV11_STRICT_EXCLUDING_TOP5": "Z_SYS_STRICT_9M_EXCL_TOP5",
    "IV12_COMPETITOR_OUTSIDE_CBSA_HOSPITALS": "Z_SYS_COMPETITOR_OUTSIDE_CBSA_9M_EXCL_CURRENT",
    "IV13_COMPETITOR_OUTSIDE_CBSA_SYSTEMS": "Z_SYS_COMPETITOR_SYSTEMS_OUTSIDE_CBSA_9M_EXCL_CURRENT",
    "IV14_COMPETITOR_OUTSIDE_CBSA_COUNTIES": "Z_SYS_COMPETITOR_COUNTIES_OUTSIDE_CBSA_9M_EXCL_CURRENT",
}

instrument_dictionary = pd.DataFrame([
    {
        "instrument_number": key.split("_")[0],
        "instrument_label": key,
        "column_name": value,
        "definition": {
            "IV01_ORIGINAL_9M_INCL": "Distinct out-of-county peer hospitals in dynamically active local systems, t-9 through t inclusive.",
            "IV02_STRICT_9M_EXCL": "Distinct out-of-county peer hospitals in dynamically active local systems, t-9 through t-1.",
            "IV03_FIXED_ROSTER_9M": "Distinct out-of-county peer hospitals using the full-sample fixed local-system roster, t-9 through t-1.",
            "IV04_DISTINCT_ACTIVE_SYSTEMS": "Distinct local systems with at least one external peer posting in t-9 through t-1.",
            "IV05_OUTSIDE_CBSA": "Strict dynamic peers outside both the focal county and the focal CBSA.",
            "IV06_COMPETITOR_ONLY": "Strict dynamic peers in locally represented systems other than the focal hospital's own system.",
            "IV07_RECENT_FLOW_3M": "Distinct out-of-county peer hospitals in dynamically active local systems, t-3 through t-1.",
            "IV08_CUMULATIVE_EXTERNAL_HOSPITALS": "All distinct fixed-roster external peer hospitals posting strictly before t.",
            "IV09_CUMULATIVE_ROLLOUT_SHARE": "Cumulative prior external peer hospitals divided by all eligible external hospitals in fixed local systems.",
            "IV10_EVER_INITIATED_SYSTEMS": "Distinct fixed-roster local systems with at least one external posting strictly before t.",
            "IV11_STRICT_EXCLUDING_TOP5": "Strict dynamic 9-month peer-hospital count excluding the five largest systems nationally.",
            "IV12_COMPETITOR_OUTSIDE_CBSA_HOSPITALS": "IV06 (competitor-only) restricted further to peers outside the focal hospital's CBSA; NaN if the focal hospital has no CBSA.",
            "IV13_COMPETITOR_OUTSIDE_CBSA_SYSTEMS": "Distinct competitor systems (not own system) with a peer outside the focal CBSA in the strict 9-month window.",
            "IV14_COMPETITOR_OUTSIDE_CBSA_COUNTIES": "Distinct competitor counties (not own system) outside the focal CBSA in the strict 9-month window.",
        }[key],
    }
    for key, value in CANONICAL_INSTRUMENTS.items()
])
instrument_dictionary.to_csv(INSTRUMENT_DIR / "HPT_SYSTEM_INSTRUMENT_DICTIONARY.csv", index=False)


In [ ]:
# =============================================================================
# 7. OPTIONAL SERVICE-SPECIFIC FIXED-ROSTER SYSTEM INSTRUMENTS
# =============================================================================


def attach_service_specific_fixed_iv(
    target: pd.DataFrame,
    provider_roster_df: pd.DataFrame,
    local_roster_df: pd.DataFrame,
    service_cols: Sequence[str],
    output_col: str,
    entry_flag: str,
    lookback_months: int = 9,
) -> pd.DataFrame:
    """
    Fixed-roster same-service system-peer count, t-lookback through t-1.

    This is optional because the county-system × peer-service expansion can be
    memory intensive. Each peer hospital-service entry occurs only once, so
    monthly counts can be summed without double-counting hospitals across months.
    """
    entries = target.loc[target[entry_flag] == 1, [
        "HOSPITAL_ID", "POST_MONTH", "COUNTY_STATE_KEY", "SYSTEM_KEY", *service_cols
    ]].dropna(subset=["SYSTEM_KEY", "COUNTY_STATE_KEY", *service_cols]).drop_duplicates()
    peers = entries.rename(columns={
        "HOSPITAL_ID": "PEER_HOSPITAL_ID",
        "POST_MONTH": "PEER_POST_MONTH",
        "COUNTY_STATE_KEY": "PEER_COUNTY",
    })
    candidates = local_roster_df[["FOCAL_COUNTY", "SYSTEM_KEY"]].merge(peers, on="SYSTEM_KEY", how="inner")
    candidates = candidates.loc[candidates["FOCAL_COUNTY"] != candidates["PEER_COUNTY"]]
    monthly = (
        candidates.groupby(["FOCAL_COUNTY", *service_cols, "PEER_POST_MONTH"], as_index=False)["PEER_HOSPITAL_ID"]
        .nunique()
        .rename(columns={"PEER_HOSPITAL_ID": "N_PEERS_MONTH"})
    )

    focal_index = target.loc[target[entry_flag] == 1, [
        "COUNTY_STATE_KEY", *service_cols, "POST_MONTH"
    ]].drop_duplicates().rename(columns={"COUNTY_STATE_KEY": "FOCAL_COUNTY"})

    result_parts: list[pd.DataFrame] = []
    group_cols = ["FOCAL_COUNTY", *service_cols]
    monthly_groups = {k: g for k, g in monthly.groupby(group_cols, sort=False)}
    for group_key, focal_group in focal_index.groupby(group_cols, sort=False):
        group_key_tuple = group_key if isinstance(group_key, tuple) else (group_key,)
        event_group = monthly_groups.get(group_key)
        focal_group = focal_group.copy().sort_values("POST_MONTH")
        if event_group is None or event_group.empty:
            focal_group[output_col] = 0
        else:
            dates = event_group["PEER_POST_MONTH"].to_numpy(dtype="datetime64[ns]")
            counts = event_group["N_PEERS_MONTH"].to_numpy(dtype=np.int64)
            order = np.argsort(dates)
            dates = dates[order]
            counts = counts[order]
            cum = np.concatenate([[0], np.cumsum(counts)])
            focal_dates = focal_group["POST_MONTH"].to_numpy(dtype="datetime64[ns]")
            lower_dates = (focal_group["POST_MONTH"] - pd.DateOffset(months=lookback_months)).to_numpy(dtype="datetime64[ns]")
            left = np.searchsorted(dates, lower_dates, side="left")
            right = np.searchsorted(dates, focal_dates, side="left")
            focal_group[output_col] = cum[right] - cum[left]
        result_parts.append(focal_group)
    lookup = pd.concat(result_parts, ignore_index=True) if result_parts else focal_index.assign(**{output_col: 0})
    lookup = lookup.rename(columns={"FOCAL_COUNTY": "COUNTY_STATE_KEY"})
    out = safe_left_merge(target, lookup, on=["COUNTY_STATE_KEY", *service_cols, "POST_MONTH"], label=output_col)
    out[output_col] = out[output_col].fillna(0).astype("int32")
    return out


if BUILD_SERVICE_SPECIFIC_SYSTEM_IV:
    msg("Building optional service-specific fixed-roster instruments ...")
    for frame_name in ["outpatient_exact", "inpatient_exact"]:
        frame = globals()[frame_name]
        frame = attach_service_specific_fixed_iv(
            frame, provider_roster, local_system_roster,
            ["BILLING_CODE_TYPE", "BILLING_CODE"],
            "Z_SYS_FIXED_SAME_EXACT_9M_STRICT",
            "IS_EXACT_CODE_ENTRY_MONTH",
        )
        frame["Z_SYS_FIXED_SAME_CONCEPT_9M_STRICT"] = frame["Z_SYS_FIXED_SAME_EXACT_9M_STRICT"]
        frame = attach_service_specific_fixed_iv(
            frame, provider_roster, local_system_roster,
            phase4_family_key_columns(),
            "Z_SYS_FIXED_SAME_FAMILY_9M_STRICT",
            "IS_FAMILY_ENTRY_MONTH",
        )
        globals()[frame_name] = frame

In [ ]:
# =============================================================================
# 8. ACS DEMOGRAPHIC CONTROLS
# =============================================================================


def collapse_acs(df: pd.DataFrame, group_cols: Sequence[str]) -> pd.DataFrame:
    required = [
        "YEAR", "PERWT", "HCOVANY", "HCOVPUB", "HCOVPRIV", "AGE", "RACE",
        "HISPAN", "EDUCD", "POVERTY", "EMPSTAT", "OWNERSHP", "HHINCOME",
        *group_cols,
    ]
    require_columns(df, required, "ACS microdata")
    work = df.copy()
    work["uninsured"] = (work["HCOVANY"] == 1).astype("int8")
    work["public_insured"] = (work["HCOVPUB"] == 2).astype("int8")
    work["private_insured"] = (work["HCOVPRIV"] == 2).astype("int8")
    work["age65plus"] = (work["AGE"] >= 65).astype("int8")
    work["black"] = (work["RACE"] == 2).astype("int8")
    work["hispanic"] = (work["HISPAN"] > 0).astype("int8")
    work["collegeplus"] = (work["EDUCD"] >= 101).astype("int8")
    work["poor"] = (work["POVERTY"] < 100).astype("int8")
    work["employed"] = (work["EMPSTAT"] == 1).astype("int8")
    work["homeowner"] = (work["OWNERSHP"] == 1).astype("int8")
    work["HHINCOME_CLEAN"] = pd.to_numeric(work["HHINCOME"], errors="coerce")
    work.loc[(work["HHINCOME_CLEAN"] < 0) | (work["HHINCOME_CLEAN"] >= 9_999_998), "HHINCOME_CLEAN"] = np.nan

    records: list[dict[str, object]] = []
    grouped = work.groupby([*group_cols, "YEAR"], observed=True, sort=False)
    for key, g in grouped:
        key_tuple = key if isinstance(key, tuple) else (key,)
        rec = dict(zip([*group_cols, "YEAR"], key_tuple))
        median_income = weighted_median(g["HHINCOME_CLEAN"], g["PERWT"])
        rec.update({
            "uninsured_rate": weighted_mean(g["uninsured"], g["PERWT"]),
            "medicaid_share": weighted_mean(g["public_insured"], g["PERWT"]),
            "employer_ins_share": weighted_mean(g["private_insured"], g["PERWT"]),
            "poverty_rate": weighted_mean(g["poor"], g["PERWT"]),
            "median_income": median_income,
            "log_median_income": np.log(median_income + 1) if np.isfinite(median_income) else np.nan,
            "age65plus_share": weighted_mean(g["age65plus"], g["PERWT"]),
            "college_share": weighted_mean(g["collegeplus"], g["PERWT"]),
            "black_share": weighted_mean(g["black"], g["PERWT"]),
            "hispanic_share": weighted_mean(g["hispanic"], g["PERWT"]),
            "employment_rate": weighted_mean(g["employed"], g["PERWT"]),
            "homeowner_rate": weighted_mean(g["homeowner"], g["PERWT"]),
            "population": pd.to_numeric(g["PERWT"], errors="coerce").sum(),
        })
        records.append(rec)
    result = pd.DataFrame(records).rename(columns={"YEAR": "YEAR"})
    return result


acs_county_controls = pd.DataFrame()
acs_cbsa_controls = pd.DataFrame()
max_acs_year: int | None = None

if BUILD_ACS_CONTROLS:
    acs_path = find_file(RAW_FILES["acs"], SEARCH_DIRS_RAW)
    puma_cbsa_path = find_file(RAW_FILES["puma_cbsa"], SEARCH_DIRS_RAW)
    acs = read_csv_file(acs_path)
    require_columns(acs, ["STATEFIP", "COUNTYFIP", "PUMA", "YEAR"], "ACS microdata")
    acs["COUNTY_FIPS"] = (
        pd.to_numeric(acs["STATEFIP"], errors="coerce").astype("Int64").astype("string").str.zfill(2)
        + pd.to_numeric(acs["COUNTYFIP"], errors="coerce").astype("Int64").astype("string").str.zfill(3)
    )
    acs_county_controls = collapse_acs(acs, ["COUNTY_FIPS"])

    puma_crosswalk = pd.read_csv(puma_cbsa_path, skiprows=1, encoding="latin1", low_memory=False)
    puma_crosswalk = normalize_column_names(puma_crosswalk)
    state_col = next(c for c in puma_crosswalk.columns if c in {"STATE_CODE", "STATEFIP", "STATE"})
    puma_col = next(c for c in puma_crosswalk.columns if "PUMA" in c and c != "STATE_PUMA")
    cbsa_col = next(c for c in puma_crosswalk.columns if "CORE_BASED_STATISTICAL_AREA_CODE" in c or c == "CBSA_CODE")
    allocation_candidates = [c for c in puma_crosswalk.columns if any(x in c for x in ["AF", "ALLOC", "RATIO", "FACTOR"])]
    puma_crosswalk["STATE_PUMA"] = (
        pd.to_numeric(puma_crosswalk[state_col], errors="coerce").astype("Int64").astype("string").str.zfill(2)
        + pd.to_numeric(puma_crosswalk[puma_col], errors="coerce").astype("Int64").astype("string").str.zfill(5)
    )
    puma_crosswalk["CBSA_CODE"] = puma_crosswalk[cbsa_col].map(normalize_cbsa)
    if allocation_candidates:
        allocation_col = allocation_candidates[0]
        puma_crosswalk[allocation_col] = pd.to_numeric(puma_crosswalk[allocation_col], errors="coerce")
        puma_crosswalk = puma_crosswalk.sort_values(allocation_col, ascending=False).drop_duplicates("STATE_PUMA")
    else:
        puma_crosswalk = puma_crosswalk.drop_duplicates("STATE_PUMA")

    acs["STATE_PUMA"] = (
        pd.to_numeric(acs["STATEFIP"], errors="coerce").astype("Int64").astype("string").str.zfill(2)
        + pd.to_numeric(acs["PUMA"], errors="coerce").astype("Int64").astype("string").str.zfill(5)
    )
    acs = acs.merge(puma_crosswalk[["STATE_PUMA", "CBSA_CODE"]], on="STATE_PUMA", how="left")
    acs_cbsa_controls = collapse_acs(acs.dropna(subset=["CBSA_CODE"]), ["CBSA_CODE"])
    max_acs_year = int(max(acs_county_controls["YEAR"].max(), acs_cbsa_controls["YEAR"].max()))

    acs_county_controls.to_csv(CONTROL_DIR / "HPT_ACS_CONTROLS_COUNTY_YEAR.csv", index=False)
    acs_cbsa_controls.to_csv(CONTROL_DIR / "HPT_ACS_CONTROLS_CBSA_YEAR.csv", index=False)
    del acs

In [ ]:
# =============================================================================
# 9. HOSPITAL OWNERSHIP CONTROLS
# =============================================================================

ownership_county = pd.DataFrame()
ownership_cbsa = pd.DataFrame()
ownership_hospital = pd.DataFrame()

OWNERSHIP_MAP = {
    "DEPARTMENT OF DEFENSE": "Government",
    "GOVERNMENT - FEDERAL": "Government",
    "GOVERNMENT - HOSPITAL DISTRICT OR AUTHORITY": "Government",
    "GOVERNMENT - LOCAL": "Government",
    "GOVERNMENT - STATE": "Government",
    "TRIBAL": "Government",
    "VETERANS HEALTH ADMINISTRATION": "Government",
    "VOLUNTARY NON-PROFIT - CHURCH": "Non-Profit",
    "VOLUNTARY NON-PROFIT - OTHER": "Non-Profit",
    "VOLUNTARY NON-PROFIT - PRIVATE": "Non-Profit",
    "PROPRIETARY": "For-Profit",
    "PHYSICIAN": "For-Profit",
}


def collapse_ownership(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    out = (
        df.dropna(subset=[group_col, "OWNERSHIP_GROUP"])
        .groupby([group_col, "OWNERSHIP_GROUP"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )
    for col in ["For-Profit", "Government", "Non-Profit"]:
        if col not in out.columns:
            out[col] = 0
    out["Total_Hospitals"] = out[["For-Profit", "Government", "Non-Profit"]].sum(axis=1)
    out["Share_ForProfit"] = out["For-Profit"] / out["Total_Hospitals"].replace(0, np.nan)
    out["Share_Government"] = out["Government"] / out["Total_Hospitals"].replace(0, np.nan)
    out["Share_NonProfit"] = out["Non-Profit"] / out["Total_Hospitals"].replace(0, np.nan)
    return out


if BUILD_OWNERSHIP_CONTROLS:
    ownership_path = find_file(RAW_FILES["ownership"], SEARCH_DIRS_RAW)
    own = read_csv_file(ownership_path)
    ownership_col = next((c for c in own.columns if c in {"HOSPITAL_OWNERSHIP", "OWNERSHIP"}), None)
    state_col = next((c for c in own.columns if c in {"STATE", "STATE_CODE"}), None)
    county_col = next((c for c in own.columns if c in {"COUNTY_PARISH", "COUNTY", "COUNTY_NAME"}), None)
    name_col = next((c for c in own.columns if c in {"FACILITY_NAME", "HOSPITAL_NAME", "PROVIDER_NAME"}), None)
    city_col = next((c for c in own.columns if c in {"CITY", "PROVIDER_CITY"}), None)
    if ownership_col is None or state_col is None or county_col is None:
        raise KeyError("Ownership file must include ownership, state, and county columns.")

    own["OWNERSHIP_GROUP"] = own[ownership_col].map(normalize_text).map(OWNERSHIP_MAP)
    own["STATE_KEY"] = own[state_col].map(normalize_text)
    own["COUNTY_STATE_KEY"] = [
        normalize_county_state_key(None, c, s)
        for c, s in zip(own[county_col], own[state_col])
    ]
    county_to_cbsa = (
        hospital_locations.dropna(subset=["COUNTY_STATE_KEY", "CBSA_CODE"])
        .groupby("COUNTY_STATE_KEY")["CBSA_CODE"]
        .agg(mode_nonmissing)
        .reset_index()
    )
    own = own.merge(county_to_cbsa, on="COUNTY_STATE_KEY", how="left")
    ownership_county = collapse_ownership(own, "COUNTY_STATE_KEY")
    ownership_cbsa = collapse_ownership(own.dropna(subset=["CBSA_CODE"]), "CBSA_CODE")

    # Optional exact provider-level ownership match by normalized name/state/city.
    if name_col is not None:
        own["PROVIDER_NAME_KEY"] = own[name_col].map(normalize_name_key)
        own["PROVIDER_CITY_KEY"] = own[city_col].map(normalize_name_key) if city_col else pd.NA
        roster_names = hospital_locations.copy()
        roster_names["PROVIDER_NAME_KEY"] = roster_names["PROVIDER_NAME"].map(normalize_name_key)
        roster_names["PROVIDER_CITY_KEY"] = roster_names["PROVIDER_CITY"].map(normalize_name_key)
        ownership_hospital = roster_names.merge(
            own[["PROVIDER_NAME_KEY", "PROVIDER_CITY_KEY", "STATE_KEY", "OWNERSHIP_GROUP"]].drop_duplicates(),
            left_on=["PROVIDER_NAME_KEY", "PROVIDER_CITY_KEY", "PROVIDER_STATE"],
            right_on=["PROVIDER_NAME_KEY", "PROVIDER_CITY_KEY", "STATE_KEY"],
            how="left",
        )[["HOSPITAL_ID", "OWNERSHIP_GROUP"]].drop_duplicates("HOSPITAL_ID")

    ownership_county.to_csv(CONTROL_DIR / "HPT_OWNERSHIP_CONTROLS_COUNTY.csv", index=False)
    ownership_cbsa.to_csv(CONTROL_DIR / "HPT_OWNERSHIP_CONTROLS_CBSA.csv", index=False)
    if not ownership_hospital.empty:
        ownership_hospital.to_csv(CONTROL_DIR / "HPT_HOSPITAL_OWNERSHIP_MATCH.csv", index=False)

In [ ]:
# =============================================================================
# 10. CMS ENFORCEMENT CONTROLS AND LEGACY INSTRUMENTS
# =============================================================================


def add_rolling_enforcement(df: pd.DataFrame, geo_col: str) -> pd.DataFrame:
    out = df.sort_values([geo_col, "POST_MONTH"]).copy()
    out["any_enforcement"] = out[ENFORCEMENT_BASE_COLS].sum(axis=1)
    out["any_fine_or_warning"] = out[["fine", "warning"]].sum(axis=1)
    out["any_fine_or_closure"] = out[["fine", "closure_notice"]].sum(axis=1)
    out["any_warning_or_closure"] = out[["warning", "closure_notice"]].sum(axis=1)
    for col in ENFORCEMENT_ROLL_COLS:
        for window in SUPPORTING_WINDOWS + (1,):
            roll_col = f"{col}_roll_{window}m"
            out[roll_col] = (
                out.groupby(geo_col, sort=False)[col]
                .transform(lambda s: s.rolling(window=window, min_periods=1).sum())
            )
            out[f"{col}_ind_{window}m"] = (out[roll_col] > 0).astype("int8")
    for base in ENFORCEMENT_BASE_COLS:
        out[f"{base}_cum"] = out.groupby(geo_col, sort=False)[base].cumsum()
    # Current-month enforcement lags.
    for col in ENFORCEMENT_ROLL_COLS:
        for lag in (1, 2, 3):
            out[f"{col}_lag_{lag}m"] = (
                out.groupby(geo_col, sort=False)[col]
                .shift(lag)
                .fillna(0)
            )

    # Strictly lagged rolling-window variants. A one-month lag removes the
    # focal month from a rolling measure and is the preferred timing when an
    # enforcement variable is used as an instrument or predetermined control.
    for base in ENFORCEMENT_ROLL_COLS:
        for window in (3, 6, 9, 12):
            col = f"{base}_roll_{window}m"
            ind = f"{base}_ind_{window}m"
            if col in out.columns:
                out[f"{col}_lag_1m"] = (
                    out.groupby(geo_col, sort=False)[col]
                    .shift(1)
                    .fillna(0)
                )
                # Backward-compatible short alias used in earlier scripts.
                out[f"{col}_lag"] = out[f"{col}_lag_1m"]
            if ind in out.columns:
                out[f"{ind}_lag_1m"] = (
                    out.groupby(geo_col, sort=False)[ind]
                    .shift(1)
                    .fillna(0)
                    .astype("int8")
                )

    # Cumulative enforcement histories and whether a geography has ever been
    # subject to each action by month t.
    for base in ENFORCEMENT_BASE_COLS:
        out[f"{base}_ever"] = (out[f"{base}_cum"] > 0).astype("int8")

    return out


def balance_enforcement(
    events: pd.DataFrame,
    geo_col: str,
    all_geos: Sequence[object],
) -> pd.DataFrame:
    # The panel used for the rolling-window CALCULATION extends well before
    # STUDY_START, so every rolling window (up to 12 months) plus its
    # 1-month lag has genuine trailing history for every month in the
    # actual study window -- rather than a truncated, artificially-growing
    # window for the early months of the study period (e.g. a "12-month"
    # count for a July 2024 posting month would otherwise only be summing
    # 1 actual month of data, not 12, since nothing before STUDY_START
    # existed in the panel to sum over). 12-month window + 1-month lag = 13
    # months of required history; padded to 15 for margin.
    ENFORCEMENT_HISTORY_BUFFER_MONTHS = 15
    extended_start = STUDY_START - pd.DateOffset(months=ENFORCEMENT_HISTORY_BUFFER_MONTHS)
    months = pd.date_range(extended_start, STUDY_END, freq="MS")
    full = pd.MultiIndex.from_product(
        [pd.Series(all_geos).dropna().unique(), months],
        names=[geo_col, "POST_MONTH"],
    ).to_frame(index=False)
    agg = (
        events.groupby([geo_col, "POST_MONTH"], as_index=False)[ENFORCEMENT_BASE_COLS]
        .sum()
    )
    out = full.merge(agg, on=[geo_col, "POST_MONTH"], how="left")
    out[ENFORCEMENT_BASE_COLS] = out[ENFORCEMENT_BASE_COLS].fillna(0)
    out = add_rolling_enforcement(out, geo_col)
    # Trim back down to the actual study window for export -- the extended
    # pre-period existed only to give the rolling windows above genuine
    # trailing history; those pre-period rows are not part of the analysis
    # sample and should not be exported as if they were hospital-month
    # observations.
    out = out[
        (out["POST_MONTH"] >= STUDY_START) & (out["POST_MONTH"] <= STUDY_END)
    ].reset_index(drop=True)
    return out


def parse_enforcement_file(path: Path, geo_level: str, all_geos: Sequence[object]) -> pd.DataFrame:
    raw = read_csv_file(path)
    if geo_level == "COUNTY":
        geo_col = "COUNTY_STATE_KEY"
        if {"FINE", "WARNING", "CLOSURE_NOTICE", "POST_MONTH"}.issubset(raw.columns):
            raw[geo_col] = raw.get("COUNTY_STATE", raw.get("COUNTY_STATE_KEY"))
            raw[geo_col] = raw[geo_col].map(lambda x: normalize_county_state_key(x, None, None))
        else:
            action_col = next((c for c in raw.columns if c in {"ACTION", "ACTION_TYPE"}), None)
            date_col = next((c for c in raw.columns if c in {"DATE_OF_ACTION", "POST_MONTH", "DATE"}), None)
            county_col = next((c for c in raw.columns if c in {"COUNTY_NAME", "COUNTY"}), None)
            state_col = next((c for c in raw.columns if c in {"STATE", "STATE_CODE"}), None)
            if None in {action_col, date_col, county_col, state_col}:
                raise KeyError("County enforcement file lacks required action/date/county/state columns.")
            action_map = {
                "CMP NOTICE": "fine",
                "WARNING NOTICE": "warning",
                "CLOSURE NOTICE": "closure_notice",
            }
            raw["ACTION_STD"] = raw[action_col].map(normalize_text)
            raw["ACTION_TYPE_STD"] = raw["ACTION_STD"].map(action_map)
            raw = raw.loc[raw["ACTION_TYPE_STD"].notna()].copy()
            for col in ENFORCEMENT_BASE_COLS:
                raw[col] = (raw["ACTION_TYPE_STD"] == col).astype("int8")
            raw[geo_col] = [
                normalize_county_state_key(None, c, s)
                for c, s in zip(raw[county_col], raw[state_col])
            ]
            raw["POST_MONTH"] = month_start(raw[date_col])
    else:
        geo_col = "CBSA_CODE"
        raw[geo_col] = raw[next(c for c in raw.columns if c in {"CBSACODE", "CBSA_CODE"})].map(normalize_cbsa)
        if "POST_MONTH" not in raw.columns:
            date_col = next(c for c in raw.columns if c in {"DATE_OF_ACTION", "DATE"})
            raw["POST_MONTH"] = month_start(raw[date_col])
        else:
            raw["POST_MONTH"] = month_start(raw["POST_MONTH"])
        for col in ENFORCEMENT_BASE_COLS:
            if col.upper() in raw.columns and col not in raw.columns:
                raw[col] = raw[col.upper()]
            if col not in raw.columns:
                raw[col] = 0
    raw["POST_MONTH"] = month_start(raw["POST_MONTH"])
    raw = raw.dropna(subset=[geo_col, "POST_MONTH"])
    return balance_enforcement(raw, geo_col, all_geos)


enforcement_county = pd.DataFrame()
enforcement_cbsa = pd.DataFrame()
if BUILD_ENFORCEMENT_CONTROLS:
    county_path = find_file(RAW_FILES["enforcement_county"], SEARCH_DIRS_RAW)
    cbsa_path = find_file(RAW_FILES["enforcement_cbsa"], SEARCH_DIRS_RAW)
    all_counties = hospital_locations["COUNTY_STATE_KEY"].dropna().unique()
    all_cbsas = hospital_locations["CBSA_CODE"].dropna().unique()
    enforcement_county = parse_enforcement_file(county_path, "COUNTY", all_counties)
    enforcement_cbsa = parse_enforcement_file(cbsa_path, "CBSA", all_cbsas)
    enforcement_county.to_csv(CONTROL_DIR / "HPT_ENFORCEMENT_COUNTY_MONTH.csv", index=False)
    enforcement_cbsa.to_csv(CONTROL_DIR / "HPT_ENFORCEMENT_CBSA_MONTH.csv", index=False)


    enforcement_dictionary_rows: list[dict[str, object]] = []
    for geography, frame, key_col, panel_prefix in [
        ("COUNTY", enforcement_county, "COUNTY_STATE_KEY", "COUNTY_ENF_"),
        ("CBSA", enforcement_cbsa, "CBSA_CODE", "CBSA_ENF_"),
    ]:
        for source_col in frame.columns:
            if source_col in {key_col, "POST_MONTH"}:
                continue

            upper_col = source_col.upper()
            output_col = panel_prefix + upper_col

            if "_ROLL_" in upper_col:
                measure_type = "ROLLING_COUNT"
            elif "_IND_" in upper_col:
                measure_type = "ROLLING_INDICATOR"
            elif "_CUM" in upper_col:
                measure_type = "CUMULATIVE_COUNT"
            elif "_EVER" in upper_col:
                measure_type = "EVER_EXPOSED_INDICATOR"
            elif "_LAG_" in upper_col or upper_col.endswith("_LAG"):
                measure_type = "LAGGED_MEASURE"
            else:
                measure_type = "CURRENT_MONTH_COUNT_OR_COMPOSITE"

            enforcement_dictionary_rows.append({
                "geography": geography,
                "source_column": source_col,
                "panel_column": output_col,
                "measure_type": measure_type,
                "strictly_predetermined_candidate": int(
                    "_LAG_1M" in upper_col or upper_col.endswith("_LAG")
                ),
                "description": (
                    f"{geography.title()}-level CMS hospital price-transparency "
                    f"enforcement measure derived from {source_col}."
                ),
            })

    pd.DataFrame(enforcement_dictionary_rows).to_csv(
        INSTRUMENT_DIR / "HPT_ENFORCEMENT_VARIABLE_DICTIONARY.csv",
        index=False,
    )

In [ ]:
# =============================================================================
# 11. MERGE CONTROLS, CLASSIFICATION, AND ANALYSIS FLAGS
# =============================================================================


def merge_controls(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Merge controls and construct exact-code analysis flags.

    REBUILT: there is no pre-baked shoppability scheme file to join anymore.
    This attaches the raw, objective codebook attributes a shoppability
    definition needs (OPPS status indicator, ASC-eligibility, add-on flag,
    medical/surgical type) for any code role -- standalone or component --
    since these are objective facts about a code, not a judgment call that
    needs role-based gating the way the old baked-in scheme did. Building
    the actual shoppability scheme(s) -- potentially several, for the
    meta-regression -- is a deliberately separate step from this function.
    """

    out = df.copy()
    msg(f"    merge_controls: starting, {len(out):,} rows")

    if "SYSTEM_KEY" not in out.columns:
        out["SYSTEM_KEY"] = create_system_key(out)

    out["YEAR_FOR_CONTROLS"] = (
        out["POST_MONTH"]
        .dt.year
    )

    if max_acs_year is not None:
        out["YEAR_FOR_CONTROLS"] = (
            out["YEAR_FOR_CONTROLS"]
            .clip(upper=max_acs_year)
        )

    if BUILD_ACS_CONTROLS:
        msg("    merge_controls: ACS controls ...")
        county_acs = acs_county_controls.rename(
            columns={
                "YEAR": "YEAR_FOR_CONTROLS",
                **{
                    column:
                        f"COUNTY_{column.upper()}"
                    for column in DEMO_COLS
                },
            }
        )

        cbsa_acs = acs_cbsa_controls.rename(
            columns={
                "YEAR": "YEAR_FOR_CONTROLS",
                **{
                    column:
                        f"CBSA_{column.upper()}"
                    for column in DEMO_COLS
                },
            }
        )

        out = safe_left_merge(
            out,
            county_acs,
            on=[
                "COUNTY_FIPS",
                "YEAR_FOR_CONTROLS",
            ],
            label="county ACS",
        )

        out = safe_left_merge(
            out,
            cbsa_acs,
            on=[
                "CBSA_CODE",
                "YEAR_FOR_CONTROLS",
            ],
            label="CBSA ACS",
        )

    if BUILD_OWNERSHIP_CONTROLS:
        msg("    merge_controls: ownership controls ...")
        county_own = ownership_county.rename(
            columns={
                column:
                    "COUNTY_"
                    + column.upper().replace("-", "_")
                for column in OWNERSHIP_COLS
            }
        )

        cbsa_own = ownership_cbsa.rename(
            columns={
                column:
                    "CBSA_"
                    + column.upper().replace("-", "_")
                for column in OWNERSHIP_COLS
            }
        )

        out = safe_left_merge(
            out,
            county_own,
            on=["COUNTY_STATE_KEY"],
            label="county ownership",
        )

        out = safe_left_merge(
            out,
            cbsa_own,
            on=["CBSA_CODE"],
            label="CBSA ownership",
        )

        if not ownership_hospital.empty:
            out = safe_left_merge(
                out,
                ownership_hospital,
                on=["HOSPITAL_ID"],
                label="hospital ownership",
            )

            if "OWNERSHIP_GROUP" in out.columns:
                out["HOSPITAL_OWNERSHIP_GROUP"] = (
                    out["OWNERSHIP_GROUP"]
                )

    if BUILD_ENFORCEMENT_CONTROLS:
        msg("    merge_controls: enforcement controls ...")
        county_enf = enforcement_county.rename(
            columns={
                column:
                    f"COUNTY_ENF_{column.upper()}"
                for column in enforcement_county.columns
                if column
                not in {
                    "COUNTY_STATE_KEY",
                    "POST_MONTH",
                }
            }
        )

        cbsa_enf = enforcement_cbsa.rename(
            columns={
                column:
                    f"CBSA_ENF_{column.upper()}"
                for column in enforcement_cbsa.columns
                if column
                not in {
                    "CBSA_CODE",
                    "POST_MONTH",
                }
            }
        )

        out = safe_left_merge(
            out,
            county_enf,
            on=[
                "COUNTY_STATE_KEY",
                "POST_MONTH",
            ],
            label="county enforcement",
        )

        out = safe_left_merge(
            out,
            cbsa_enf,
            on=[
                "CBSA_CODE",
                "POST_MONTH",
            ],
            label="CBSA enforcement",
        )

        for suffix in [
            "3M",
            "6M",
            "9M",
            "12M",
        ]:
            source = (
                f"COUNTY_ENF_FINE_ROLL_{suffix}"
            )

            if source in out.columns:
                out[
                    f"LEGACY_FINE_ROLL_{suffix}"
                ] = out[source]

        if (
            "COUNTY_ENF_FINE_ROLL_6M_LAG"
            in out.columns
        ):
            out[
                "LEGACY_FINE_ROLL_6M_LAG"
            ] = out[
                "COUNTY_ENF_FINE_ROLL_6M_LAG"
            ]

    # Attach codebook attributes not already present on this frame. Several
    # (CONTRAST_VARIANT, IS_CMS70_CODE, DRG_ACUITY_KEYWORD_TAG) were already
    # carried through Phase 3/4 and are on `out` already -- only the ones
    # that were not are pulled from the codebook here, keyed on the exact
    # billing code (codebook attributes are properties of the code itself,
    # not of the concept/family it rolls into).
    codebook_attrs = [
        "OPPS_STATUS_INDICATORS",
        "IS_ASC_COVERED_FLAG",
        "IS_NCCI_ADDON_FLAG",
        "MEDICAL_SURGICAL_TYPE",
    ]
    codebook_cols_needed = [
        c for c in codebook_attrs
        if c in codebook.columns and c not in out.columns
    ]
    if codebook_cols_needed:
        msg("    merge_controls: codebook attributes ...")
        codebook_slice = (
            codebook[["BILLING_CODE_TYPE", "BILLING_CODE", *codebook_cols_needed]]
            .drop_duplicates(["BILLING_CODE_TYPE", "BILLING_CODE"])
        )
        out = safe_left_merge(
            out,
            codebook_slice,
            on=["BILLING_CODE_TYPE", "BILLING_CODE"],
            label="codebook shoppability attributes",
        )

    msg("    merge_controls: derived IDs and sample flags ...")
    out["LOG_TOTAL_BEDS"] = np.log(
        pd.to_numeric(
            out["TOTAL_BEDS"],
            errors="coerce",
        ).clip(lower=1)
    )

    out["MARKET_ID_COUNTY_EXACT"] = (
        out["COUNTY_STATE_KEY"].astype("string")
        + "::"
        + out["BILLING_CODE_TYPE"].astype("string")
        + "::"
        + out["BILLING_CODE"].astype("string")
    )

    out["MARKET_ID_CITY_EXACT"] = (
        out["CITY_STATE_KEY"].astype("string")
        + "::"
        + out["BILLING_CODE_TYPE"].astype("string")
        + "::"
        + out["BILLING_CODE"].astype("string")
    )

    out["MARKET_ID_CBSA_EXACT"] = (
        out["CBSA_CODE"].astype("string")
        + "::"
        + out["BILLING_CODE_TYPE"].astype("string")
        + "::"
        + out["BILLING_CODE"].astype("string")
    )

    out["FINAL_CONCEPT_KEY"] = build_composite_service_id(
        out,
        phase4_concept_key_columns(),
    )

    out["FINAL_FAMILY_KEY"] = build_composite_service_id(
        out,
        phase4_family_key_columns(),
    )

    out["MARKET_ID_COUNTY_CONCEPT"] = (
        out["COUNTY_STATE_KEY"].astype("string")
        + "::"
        + out["FINAL_CONCEPT_KEY"]
    )

    out["MARKET_ID_COUNTY_FAMILY"] = (
        out["COUNTY_STATE_KEY"].astype("string")
        + "::"
        + out["FINAL_FAMILY_KEY"]
    )

    out["COUNTY_MONTH_ID"] = (
        out["COUNTY_STATE_KEY"].astype("string")
        + "::"
        + out["POST_MONTH"].astype("string")
    )

    out["CBSA_MONTH_ID"] = (
        out["CBSA_CODE"].astype("string")
        + "::"
        + out["POST_MONTH"].astype("string")
    )

    out["SYSTEM_MONTH_ID"] = (
        out["SYSTEM_KEY"].astype("string")
        + "::"
        + out["POST_MONTH"].astype("string")
    )

    out["SAMPLE_HOSPITAL_FIRST_POST"] = (
        out["IS_HOSPITAL_FIRST_POST_MONTH"]
        .fillna(0)
        .astype("int8")
    )

    out["SAMPLE_EXACT_ENTRY"] = (
        out["IS_EXACT_CODE_ENTRY_MONTH"]
        .astype("int8")
    )

    out["SAMPLE_VALID_PRIMARY_PRICE"] = (
        out["MEDIAN_PRICE"]
        .notna()
        .astype("int8")
    )

    out["SAMPLE_FIVE_PLUS_PAYER_CELLS"] = (
        out["N_PAYER_CELLS"] >= 5
    ).astype("int8")

    out["SAMPLE_TEN_PLUS_PAYER_CELLS"] = (
        out["N_PAYER_CELLS"] >= 10
    ).astype("int8")

    if "N_DISTINCT_PAYERS" in out.columns:
        out[
            "SAMPLE_THREE_PLUS_DISTINCT_PAYERS"
        ] = (
            out["N_DISTINCT_PAYERS"] >= 3
        ).astype("int8")

        out[
            "SAMPLE_FIVE_PLUS_DISTINCT_PAYERS"
        ] = (
            out["N_DISTINCT_PAYERS"] >= 5
        ).astype("int8")

        # Backward-compatible name used by the coverage section.
        out["SAMPLE_FIVE_PLUS_PAYERS"] = (
            out["SAMPLE_FIVE_PLUS_DISTINCT_PAYERS"]
        )

    out["SAMPLE_PREFERRED_SUPPORT"] = (
        (out["N_PAYER_CELLS"] >= 5)
        &
        (
            pd.to_numeric(
                out["N_DISTINCT_PAYERS"],
                errors="coerce",
            ) >= 3
        )
    ).astype("int8")

    out["SAMPLE_STRONG_SUPPORT"] = (
        (out["N_PAYER_CELLS"] >= 10)
        &
        (
            pd.to_numeric(
                out["N_DISTINCT_PAYERS"],
                errors="coerce",
            ) >= 5
        )
    ).astype("int8")

    if "STRICT_PRICE_AVAILABLE_FLAG" in out.columns:
        out[
            "SAMPLE_STRICT_PRICE_AVAILABLE"
        ] = (
            out[
                "STRICT_PRICE_AVAILABLE_FLAG"
            ]
            .fillna(0)
            .astype("int8")
        )

    out["SAMPLE_BASELINE_COUNTY"] = (
        (out["IS_EXACT_CODE_ENTRY_MONTH"] == 1)
        & out["MEDIAN_PRICE"].notna()
        & out["COUNTY_STATE_KEY"].notna()
        & out[
            "Z_SYS_STRICT_9M_EXCL_CURRENT"
        ].notna()
    ).astype("int8")

    out[
        "SAMPLE_BASELINE_COUNTY_POSITIVE_PRIOR"
    ] = (
        (out["SAMPLE_BASELINE_COUNTY"] == 1)
        &
        (
            out[
                "N_PRIOR_POSTERS_COUNTY_EXACT"
            ] > 0
        )
    ).astype("int8")

    out["SAMPLE_COHORT_COUNTY"] = (
        (out["IS_HOSPITAL_FIRST_POST_MONTH"] == 1)
        & out["MEDIAN_PRICE"].notna()
        & out["COUNTY_STATE_KEY"].notna()
    ).astype("int8")

    out["SAMPLE_COHORT_COUNTY_PREFERRED"] = (
        (out["SAMPLE_COHORT_COUNTY"] == 1)
        & (out["SAMPLE_PREFERRED_SUPPORT"] == 1)
    ).astype("int8")

    out["SAMPLE_COHORT_COUNTY_STRONG"] = (
        (out["SAMPLE_COHORT_COUNTY"] == 1)
        & (out["SAMPLE_STRONG_SUPPORT"] == 1)
    ).astype("int8")

    msg(f"    merge_controls: done, {len(out):,} rows x {out.shape[1]} columns")
    return out


msg(
    "Merging controls and constructing exact-code analysis flags ..."
)

def merge_controls_chunked(df: pd.DataFrame, chunk_size: int = 150_000) -> pd.DataFrame:
    """
    Run merge_controls on a DataFrame in row-count-bounded chunks.

    Safe to do arbitrarily (not grouped by any key) because merge_controls
    never aggregates within the frame -- every operation is either a
    row-wise computation or a left-join against a small external lookup
    table (ACS, ownership, enforcement, codebook). Splitting into pieces
    and concatenating the results is mathematically identical to running it
    on the whole frame at once; it only bounds peak memory per chunk,
    which is the actual problem at this frame width (many prior-poster and
    instrument columns already added in cells 6-8, on top of the original
    price data).
    """
    if df.empty:
        return df.copy()

    n_chunks = max(1, math.ceil(len(df) / chunk_size))
    pieces = []
    for i in range(n_chunks):
        chunk = df.iloc[i * chunk_size:(i + 1) * chunk_size].copy()
        msg(f"    chunk {i + 1}/{n_chunks} ({len(chunk):,} rows) ...")
        pieces.append(merge_controls(chunk))
        del chunk
        gc.collect()
    result = pd.concat(pieces, ignore_index=True, sort=False)
    del pieces
    gc.collect()
    return result


# Each frame's source is freed immediately after producing its enriched
# "master" version, and each frame is itself processed in row-bounded
# chunks (not all 1.6M+/1.7M+ rows through merge_controls at once) -- by
# this point these frames are much wider than when they started (cells 6-8
# added dozens of prior-poster and instrument columns), and that width is
# what makes even one frame, processed whole, a crash risk on a laptop.
msg("  Processing outpatient (CPT/HCPCS) frame ...")
outpatient_master = merge_controls_chunked(outpatient_exact)
del outpatient_exact
gc.collect()

msg("  Processing inpatient (MS-DRG) frame ...")
inpatient_master = merge_controls_chunked(inpatient_exact)
del inpatient_exact
gc.collect()

msg("  Processing component (add-on) frame ...")
if not exact_component.empty:
    exact_component_master = merge_controls_chunked(exact_component)
else:
    exact_component_master = pd.DataFrame()
del exact_component
gc.collect()

# Save the codebook itself as the reference for building shoppability
# schemes downstream (in this notebook or in R) -- this replaces the old
# single pre-baked "HPT_SHOPPABILITY_SCHEMES_LONG.csv" scheme file. Every
# column needed to construct a scheme (OPPS_STATUS_INDICATORS,
# IS_ASC_COVERED_FLAG, IS_NCCI_ADDON_FLAG, MEDICAL_SURGICAL_TYPE,
# IS_CMS70_CODE, DRG_ACUITY_KEYWORD_TAG, CONTRAST_VARIANT) is on it.
codebook.to_csv(
    R_PANEL_DIR / "HPT_CODEBOOK_FOR_SHOPPABILITY_SCHEMES.csv",
    index=False,
)


# outpatient_exact / inpatient_exact / exact_component were already deleted
# above, immediately after each was consumed -- nothing left to release here.
gc.collect()

msg(
    "Released pre-master exact-code DataFrames before Section 12."
)


In [ ]:
# =============================================================================
# 12. LOW-MEMORY CONCEPT-, STRICT-CONCEPT-, AND FAMILY-LEVEL MEASURES
# =============================================================================
#
# This section is deliberately disk-backed. Concatenating both wide exact-code
# master panels and then holding concepts, strict concepts, family prices, and
# several merge copies in memory at once will exhaust RAM on a 16 GB or 24 GB
# machine, and the operating system terminates the kernel without a Python
# traceback when it does -- a failure with no error message to debug.
#
# The low-memory design:
#   1. caches exact-code master panels to Parquet;
#   2. creates narrow hospital-month and concept/family treatment lookups;
#   3. releases the wide exact-code panels from RAM;
#   4. enriches one billing-code domain at a time;
#   5. writes each enriched panel to a temporary Parquet cache;
#   6. releases each completed panel before starting the next one.
# =============================================================================

WORKING_PANEL_DIR = (
    Path.home()
    / "Library"
    / "Caches"
    / "HPT_Python_Working_Panels"
)

WORKING_PANEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Remove stale Parquet files from a prior interrupted Section 12 run.
for _stale_file in WORKING_PANEL_DIR.glob("*.parquet"):
    _stale_file.unlink()

msg(
    f"Cleared stale working-panel cache: {WORKING_PANEL_DIR}"
)


def aggregation_keys(
    aggregation: str,
) -> list[str]:
    aggregation = aggregation.upper()

    if aggregation == "CONCEPT":
        return [
            "HOSPITAL_ID",
            "POST_MONTH",
            *phase4_concept_key_columns(),
        ]

    if aggregation == "FAMILY":
        return [
            "HOSPITAL_ID",
            "POST_MONTH",
            *phase4_family_key_columns(),
        ]

    raise ValueError(
        f"Unsupported aggregation: {aggregation}"
    )


STABLE_HOSPITAL_MONTH_COLUMNS = {
    "COUNTY_STATE_KEY",
    "COUNTY_FIPS",
    "COUNTY_GEOGRAPHY_METHOD",
    "CBSA_CODE",
    "CITY_STATE_KEY",
    "SYSTEM_KEY",
    "COUNTY_MONTH_ID",
    "CBSA_MONTH_ID",
    "SYSTEM_MONTH_ID",
    "HOSPITAL_OWNERSHIP_GROUP",
    "YEAR_FOR_CONTROLS",
    "LOG_TOTAL_BEDS",
    "HOSPITAL_FIRST_POST_MONTH",
    "HOSPITAL_LAST_POST_MONTH",
    "N_OBSERVED_POST_MONTHS",
    "SINGLE_POST_MONTH_FLAG",
    "MULTIPLE_POST_MONTH_FLAG",
    "IS_HOSPITAL_FIRST_POST_MONTH",
    "DEC2024_OR_LATER_POSTING_COHORT_FLAG",
    "JAN2025_OR_LATER_POSTING_COHORT_FLAG",
    "COUNTY_N_PRIOR_POSTERS",
    "COUNTY_N_SAME_COHORT_POSTERS",
    "COUNTY_N_CUMULATIVE_POSTERS",
    "COUNTY_FINAL_TOTAL_POSTERS",
    "COUNTY_POSTING_SHARE_REACHED",
    "COUNTY_POSTING_COHORT_ORDER",
    "COUNTY_FIRST_POST_MONTH",
    "COUNTY_MONTHS_SINCE_FIRST_POST",
    "CBSA_N_PRIOR_POSTERS",
    "CBSA_N_SAME_COHORT_POSTERS",
    "CBSA_N_CUMULATIVE_POSTERS",
    "CBSA_FINAL_TOTAL_POSTERS",
    "CBSA_POSTING_SHARE_REACHED",
    "CBSA_POSTING_COHORT_ORDER",
    "CBSA_FIRST_POST_MONTH",
    "CBSA_MONTHS_SINCE_FIRST_POST",
}


def is_stable_hospital_month_column(
    column: str,
) -> bool:
    return (
        column in STABLE_HOSPITAL_MONTH_COLUMNS
        or column.startswith("Z_SYS_")
        or column.startswith("COUNTY_ENF_")
        or column.startswith("CBSA_ENF_")
        or column.startswith("LEGACY_FINE_")
        or column.startswith("COUNTY_DEMO_")
        or column.startswith("CBSA_DEMO_")
        or column.startswith("COUNTY_OWN_")
        or column.startswith("CBSA_OWN_")
    )


def is_level_measure_column(
    column: str,
    aggregation: str,
) -> bool:
    aggregation = aggregation.upper()

    treatment_prefix = (
        column.startswith("N_PRIOR_POSTERS_")
        or column.startswith("N_CUMULATIVE_POSTERS_")
        or column.startswith("N_POSTERS_THIS_MONTH_")
        or column.startswith("IS_FIRST_WAVE_")
    )

    if treatment_prefix and column.endswith(
        f"_{aggregation}"
    ):
        return True

    expected_entry = {
        "CONCEPT": "IS_CONCEPT_ENTRY_MONTH",
        "FAMILY": "IS_FAMILY_ENTRY_MONTH",
    }[aggregation]

    return column == expected_entry


def make_first_row_lookup(
    master: pd.DataFrame,
    keys: list[str],
    value_columns: list[str],
    label: str,
) -> pd.DataFrame:
    """
    Build a narrow unique lookup using only the first row for each key.

    The values selected here were already attached to every exact-code row by
    earlier validated merges. Using drop_duplicates on the key alone avoids
    hashing every wide control column across 1.7 million rows.
    """

    available_values = [
        column
        for column in value_columns
        if column in master.columns
        and column not in keys
    ]

    selected = (
        keys
        + list(dict.fromkeys(available_values))
    )

    lookup = (
        master.loc[:, selected]
        .drop_duplicates(
            subset=keys,
            keep="first",
        )
        .reset_index(drop=True)
    )

    duplicate_keys = int(
        lookup.duplicated(
            keys,
            keep=False,
        ).sum()
    )

    if duplicate_keys:
        raise RuntimeError(
            f"{label} contains {duplicate_keys:,} "
            "duplicate lookup keys."
        )

    return lookup


def write_lookup(
    frame: pd.DataFrame,
    path: Path,
) -> None:
    write_parquet(
        frame,
        path,
    )
    del frame
    gc.collect()


def read_filtered_parquet(
    path: Path,
    billing_code_type: str | None = None,
) -> pd.DataFrame:
    """Read one cached panel, optionally using a Parquet predicate filter."""

    if billing_code_type is None:
        return pd.read_parquet(path)

    try:
        return pd.read_parquet(
            path,
            filters=[
                (
                    "BILLING_CODE_TYPE",
                    "==",
                    billing_code_type,
                )
            ],
        )

    except (TypeError, ValueError):
        frame = pd.read_parquet(path)

        out = frame.loc[
            frame["BILLING_CODE_TYPE"]
            == billing_code_type
        ].copy()

        del frame
        gc.collect()

        return out


def merge_only_new_columns(
    left: pd.DataFrame,
    right: pd.DataFrame,
    keys: list[str],
    label: str,
    validate: str,
) -> pd.DataFrame:
    additions = [
        column
        for column in right.columns
        if column not in keys
        and column not in left.columns
    ]

    if not additions:
        return left

    reduced_right = right[
        keys + additions
    ]

    return safe_left_merge(
        left,
        reduced_right,
        on=keys,
        label=label,
        validate=validate,
    )


# -----------------------------------------------------------------------------
# 12A. Cache the enriched exact-code masters and construct narrow lookups.
# -----------------------------------------------------------------------------

exact_master_cache_paths = {
    "outpatient": (
        WORKING_PANEL_DIR
        / "outpatient_exact_master.parquet"
    ),
    "inpatient": (
        WORKING_PANEL_DIR
        / "inpatient_exact_master.parquet"
    ),
    "component": (
        WORKING_PANEL_DIR
        / "component_exact_master.parquet"
    ),
}

# NOTE: "main_concept" and "strict_concept" keys are kept even though this
# pipeline has no separate strict-tier table -- the source parquet for
# "strict_concept" is simply never written below, so the .exists() check
# further down gracefully skips that entire (dead) code path rather than
# needing every strict-related reference removed throughout this cell.
source_panel_cache_paths = {
    "main_concept": (
        WORKING_PANEL_DIR
        / "source_main_concept.parquet"
    ),
    "strict_concept": (
        WORKING_PANEL_DIR
        / "source_strict_concept.parquet"
    ),
    "family": (
        WORKING_PANEL_DIR
        / "source_family.parquet"
    ),
}

lookup_cache_paths = {
    "outpatient_stable": (
        WORKING_PANEL_DIR
        / "lookup_outpatient_hospital_month.parquet"
    ),
    "inpatient_stable": (
        WORKING_PANEL_DIR
        / "lookup_inpatient_hospital_month.parquet"
    ),
    "outpatient_concept": (
        WORKING_PANEL_DIR
        / "lookup_outpatient_concept.parquet"
    ),
    "inpatient_concept": (
        WORKING_PANEL_DIR
        / "lookup_inpatient_concept.parquet"
    ),
    "outpatient_family": (
        WORKING_PANEL_DIR
        / "lookup_outpatient_family.parquet"
    ),
    "inpatient_family": (
        WORKING_PANEL_DIR
        / "lookup_inpatient_family.parquet"
    ),
}

msg(
    "\nCaching exact-code master panels and building narrow lookups ..."
)

# Each wide global is deleted immediately after it's cached, rather than
# holding outpatient_master + inpatient_master + exact_component_master +
# concept_prices + family_prices all in memory simultaneously until one
# batch cleanup at the end. Peak memory during this section is what carries
# forward into the enrichment phase below, so keeping it lean here matters
# even though every individual write succeeded on its own in testing.

for domain in ["outpatient", "inpatient"]:
    master = globals()[f"{domain}_master"]

    write_parquet(
        master,
        exact_master_cache_paths[domain],
    )

    stable_values = [
        column
        for column in master.columns
        if is_stable_hospital_month_column(
            column
        )
    ]

    stable_lookup = make_first_row_lookup(
        master=master,
        keys=[
            "HOSPITAL_ID",
            "POST_MONTH",
        ],
        value_columns=stable_values,
        label=f"{domain} hospital-month lookup",
    )

    write_lookup(
        stable_lookup,
        lookup_cache_paths[
            f"{domain}_stable"
        ],
    )

    for aggregation in [
        "CONCEPT",
        "FAMILY",
    ]:
        level_values = [
            column
            for column in master.columns
            if is_level_measure_column(
                column,
                aggregation,
            )
        ]

        level_lookup = make_first_row_lookup(
            master=master,
            keys=aggregation_keys(
                aggregation
            ),
            value_columns=level_values,
            label=(
                f"{domain} "
                f"{aggregation.lower()} lookup"
            ),
        )

        write_lookup(
            level_lookup,
            lookup_cache_paths[
                f"{domain}_{aggregation.lower()}"
            ],
        )

    del master
    del globals()[f"{domain}_master"]
    gc.collect()
    msg(f"  {domain}_master released from RAM.")

if not exact_component_master.empty:
    write_parquet(
        exact_component_master,
        exact_master_cache_paths["component"],
    )
del exact_component_master
gc.collect()

write_parquet(
    concept_prices,
    source_panel_cache_paths["main_concept"],
)
del concept_prices
gc.collect()

# strict_concept has no source in this pipeline (no separate strict-tier
# table was built) -- its parquet is deliberately never written, so the
# .exists() check further down finds nothing and skips that whole path.

write_parquet(
    family_prices,
    source_panel_cache_paths["family"],
)
del family_prices
gc.collect()

# All wide source and exact-master DataFrames are now safely cached; the
# per-domain loop above already deleted outpatient_master/inpatient_master,
# and the three lines above deleted the rest immediately after their own
# write. This final pass is just a safety net for anything still lingering.
for _name in [
    "outpatient_master",
    "inpatient_master",
    "exact_component_master",
    "concept_prices",
    "strict_concept",
    "family_prices",
]:
    if _name in globals():
        del globals()[_name]

gc.collect()

msg(
    "Wide source panels released from RAM. "
    "Beginning domain-at-a-time enrichment ..."
)


# -----------------------------------------------------------------------------
# 12B. Shoppability: no scheme table exists in this pipeline (see cell 13 --
# schemes are built explicitly from the codebook's raw attribute columns,
# not joined from a single pre-baked table), so there is nothing to attach
# here. The old shop_lookup construction and its merge inside
# enrich_concept_domain are both removed below.
# -----------------------------------------------------------------------------


def _enrich_concept_chunk(
    chunk: pd.DataFrame,
    *,
    stable_lookup: pd.DataFrame,
    concept_lookup: pd.DataFrame,
    domain: str,
    strict: bool,
) -> pd.DataFrame:
    """Apply the two lookup merges and derived columns to one row-chunk."""

    chunk = merge_only_new_columns(
        chunk,
        stable_lookup,
        keys=["HOSPITAL_ID", "POST_MONTH"],
        label=(
            f"{domain} {'strict ' if strict else ''}"
            "concept hospital-month measures"
        ),
        validate="m:1",
    )

    chunk = merge_only_new_columns(
        chunk,
        concept_lookup,
        keys=aggregation_keys("CONCEPT"),
        label=(
            f"{domain} {'strict ' if strict else ''}"
            "concept treatment measures"
        ),
        validate="1:1",
    )

    # No shoppability merge here (see Section 12B note above).

    if strict:
        strict_aliases = {
            "STRICT_MEDIAN_PRICE": "MEDIAN_PRICE",
            "STRICT_MEAN_PRICE": "MEAN_PRICE",
            "STRICT_MIN_PRICE": "MIN_PRICE",
            "STRICT_MAX_PRICE": "MAX_PRICE",
            "STRICT_P25_PRICE": "P25_PRICE",
            "STRICT_P75_PRICE": "P75_PRICE",
            "STRICT_IQR_PRICE": "IQR_PRICE",
        }
        for source, target in strict_aliases.items():
            chunk[target] = chunk[source]
    else:
        chunk["SAMPLE_CONCEPT_ENTRY"] = (
            chunk["IS_CONCEPT_ENTRY_MONTH"]
            .fillna(0)
            .astype("int8")
        )

        chunk["SAMPLE_COHORT_CONCEPT"] = (
            (chunk["IS_HOSPITAL_FIRST_POST_MONTH"] == 1)
            & chunk["MEDIAN_PRICE"].notna()
            & chunk["COUNTY_STATE_KEY"].notna()
        ).astype("int8")

        # PREFERRED_CONCEPT_SUPPORT_FLAG and STRONG_CONCEPT_SUPPORT_FLAG are
        # not built in Phase 4 SQL and are not on the concept table. The two
        # derived columns that depend on them are guarded rather than assumed
        # present, so their absence skips the derivation instead of raising.
        # N_CODES_IN_CONCEPT is available as a rough support proxy.
        if "PREFERRED_CONCEPT_SUPPORT_FLAG" in chunk.columns:
            chunk["SAMPLE_COHORT_CONCEPT_PREFERRED"] = (
                (chunk["SAMPLE_COHORT_CONCEPT"] == 1)
                & (chunk["PREFERRED_CONCEPT_SUPPORT_FLAG"] == 1)
            ).astype("int8")

        if "STRONG_CONCEPT_SUPPORT_FLAG" in chunk.columns:
            chunk["SAMPLE_COHORT_CONCEPT_STRONG"] = (
                (chunk["SAMPLE_COHORT_CONCEPT"] == 1)
                & (chunk["STRONG_CONCEPT_SUPPORT_FLAG"] == 1)
            ).astype("int8")

    return chunk


def enrich_concept_domain(
    *,
    billing_code_type: str,
    domain: str,
    strict: bool,
    output_path: Path,
    chunk_size: int = 150_000,
) -> dict[str, object]:
    """
    CHUNKED: the merges here add hundreds of columns from the stable and
    concept lookups (the stable lookup alone has 435 columns, from all the
    ACS/ownership/enforcement/instrument columns cell 13 attached). Running
    this on the full 1.4-1.8M row filtered panel at once is the same class
    of memory risk that required chunking in cell 13 -- succeeding once for
    outpatient doesn't guarantee headroom remains for inpatient right after.
    """
    source_key = (
        "strict_concept"
        if strict
        else "main_concept"
    )

    panel = read_filtered_parquet(
        source_panel_cache_paths[source_key],
        billing_code_type=billing_code_type,
    )

    stable_lookup = pd.read_parquet(
        lookup_cache_paths[f"{domain}_stable"]
    )
    concept_lookup = pd.read_parquet(
        lookup_cache_paths[f"{domain}_concept"]
    )

    n_chunks = max(1, math.ceil(len(panel) / chunk_size))
    enriched_chunks = []
    for i in range(n_chunks):
        chunk = panel.iloc[i * chunk_size:(i + 1) * chunk_size].copy()
        msg(f"    {domain} concept chunk {i + 1}/{n_chunks} ({len(chunk):,} rows) ...")
        chunk = _enrich_concept_chunk(
            chunk,
            stable_lookup=stable_lookup,
            concept_lookup=concept_lookup,
            domain=domain,
            strict=strict,
        )
        enriched_chunks.append(chunk)
        del chunk
        gc.collect()

    del panel, stable_lookup, concept_lookup
    gc.collect()

    panel = pd.concat(enriched_chunks, ignore_index=True, sort=False)
    del enriched_chunks
    gc.collect()

    qa_rows.append(
        unique_key_qa(
            panel,
            aggregation_keys("CONCEPT"),
            (
                f"{domain}_"
                f"{'strict_' if strict else ''}"
                "concept_panel"
            ),
        )
    )

    summary = {
        "panel": (
            f"{domain}_"
            f"{'strict_' if strict else ''}"
            "concept"
        ),
        "rows": len(panel),
        "hospitals": int(
            panel["HOSPITAL_ID"].nunique()
        ),
        "service_ids": int(
            panel[
                phase4_concept_key_columns()
            ]
            .drop_duplicates()
            .shape[0]
        ),
        "path": str(output_path),
    }

    write_parquet(
        panel,
        output_path,
    )

    del panel
    gc.collect()

    return summary


enriched_panel_cache_paths = {
    "outpatient_concept": (
        WORKING_PANEL_DIR
        / "enriched_outpatient_concept.parquet"
    ),
    "inpatient_concept": (
        WORKING_PANEL_DIR
        / "enriched_inpatient_concept.parquet"
    ),
    "outpatient_strict_concept": (
        WORKING_PANEL_DIR
        / "enriched_outpatient_strict_concept.parquet"
    ),
    "inpatient_strict_concept": (
        WORKING_PANEL_DIR
        / "enriched_inpatient_strict_concept.parquet"
    ),
    "family": (
        WORKING_PANEL_DIR
        / "enriched_family.parquet"
    ),
}

panel_run_summary: list[dict[str, object]] = []

panel_run_summary.append(
    enrich_concept_domain(
        billing_code_type="CPT_HCPCS",
        domain="outpatient",
        strict=False,
        output_path=(
            enriched_panel_cache_paths[
                "outpatient_concept"
            ]
        ),
    )
)

panel_run_summary.append(
    enrich_concept_domain(
        billing_code_type="MS_DRG",
        domain="inpatient",
        strict=False,
        output_path=(
            enriched_panel_cache_paths[
                "inpatient_concept"
            ]
        ),
    )
)

if source_panel_cache_paths[
    "strict_concept"
].exists():
    panel_run_summary.append(
        enrich_concept_domain(
            billing_code_type="CPT_HCPCS",
            domain="outpatient",
            strict=True,
            output_path=(
                enriched_panel_cache_paths[
                    "outpatient_strict_concept"
                ]
            ),
        )
    )

    panel_run_summary.append(
        enrich_concept_domain(
            billing_code_type="MS_DRG",
            domain="inpatient",
            strict=True,
            output_path=(
                enriched_panel_cache_paths[
                    "inpatient_strict_concept"
                ]
            ),
        )
    )


# -----------------------------------------------------------------------------
# 12C. Family panel.
# -----------------------------------------------------------------------------

family_panel = pd.read_parquet(
    source_panel_cache_paths["family"]
)

stable_lookup = pd.concat(
    [
        pd.read_parquet(
            lookup_cache_paths[
                "outpatient_stable"
            ]
        ),
        pd.read_parquet(
            lookup_cache_paths[
                "inpatient_stable"
            ]
        ),
    ],
    ignore_index=True,
    sort=False,
).drop_duplicates(
    subset=[
        "HOSPITAL_ID",
        "POST_MONTH",
    ],
    keep="first",
)

family_panel = merge_only_new_columns(
    family_panel,
    stable_lookup,
    keys=[
        "HOSPITAL_ID",
        "POST_MONTH",
    ],
    label="family hospital-month measures",
    validate="m:1",
)

del stable_lookup
gc.collect()

family_lookup = pd.concat(
    [
        pd.read_parquet(
            lookup_cache_paths[
                "outpatient_family"
            ]
        ),
        pd.read_parquet(
            lookup_cache_paths[
                "inpatient_family"
            ]
        ),
    ],
    ignore_index=True,
    sort=False,
)

family_lookup = family_lookup.drop_duplicates(
    subset=aggregation_keys("FAMILY"),
    keep="first",
)

family_left_duplicate_rows = int(
    family_panel.duplicated(
        aggregation_keys("FAMILY"),
        keep=False,
    ).sum()
)

family_right_duplicate_rows = int(
    family_lookup.duplicated(
        aggregation_keys("FAMILY"),
        keep=False,
    ).sum()
)

msg(
    "Family merge-key QA: "
    f"left duplicate rows={family_left_duplicate_rows:,}; "
    f"right duplicate rows={family_right_duplicate_rows:,}"
)

if family_left_duplicate_rows or family_right_duplicate_rows:
    raise RuntimeError(
        "The complete Phase 4 family key is not unique before merge. "
        "Inspect FINAL_SUPERFAMILY_ID and FINAL_FAMILY_ID."
    )

family_panel = merge_only_new_columns(
    family_panel,
    family_lookup,
    keys=aggregation_keys("FAMILY"),
    label="family treatment measures",
    validate="1:1",
)

del family_lookup
gc.collect()

family_panel[
    "SAMPLE_FAMILY_ENTRY"
] = (
    family_panel[
        "IS_FAMILY_ENTRY_MONTH"
    ]
    .fillna(0)
    .astype("int8")
)

family_panel[
    "SAMPLE_COHORT_FAMILY"
] = (
    (
        family_panel[
            "IS_HOSPITAL_FIRST_POST_MONTH"
        ] == 1
    )
    & family_panel["MEDIAN_PRICE"].notna()
    & family_panel[
        "COUNTY_STATE_KEY"
    ].notna()
).astype("int8")

qa_rows.append(
    unique_key_qa(
        family_panel,
        aggregation_keys("FAMILY"),
        "family_panel",
    )
)

panel_run_summary.append({
    "panel": "family",
    "rows": len(family_panel),
    "hospitals": int(
        family_panel[
            "HOSPITAL_ID"
        ].nunique()
    ),
    "service_ids": int(
        family_panel[
            phase4_family_key_columns()
        ]
        .drop_duplicates()
        .shape[0]
    ),
    "path": str(
        enriched_panel_cache_paths["family"]
    ),
})

write_parquet(
    family_panel,
    enriched_panel_cache_paths["family"],
)

del family_panel
gc.collect()

panel_run_summary_df = pd.DataFrame(
    panel_run_summary
)

panel_run_summary_df.to_csv(
    QA_DIR
    / "HPT_SECTION12_DISK_BACKED_PANEL_SUMMARY.csv",
    index=False,
)

msg(
    "\nDisk-backed aggregated panels rebuilt successfully"
)

for row in panel_run_summary:
    msg(
        f"  {row['panel']:<30}: "
        f"{int(row['rows']):>10,} rows; "
        f"{int(row['hospitals']):>5,} hospitals; "
        f"{int(row['service_ids']):>4,} service IDs"
    )

msg(
    "\nSection 12 complete. "
    "No full concept or family panel remains in RAM."
)


In [ ]:
# =============================================================================
# 13. LOW-MEMORY, CLEARLY LABELED R-READY OUTPUTS
# =============================================================================
#
# Every cached panel is loaded, exported, and released before the next panel is
# loaded. Geographic views use shallow copies so existing columns are not
# duplicated in memory.
# =============================================================================


def make_geo_view(
    master: pd.DataFrame,
    level: str,
    aggregation: str,
) -> pd.DataFrame:
    """
    Create an R panel with generic market and treatment aliases.

    level:
        COUNTY, CITY, or CBSA

    aggregation:
        EXACT, CONCEPT, or FAMILY
    """

    level = level.upper()
    aggregation = aggregation.upper()

    geo_cols = {
        "COUNTY": "COUNTY_STATE_KEY",
        "CITY": "CITY_STATE_KEY",
        "CBSA": "CBSA_CODE",
    }

    if level not in geo_cols:
        raise ValueError(
            f"Unsupported geography: {level}"
        )

    if aggregation not in {
        "EXACT",
        "CONCEPT",
        "FAMILY",
    }:
        raise ValueError(
            f"Unsupported aggregation: {aggregation}"
        )

    geo_col = geo_cols[level]

    treatment = (
        f"N_PRIOR_POSTERS_{level}_{aggregation}"
    )

    cumulative = (
        f"N_CUMULATIVE_POSTERS_{level}_{aggregation}"
    )

    current_wave = (
        f"N_POSTERS_THIS_MONTH_{level}_{aggregation}"
    )

    first_wave = (
        f"IS_FIRST_WAVE_{level}_{aggregation}"
    )

    entry_col = {
        "EXACT": "IS_EXACT_CODE_ENTRY_MONTH",
        "CONCEPT": "IS_CONCEPT_ENTRY_MONTH",
        "FAMILY": "IS_FAMILY_ENTRY_MONTH",
    }[aggregation]

    service_id_columns = {
        "EXACT": [
            "BILLING_CODE",
        ],
        "CONCEPT": phase4_concept_key_columns(),
        "FAMILY": phase4_family_key_columns(),
    }[aggregation]

    required = [
        geo_col,
        treatment,
        cumulative,
        current_wave,
        first_wave,
        *service_id_columns,
        "MEDIAN_PRICE",
        entry_col,
        "IS_HOSPITAL_FIRST_POST_MONTH",
        "BILLING_CODE_TYPE",
    ]

    missing = [
        column
        for column in required
        if column not in master.columns
    ]

    if missing:
        raise KeyError(
            f"Cannot build {level}-{aggregation} view; "
            f"missing columns: {missing}"
        )

    # Existing blocks are shared; only the new generic alias columns allocate
    # additional memory.
    out = master.copy(deep=False)

    out["ANALYSIS_GEOGRAPHY"] = level
    out["ANALYSIS_AGGREGATION"] = aggregation
    out["ANALYSIS_MARKET"] = out[geo_col]
    out["ANALYSIS_SERVICE_ID"] = build_composite_service_id(
        out,
        service_id_columns,
    )
    out["N_PRIOR_POSTERS"] = out[treatment]
    out[
        "N_HOSPITALS_CUMULATIVE_MARKET"
    ] = out[cumulative]
    out["N_POSTERS_THIS_MONTH"] = (
        out[current_wave]
    )
    out["IS_FIRST_WAVE_ANALYSIS"] = (
        out[first_wave]
    )
    out["IS_ENTRANT_MONTH_ANALYSIS"] = (
        out[entry_col]
    )

    out["MARKET_ID"] = (
        out[geo_col].astype("string")
        + "::"
        + out["ANALYSIS_SERVICE_ID"]
    )

    out["RECOMMENDED_BASELINE_SAMPLE"] = (
        (
            out[
                "IS_ENTRANT_MONTH_ANALYSIS"
            ] == 1
        )
        & out["ANALYSIS_MARKET"].notna()
        & out["MEDIAN_PRICE"].notna()
    ).astype("int8")

    out[
        "RECOMMENDED_POSTING_COHORT_SAMPLE"
    ] = (
        (
            out[
                "IS_HOSPITAL_FIRST_POST_MONTH"
            ] == 1
        )
        & out["ANALYSIS_MARKET"].notna()
        & out["MEDIAN_PRICE"].notna()
    ).astype("int8")

    return out


panel_export_specs = [
    {
        "cache_path": exact_master_cache_paths[
            "outpatient"
        ],
        "master_file": (
            "HPT_R_MASTER_OUTPATIENT_EXACT_CODE_"
            "ALL_GEOGRAPHIES.parquet"
        ),
        "domain": "OUTPATIENT",
        "unit": "EXACT_CODE",
        "aggregation": "EXACT",
        "role_family": "MAIN",
        "write_geo": True,
    },
    {
        "cache_path": exact_master_cache_paths[
            "inpatient"
        ],
        "master_file": (
            "HPT_R_MASTER_INPATIENT_DRG_"
            "ALL_GEOGRAPHIES.parquet"
        ),
        "domain": "INPATIENT",
        "unit": "DRG",
        "aggregation": "EXACT",
        "role_family": "MAIN",
        "write_geo": True,
    },
    {
        "cache_path": enriched_panel_cache_paths[
            "outpatient_concept"
        ],
        "master_file": (
            "HPT_R_MASTER_OUTPATIENT_CONCEPT_"
            "ALL_GEOGRAPHIES.parquet"
        ),
        "domain": "OUTPATIENT",
        "unit": "CONCEPT",
        "aggregation": "CONCEPT",
        "role_family": "MAIN",
        "write_geo": True,
    },
    {
        "cache_path": enriched_panel_cache_paths[
            "inpatient_concept"
        ],
        "master_file": (
            "HPT_R_MASTER_INPATIENT_CONCEPT_"
            "ALL_GEOGRAPHIES.parquet"
        ),
        "domain": "INPATIENT",
        "unit": "CONCEPT",
        "aggregation": "CONCEPT",
        "role_family": "MAIN",
        "write_geo": True,
    },
    {
        "cache_path": enriched_panel_cache_paths[
            "family"
        ],
        "master_file": (
            "HPT_R_MASTER_SERVICE_FAMILY_"
            "ALL_GEOGRAPHIES.parquet"
        ),
        "domain": "COMBINED",
        "unit": "SERVICE_FAMILY",
        "aggregation": "FAMILY",
        "role_family": "MAIN",
        "write_geo": True,
    },
]

for domain, cache_key, master_file in [
    (
        "OUTPATIENT",
        "outpatient_strict_concept",
        (
            "HPT_R_ROBUSTNESS_OUTPATIENT_STRICT_"
            "CONCEPT_ALL_GEOGRAPHIES.parquet"
        ),
    ),
    (
        "INPATIENT",
        "inpatient_strict_concept",
        (
            "HPT_R_ROBUSTNESS_INPATIENT_STRICT_"
            "CONCEPT_ALL_GEOGRAPHIES.parquet"
        ),
    ),
]:
    cache_path = enriched_panel_cache_paths[
        cache_key
    ]

    if cache_path.exists():
        panel_export_specs.append({
            "cache_path": cache_path,
            "master_file": master_file,
            "domain": domain,
            "unit": "STRICT_CONCEPT",
            "aggregation": "CONCEPT",
            "role_family": "STRICT_ROBUSTNESS",
            "write_geo": True,
        })

# There is no "robustness" tier in this pipeline, and exact_master_cache_paths
# carries no "robustness" key. A dead .exists() guard on that key would raise
# KeyError on the missing dict entry rather than evaluating False, so no guard
# is written here at all.

if exact_master_cache_paths[
    "component"
].exists():
    panel_export_specs.append({
        "cache_path": exact_master_cache_paths[
            "component"
        ],
        "master_file": (
            "HPT_R_SUPPORTING_COMPONENT_EXACT_"
            "CODE_ALL_GEOGRAPHIES.parquet"
        ),
        "domain": "COMBINED",
        "unit": "COMPONENT_EXACT_CODE",
        "aggregation": "EXACT",
        "role_family": "SUPPORTING",
        "write_geo": False,
    })


r_manifest_rows: list[dict[str, object]] = []

for spec in panel_export_specs:
    msg(
        "\nLoading one cached panel for export:\n"
        f"  {spec['cache_path']}"
    )

    frame = pd.read_parquet(
        spec["cache_path"]
    )

    write_parquet(
        frame,
        R_PANEL_DIR / spec["master_file"],
    )

    if (
        WRITE_ALL_R_GEOGRAPHY_DATASETS
        and spec["write_geo"]
    ):
        for level in [
            "COUNTY",
            "CITY",
            "CBSA",
        ]:
            view = make_geo_view(
                frame,
                level=level,
                aggregation=spec["aggregation"],
            )

            geography_role = (
                "PRIMARY"
                if level == "COUNTY"
                else "ROBUSTNESS"
            )

            file_name = (
                f"HPT_R_{spec['role_family']}_"
                f"{geography_role}_{level}_"
                f"{spec['domain']}_{spec['unit']}.parquet"
            )

            write_parquet(
                view,
                R_PANEL_DIR / file_name,
            )

            r_manifest_rows.append({
                "file_name": file_name,
                "dataset_role": spec["role_family"],
                "geography_role": geography_role,
                "geography": level,
                "domain": spec["domain"],
                "unit_of_observation": spec["unit"],
                "treatment_column_original": (
                    f"N_PRIOR_POSTERS_{level}_"
                    f"{spec['aggregation']}"
                ),
                "treatment_column_generic": (
                    "N_PRIOR_POSTERS"
                ),
                "market_column_generic": (
                    "ANALYSIS_MARKET"
                ),
                "market_fixed_effect_generic": (
                    "MARKET_ID"
                ),
                "recommended_use": (
                    "Preferred county-market specification"
                    if level == "COUNTY"
                    else (
                        f"{level.title()} "
                        "market-definition robustness"
                    )
                ),
                "n_rows": len(view),
                "n_hospitals": int(
                    view[
                        "HOSPITAL_ID"
                    ].nunique()
                ),
            })

            del view
            gc.collect()

    del frame
    gc.collect()


r_manifest = pd.DataFrame(
    r_manifest_rows
)

r_manifest.to_csv(
    R_PANEL_DIR / "HPT_R_PANEL_MANIFEST.csv",
    index=False,
)

msg(
    "\nR-ready output panels have been written to:\n"
    f"  {R_PANEL_DIR}"
)


In [ ]:
# =============================================================================
# 14. COUNTY COVERAGE MAP AND POPULATION COVERAGE
# =============================================================================

from io import BytesIO
from pathlib import Path

import re
import warnings

import certifi
import numpy as np
import pandas as pd
import requests


# -----------------------------------------------------------------------------
# 14A. CONFIGURATION
# -----------------------------------------------------------------------------

PEP_2023_URL = (
    "https://www2.census.gov/programs-surveys/popest/datasets/"
    "2020-2023/counties/totals/co-est2023-alldata.csv"
)

# Cache the Census population file after the first successful download.
PEP_2023_CACHE = (
    COVERAGE_DIR
    / "co-est2023-alldata.csv"
)

COVERAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

QA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# -----------------------------------------------------------------------------
# 14B. IDENTIFY COUNTIES REPRESENTED IN EACH ANALYSIS SAMPLE
# -----------------------------------------------------------------------------

def get_analysis_counties(
    df: pd.DataFrame,
    sample_col: str | None = None,
) -> set[str]:
    """
    Return valid five-digit county FIPS codes represented in an analysis sample.

    Parameters
    ----------
    df
        Analysis dataset containing COUNTY_FIPS.

    sample_col
        Optional binary sample indicator. If provided, only rows where the
        indicator equals one are used.
    """

    require_columns(
        df,
        ["COUNTY_FIPS"],
        "Coverage-analysis dataset",
    )

    if sample_col is None:

        subset = df

    else:

        if sample_col not in df.columns:

            raise KeyError(
                "Requested coverage sample column is missing: "
                f"{sample_col}"
            )

        subset = df.loc[
            df[sample_col] == 1
        ]

    county_fips = (
        subset["COUNTY_FIPS"]
        .dropna()
        .astype("string")
        .str.strip()
        .str.zfill(5)
    )

    valid_mask = county_fips.str.fullmatch(
        r"\d{5}",
        na=False,
    )

    county_fips = county_fips.loc[
        valid_mask
    ]

    return set(
        county_fips.unique()
    )



# Read only the columns required for coverage accounting. The full exact-code
# panels remain on disk so Section 14 does not reload several gigabytes.
#
# NOTE: pd.read_parquet(..., columns=[...]) raises a hard error if ANY listed
# column is absent from the file's schema -- unlike most of this notebook's
# column handling, this is not something a simple "if column in df.columns"
# check can guard after the fact, since the read itself is what fails.
# SAMPLE_STRICT_PRICE_AVAILABLE in particular is only ever created in cell 13
# when a STRICT_PRICE_AVAILABLE_FLAG source column exists, which it never
# does in this pipeline (no strict tier), so that column was never written.
# The desired list is filtered against the actual on-disk schema (a cheap,
# data-free read via pyarrow) before being passed to pd.read_parquet.

import pyarrow.parquet as pq


def existing_parquet_columns(path: Path, desired: list[str]) -> list[str]:
    available = set(pq.ParquetFile(path).schema.names)
    missing = [c for c in desired if c not in available]
    if missing:
        msg(f"  (columns not present in {path.name}, skipped: {missing})")
    return [c for c in desired if c in available]


coverage_outpatient_columns = [
    "COUNTY_FIPS",
    "SAMPLE_BASELINE_COUNTY",
    "SAMPLE_BASELINE_COUNTY_POSITIVE_PRIOR",
    "SAMPLE_STRICT_PRICE_AVAILABLE",
    "SAMPLE_FIVE_PLUS_PAYER_CELLS",
    "SAMPLE_TEN_PLUS_PAYER_CELLS",
    "SAMPLE_FIVE_PLUS_PAYERS",
]

coverage_inpatient_columns = [
    "COUNTY_FIPS",
]

_outpatient_master_path = (
    R_PANEL_DIR
    / "HPT_R_MASTER_OUTPATIENT_EXACT_CODE_ALL_GEOGRAPHIES.parquet"
)
_inpatient_master_path = (
    R_PANEL_DIR
    / "HPT_R_MASTER_INPATIENT_DRG_ALL_GEOGRAPHIES.parquet"
)

coverage_outpatient = pd.read_parquet(
    _outpatient_master_path,
    columns=existing_parquet_columns(_outpatient_master_path, coverage_outpatient_columns),
)

coverage_inpatient = pd.read_parquet(
    _inpatient_master_path,
    columns=existing_parquet_columns(_inpatient_master_path, coverage_inpatient_columns),
)


# Use a set union rather than concatenating both large datasets.
any_exact_code_counties = (
    get_analysis_counties(
        coverage_outpatient
    )
    |
    get_analysis_counties(
        coverage_inpatient
    )
)


coverage_samples: dict[str, set[str]] = {
    "ANY_FINAL_EXACT_CODE_DATA":
        any_exact_code_counties,

    "OUTPATIENT_EXACT_CODE_DATA":
        get_analysis_counties(
            coverage_outpatient
        ),

    "INPATIENT_DRG_DATA":
        get_analysis_counties(
            coverage_inpatient
        ),

    "BASELINE_COUNTY_ANALYSIS":
        get_analysis_counties(
            coverage_outpatient,
            "SAMPLE_BASELINE_COUNTY",
        ),

    "BASELINE_COUNTY_POSITIVE_PRIOR":
        get_analysis_counties(
            coverage_outpatient,
            "SAMPLE_BASELINE_COUNTY_POSITIVE_PRIOR",
        ),
}


# Strict-price coverage sample.
if (
    "SAMPLE_STRICT_PRICE_AVAILABLE"
    in coverage_outpatient.columns
):

    strict_subset = coverage_outpatient.loc[
        (
            coverage_outpatient[
                "SAMPLE_BASELINE_COUNTY"
            ] == 1
        )
        &
        (
            coverage_outpatient[
                "SAMPLE_STRICT_PRICE_AVAILABLE"
            ] == 1
        )
    ]

    coverage_samples[
        "BASELINE_COUNTY_STRICT_PRICE"
    ] = get_analysis_counties(
        strict_subset
    )


# Optional payer-support coverage samples.
optional_coverage_flags = {
    "SAMPLE_FIVE_PLUS_PAYER_CELLS":
        "BASELINE_COUNTY_FIVE_PLUS_PAYER_CELLS",

    "SAMPLE_TEN_PLUS_PAYER_CELLS":
        "BASELINE_COUNTY_TEN_PLUS_PAYER_CELLS",

    "SAMPLE_FIVE_PLUS_PAYERS":
        "BASELINE_COUNTY_FIVE_PLUS_DISTINCT_PAYERS",
}


for flag_column, sample_name in optional_coverage_flags.items():

    if flag_column not in coverage_outpatient.columns:
        continue

    support_subset = coverage_outpatient.loc[
        (
            coverage_outpatient[
                "SAMPLE_BASELINE_COUNTY"
            ] == 1
        )
        &
        (
            coverage_outpatient[
                flag_column
            ] == 1
        )
    ]

    coverage_samples[
        sample_name
    ] = get_analysis_counties(
        support_subset
    )


del coverage_outpatient
del coverage_inpatient
gc.collect()

msg(
    "\nCounty sets created for coverage analysis"
)

for sample_name, county_set in coverage_samples.items():

    msg(
        f"  {sample_name}: "
        f"{len(county_set):,} counties"
    )


# -----------------------------------------------------------------------------
# 14C. LOAD 2023 CENSUS COUNTY POPULATION ESTIMATES
# -----------------------------------------------------------------------------

def load_pep_2023() -> pd.DataFrame:
    """
    Load Census 2023 county population estimates.

    Loading sequence
    ----------------
    1. Use the cached file in COVERAGE_DIR.
    2. Search the project directories for a local copy.
    3. Download the official Census file using requests and certifi.
    4. Cache the downloaded file for future runs.
    """

    local_path: Path | None = None

    # Prefer the designated cached file.
    if PEP_2023_CACHE.exists():

        local_path = PEP_2023_CACHE

    else:

        existing_local = optional_file(
            "co-est2023-alldata.csv",
            [
                COVERAGE_DIR,
                RAW_DIR,
                DATA_DIR,
                PROJECT_ROOT,
            ],
        )

        if existing_local is not None:

            local_path = existing_local

    # -------------------------------------------------------------------------
    # Read a local copy when available.
    # -------------------------------------------------------------------------

    if local_path is not None:

        msg(
            "\nReading local Census population file:\n"
            f"  {local_path}"
        )

        pep = pd.read_csv(
            local_path,
            encoding="latin1",
            low_memory=False,
        )

    # -------------------------------------------------------------------------
    # Otherwise download with an explicit CA certificate bundle.
    # -------------------------------------------------------------------------

    else:

        msg(
            "\nDownloading 2023 Census county "
            "population estimates ..."
        )

        try:

            response = requests.get(
                PEP_2023_URL,
                timeout=120,
                verify=certifi.where(),
                headers={
                    "User-Agent": (
                        "Mozilla/5.0 "
                        "HPT-Academic-Research-Pipeline/1.0"
                    )
                },
            )

            response.raise_for_status()

        except requests.RequestException as exc:

            raise RuntimeError(
                "\nCould not download the 2023 Census county "
                "population file, and no local copy was found.\n\n"
                "Download this file manually:\n"
                f"{PEP_2023_URL}\n\n"
                "Save it here:\n"
                f"{PEP_2023_CACHE}\n\n"
                "Then rerun Section 14.\n\n"
                f"Original error:\n{exc}"
            ) from exc

        if not response.content:

            raise RuntimeError(
                "The Census request returned an empty response."
            )

        pep = pd.read_csv(
            BytesIO(
                response.content
            ),
            encoding="latin1",
            low_memory=False,
        )

        PEP_2023_CACHE.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        PEP_2023_CACHE.write_bytes(
            response.content
        )

        msg(
            "Saved Census population cache:\n"
            f"  {PEP_2023_CACHE}"
        )

    # -------------------------------------------------------------------------
    # Standardize columns.
    # -------------------------------------------------------------------------

    pep = normalize_column_names(
        pep
    )

    require_columns(
        pep,
        [
            "STATE",
            "COUNTY",
            "POPESTIMATE2023",
        ],
        "Census 2023 county population file",
    )

    pep["STATE"] = pd.to_numeric(
        pep["STATE"],
        errors="coerce",
    )

    pep["COUNTY"] = pd.to_numeric(
        pep["COUNTY"],
        errors="coerce",
    )

    pep["POPESTIMATE2023"] = pd.to_numeric(
        pep["POPESTIMATE2023"],
        errors="coerce",
    )

    # COUNTY == 0 denotes a state total, not a county observation.
    pep = pep.loc[
        pep["STATE"].notna()
        &
        pep["COUNTY"].notna()
        &
        pep["POPESTIMATE2023"].notna()
        &
        (
            pep["COUNTY"] != 0
        ),
        [
            "STATE",
            "COUNTY",
            "POPESTIMATE2023",
        ],
    ].copy()

    pep["STATEFP"] = (
        pep["STATE"]
        .astype("Int64")
        .astype("string")
        .str.zfill(2)
    )

    # Keep the 50 states and Washington, DC.
    pep = pep.loc[
        pep["STATEFP"].isin(
            VALID_STATE_FIPS
        )
    ].copy()

    pep["COUNTYFP"] = (
        pep["COUNTY"]
        .astype("Int64")
        .astype("string")
        .str.zfill(3)
    )

    pep["COUNTY_FIPS"] = (
        pep["STATEFP"]
        +
        pep["COUNTYFP"]
    )

    pep["STATE_ABBREV"] = (
        pep["STATEFP"]
        .map(
            FIPS_TO_ABBREV
        )
        .astype("string")
    )

    # -------------------------------------------------------------------------
    # Validate county FIPS.
    # -------------------------------------------------------------------------

    valid_fips_mask = (
        pep["COUNTY_FIPS"]
        .astype("string")
        .str.fullmatch(
            r"\d{5}",
            na=False,
        )
    )

    invalid_fips = int(
        (
            ~valid_fips_mask
        ).sum()
    )

    if invalid_fips != 0:

        invalid_examples = (
            pep.loc[
                ~valid_fips_mask,
                [
                    "STATE",
                    "COUNTY",
                    "STATEFP",
                    "COUNTYFP",
                    "COUNTY_FIPS",
                ],
            ]
            .head(20)
        )

        raise RuntimeError(
            "The Census population file contains invalid county "
            f"FIPS codes: {invalid_fips:,}\n"
            f"{invalid_examples.to_string(index=False)}"
        )

    # -------------------------------------------------------------------------
    # Validate uniqueness.
    # -------------------------------------------------------------------------

    duplicate_mask = pep.duplicated(
        "COUNTY_FIPS",
        keep=False,
    )

    duplicate_counties = int(
        duplicate_mask.sum()
    )

    if duplicate_counties != 0:

        duplicate_examples = (
            pep.loc[
                duplicate_mask,
                [
                    "COUNTY_FIPS",
                    "POPESTIMATE2023",
                ],
            ]
            .sort_values(
                "COUNTY_FIPS"
            )
            .head(20)
        )

        raise RuntimeError(
            "The Census population file is not unique by "
            f"COUNTY_FIPS. Duplicate rows: {duplicate_counties:,}\n"
            f"{duplicate_examples.to_string(index=False)}"
        )

    # -------------------------------------------------------------------------
    # Validate population.
    # -------------------------------------------------------------------------

    negative_population_count = int(
        (
            pep["POPESTIMATE2023"] < 0
        ).sum()
    )

    if negative_population_count != 0:

        raise RuntimeError(
            "The Census population file contains "
            f"{negative_population_count:,} negative population values."
        )

    pep = (
        pep[
            [
                "COUNTY_FIPS",
                "STATEFP",
                "COUNTYFP",
                "STATE_ABBREV",
                "POPESTIMATE2023",
            ]
        ]
        .sort_values(
            "COUNTY_FIPS"
        )
        .reset_index(
            drop=True
        )
    )

    msg(
        "\nCensus population data loaded successfully"
    )

    msg(
        f"  Counties: "
        f"{pep['COUNTY_FIPS'].nunique():,}"
    )

    msg(
        f"  2023 population: "
        f"{pep['POPESTIMATE2023'].sum():,.0f}"
    )

    return pep


pep = load_pep_2023()


# -----------------------------------------------------------------------------
# 14D. CALCULATE NATIONAL COUNTY AND POPULATION COVERAGE
# -----------------------------------------------------------------------------

total_population = float(
    pep["POPESTIMATE2023"].sum()
)

total_counties = int(
    pep["COUNTY_FIPS"].nunique()
)

coverage_rows: list[dict[str, object]] = []

unmatched_rows: list[dict[str, object]] = []


for sample_name, county_set in coverage_samples.items():

    covered = pep.loc[
        pep["COUNTY_FIPS"].isin(
            county_set
        )
    ].copy()

    matched_counties = set(
        covered["COUNTY_FIPS"]
        .dropna()
        .astype("string")
        .unique()
    )

    unmatched_counties = (
        county_set
        -
        matched_counties
    )

    covered_population = float(
        covered["POPESTIMATE2023"].sum()
    )

    n_counties_in_sample = len(
        county_set
    )

    n_counties_matched = int(
        covered["COUNTY_FIPS"].nunique()
    )

    county_coverage_share = (
        n_counties_matched
        / total_counties
        if total_counties > 0
        else np.nan
    )

    population_coverage_share = (
        covered_population
        / total_population
        if total_population > 0
        else np.nan
    )

    coverage_rows.append({
        "sample":
            sample_name,

        "n_counties_in_analysis_data":
            n_counties_in_sample,

        "n_population_counties_matched":
            n_counties_matched,

        "n_counties_unmatched_to_population":
            len(
                unmatched_counties
            ),

        "total_us_counties_50_states_dc":
            total_counties,

        "county_coverage_share":
            county_coverage_share,

        "county_coverage_percent":
            100 * county_coverage_share,

        "population_covered_2023":
            covered_population,

        "total_us_population_2023_50_states_dc":
            total_population,

        "population_coverage_share":
            population_coverage_share,

        "population_coverage_percent":
            100 * population_coverage_share,
    })

    for county_fips in sorted(
        unmatched_counties
    ):

        unmatched_rows.append({
            "sample":
                sample_name,

            "COUNTY_FIPS":
                county_fips,
        })


population_coverage = pd.DataFrame(
    coverage_rows
)

population_coverage = (
    population_coverage
    .sort_values(
        "sample"
    )
    .reset_index(
        drop=True
    )
)

population_coverage.to_csv(
    COVERAGE_DIR
    / "HPT_POPULATION_COVERAGE_SUMMARY.csv",
    index=False,
)


unmatched_population_counties = pd.DataFrame(
    unmatched_rows,
    columns=[
        "sample",
        "COUNTY_FIPS",
    ],
)

unmatched_population_counties.to_csv(
    COVERAGE_DIR
    / "HPT_POPULATION_COVERAGE_UNMATCHED_COUNTIES.csv",
    index=False,
)


msg(
    "\nPopulation coverage summary"
)

msg(
    population_coverage.to_string(
        index=False
    )
)


# -----------------------------------------------------------------------------
# 14E. STATE-LEVEL COUNTY COVERAGE
# -----------------------------------------------------------------------------

require_columns(
    counties_gdf,
    [
        "COUNTY_FIPS",
        "STATEFP",
        "STATE_ABBREV",
    ],
    "County shapefile",
)

county_state_ref = (
    counties_gdf[
        [
            "COUNTY_FIPS",
            "STATEFP",
            "STATE_ABBREV",
        ]
    ]
    .drop_duplicates(
        "COUNTY_FIPS"
    )
    .copy()
)

baseline_counties = coverage_samples[
    "BASELINE_COUNTY_ANALYSIS"
]

county_state_ref["IN_BASELINE"] = (
    county_state_ref["COUNTY_FIPS"]
    .isin(
        baseline_counties
    )
    .astype("int8")
)

state_county_coverage = (
    county_state_ref
    .groupby(
        [
            "STATEFP",
            "STATE_ABBREV",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        N_COUNTIES_TOTAL=(
            "COUNTY_FIPS",
            "nunique",
        ),

        N_COUNTIES_COVERED=(
            "IN_BASELINE",
            "sum",
        ),
    )
)

state_county_coverage[
    "COUNTY_COVERAGE_SHARE"
] = (
    state_county_coverage[
        "N_COUNTIES_COVERED"
    ]
    /
    state_county_coverage[
        "N_COUNTIES_TOTAL"
    ]
)

state_county_coverage[
    "COUNTY_COVERAGE_PERCENT"
] = (
    100
    *
    state_county_coverage[
        "COUNTY_COVERAGE_SHARE"
    ]
)


# -----------------------------------------------------------------------------
# 14F. STATE-LEVEL POPULATION COVERAGE
# -----------------------------------------------------------------------------

pep_state = pep.copy()

pep_state["IN_BASELINE"] = (
    pep_state["COUNTY_FIPS"]
    .isin(
        baseline_counties
    )
    .astype("int8")
)

pep_state["COVERED_POPULATION"] = (
    pep_state["POPESTIMATE2023"]
    *
    pep_state["IN_BASELINE"]
)

state_population_coverage = (
    pep_state
    .groupby(
        [
            "STATEFP",
            "STATE_ABBREV",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        TOTAL_STATE_POPULATION_2023=(
            "POPESTIMATE2023",
            "sum",
        ),

        COVERED_STATE_POPULATION_2023=(
            "COVERED_POPULATION",
            "sum",
        ),
    )
)

state_population_coverage[
    "STATE_POPULATION_COVERAGE_SHARE"
] = (
    state_population_coverage[
        "COVERED_STATE_POPULATION_2023"
    ]
    /
    state_population_coverage[
        "TOTAL_STATE_POPULATION_2023"
    ]
)

state_population_coverage[
    "STATE_POPULATION_COVERAGE_PERCENT"
] = (
    100
    *
    state_population_coverage[
        "STATE_POPULATION_COVERAGE_SHARE"
    ]
)


state_coverage = state_county_coverage.merge(
    state_population_coverage,
    on=[
        "STATEFP",
        "STATE_ABBREV",
    ],
    how="outer",
    validate="1:1",
)

state_coverage = (
    state_coverage
    .sort_values(
        "STATE_ABBREV"
    )
    .reset_index(
        drop=True
    )
)

state_coverage.to_csv(
    COVERAGE_DIR
    / "HPT_STATE_COUNTY_COVERAGE.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# 14G. COUNTY-LEVEL COVERAGE CROSSWALK
# -----------------------------------------------------------------------------

county_coverage_crosswalk = (
    county_state_ref[
        [
            "COUNTY_FIPS",
            "STATEFP",
            "STATE_ABBREV",
        ]
    ]
    .merge(
        pep[
            [
                "COUNTY_FIPS",
                "POPESTIMATE2023",
            ]
        ],
        on="COUNTY_FIPS",
        how="left",
        validate="1:1",
    )
)


for sample_name, county_set in coverage_samples.items():

    safe_column_name = re.sub(
        r"[^A-Z0-9]+",
        "_",
        sample_name.upper(),
    ).strip("_")

    county_coverage_crosswalk[
        f"IN_{safe_column_name}"
    ] = (
        county_coverage_crosswalk[
            "COUNTY_FIPS"
        ]
        .isin(
            county_set
        )
        .astype("int8")
    )


county_coverage_crosswalk.to_csv(
    COVERAGE_DIR
    / "HPT_COUNTY_COVERAGE_CROSSWALK.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# 14H. CREATE CONTIGUOUS U.S. COUNTY COVERAGE MAP
# -----------------------------------------------------------------------------

if BUILD_COVERAGE_MAP:

    try:

        import matplotlib.patches as mpatches
        import matplotlib.pyplot as plt

        require_columns(
            states_gdf,
            [
                "STATEFP",
                "geometry",
            ],
            "State shapefile",
        )

        map_counties = counties_gdf.loc[
            counties_gdf[
                "STATEFP"
            ].isin(
                VALID_STATE_FIPS
            )
        ].copy()

        map_counties[
            "IN_ANALYSIS"
        ] = (
            map_counties[
                "COUNTY_FIPS"
            ]
            .isin(
                baseline_counties
            )
        )

        # Alaska and Hawaii remain in the population statistics but are omitted
        # from the main contiguous-U.S. paper figure.
        non_conus = {
            "02",
            "15",
        }

        conus_counties = map_counties.loc[
            ~map_counties[
                "STATEFP"
            ].isin(
                non_conus
            )
        ].copy()

        conus_states = states_gdf.loc[
            states_gdf[
                "STATEFP"
            ].isin(
                VALID_STATE_FIPS
                -
                non_conus
            )
        ].copy()

        # Continental equal-area projection.
        conus_counties = conus_counties.to_crs(
            "EPSG:5070"
        )

        conus_states = conus_states.to_crs(
            "EPSG:5070"
        )

        conus_counties[
            "geometry"
        ] = (
            conus_counties
            .geometry
            .simplify(
                1000,
                preserve_topology=True,
            )
        )

        conus_states[
            "geometry"
        ] = (
            conus_states
            .geometry
            .simplify(
                1000,
                preserve_topology=True,
            )
        )

        n_baseline_counties = len(
            baseline_counties
        )

        baseline_population_row = (
            population_coverage.loc[
                population_coverage[
                    "sample"
                ]
                ==
                "BASELINE_COUNTY_ANALYSIS"
            ]
        )

        if not baseline_population_row.empty:

            baseline_population_percent = float(
                baseline_population_row[
                    "population_coverage_percent"
                ].iloc[0]
            )

            subtitle = (
                f"{n_baseline_counties:,} counties; "
                f"{baseline_population_percent:.1f}% "
                "of the 2023 U.S. population"
            )

        else:

            subtitle = (
                f"{n_baseline_counties:,} counties"
            )

        fig, ax = plt.subplots(
            figsize=(
                11,
                7.2,
            )
        )

        conus_counties.loc[
            ~conus_counties[
                "IN_ANALYSIS"
            ]
        ].plot(
            ax=ax,
            facecolor="white",
            edgecolor="0.78",
            linewidth=0.12,
        )

        conus_counties.loc[
            conus_counties[
                "IN_ANALYSIS"
            ]
        ].plot(
            ax=ax,
            facecolor="0.35",
            edgecolor="0.20",
            linewidth=0.15,
        )

        conus_states.boundary.plot(
            ax=ax,
            edgecolor="black",
            linewidth=0.45,
        )

        ax.set_axis_off()

        ax.set_title(
            "Counties Represented in the Baseline "
            "Hospital Price Analysis\n"
            f"{subtitle}",
            fontsize=14,
            pad=10,
        )

        covered_patch = mpatches.Patch(
            facecolor="0.35",
            edgecolor="0.20",
            label="County represented",
        )

        other_patch = mpatches.Patch(
            facecolor="white",
            edgecolor="0.65",
            label="County not represented",
        )

        ax.legend(
            handles=[
                covered_patch,
                other_patch,
            ],
            loc="lower left",
            frameon=False,
            fontsize=10,
        )

        fig.tight_layout()

        pdf_path = (
            COVERAGE_DIR
            / "HPT_COUNTY_COVERAGE_MAP_CONUS.pdf"
        )

        png_path = (
            COVERAGE_DIR
            / "HPT_COUNTY_COVERAGE_MAP_CONUS.png"
        )

        fig.savefig(
            pdf_path,
            bbox_inches="tight",
        )

        fig.savefig(
            png_path,
            dpi=400,
            bbox_inches="tight",
        )

        plt.close(
            fig
        )

        msg(
            "\nCoverage map saved successfully"
        )

        msg(
            f"  PDF: {pdf_path}"
        )

        msg(
            f"  PNG: {png_path}"
        )

    except Exception as exc:

        warnings.warn(
            "Coverage map was not created. "
            f"Reason: {exc}"
        )


# -----------------------------------------------------------------------------
# 14I. FINAL COVERAGE QA
# -----------------------------------------------------------------------------

baseline_coverage_rows = population_coverage.loc[
    population_coverage[
        "sample"
    ]
    ==
    "BASELINE_COUNTY_ANALYSIS"
]

if baseline_coverage_rows.empty:

    baseline_unmatched_count = np.nan

else:

    baseline_unmatched_count = int(
        baseline_coverage_rows[
            "n_counties_unmatched_to_population"
        ].iloc[0]
    )


coverage_qa_rows: list[dict[str, object]] = [
    {
        "check":
            "Census county population unique key",

        "status":
            (
                "PASS"
                if not pep.duplicated(
                    "COUNTY_FIPS"
                ).any()
                else "FAIL"
            ),

        "value":
            int(
                pep.duplicated(
                    "COUNTY_FIPS"
                ).sum()
            ),

        "expected":
            0,
    },

    {
        "check":
            "Census population positive",

        "status":
            (
                "PASS"
                if total_population > 0
                else "FAIL"
            ),

        "value":
            total_population,

        "expected":
            "> 0",
    },

    {
        "check":
            "Baseline analysis has counties",

        "status":
            (
                "PASS"
                if len(
                    baseline_counties
                ) > 0
                else "FAIL"
            ),

        "value":
            len(
                baseline_counties
            ),

        "expected":
            "> 0",
    },

    {
        "check":
            "Baseline counties matched to population",

        "status":
            (
                "PASS"
                if (
                    pd.notna(
                        baseline_unmatched_count
                    )
                    and baseline_unmatched_count == 0
                )
                else "WARN"
            ),

        "value":
            baseline_unmatched_count,

        "expected":
            0,
    },
]


coverage_qa = pd.DataFrame(
    coverage_qa_rows
)

coverage_qa.to_csv(
    QA_DIR
    / "HPT_COVERAGE_QA.csv",
    index=False,
)


msg(
    "\nCoverage QA"
)

msg(
    coverage_qa.to_string(
        index=False
    )
)


msg(
    "\nCoverage outputs saved in:\n"
    f"  {COVERAGE_DIR}"
)

In [ ]:
# =============================================================================
# 15. FINAL QA, INVENTORY, AND DATA DICTIONARIES
# =============================================================================

# -----------------------------------------------------------------------------
# 15A. Main panel QA
# -----------------------------------------------------------------------------

exact_qa_specs = [
    (
        R_PANEL_DIR
        / "HPT_R_MASTER_OUTPATIENT_EXACT_CODE_ALL_GEOGRAPHIES.parquet",
        "outpatient_master",
    ),
    (
        R_PANEL_DIR
        / "HPT_R_MASTER_INPATIENT_DRG_ALL_GEOGRAPHIES.parquet",
        "inpatient_master",
    ),
]

main_exact_mapping_missing = 0

for path, label in exact_qa_specs:
    frame = pd.read_parquet(
        path,
        columns=[
            "HOSPITAL_ID",
            "POST_MONTH",
            "BILLING_CODE_TYPE",
            "BILLING_CODE",
            "FINAL_CONCEPT_ID",
            "N_PRIOR_POSTERS_COUNTY_EXACT",
            "COUNTY_STATE_KEY",
            "Z_SYS_STRICT_9M_EXCL_CURRENT",
        ],
    )

    qa_rows.append(
        unique_key_qa(
            frame,
            [
                "HOSPITAL_ID",
                "POST_MONTH",
                "BILLING_CODE_TYPE",
                "BILLING_CODE",
            ],
            label,
        )
    )

    negative_count = int(
        (
            frame[
                "N_PRIOR_POSTERS_COUNTY_EXACT"
            ] < 0
        ).sum()
    )

    qa_rows.append({
        "check": (
            f"{label}_negative_county_prior"
        ),
        "status": (
            "PASS"
            if negative_count == 0
            else "FAIL"
        ),
        "value": negative_count,
        "expected": 0,
        "details": (
            "Strictly prior county × exact-code hospital count"
        ),
    })

    missing_iv = int(
        frame.loc[
            frame[
                "COUNTY_STATE_KEY"
            ].notna(),
            "Z_SYS_STRICT_9M_EXCL_CURRENT",
        ].isna().sum()
    )

    qa_rows.append({
        "check": (
            f"{label}_missing_strict_system_iv_valid_county"
        ),
        "status": (
            "PASS"
            if missing_iv == 0
            else "FAIL"
        ),
        "value": missing_iv,
        "expected": 0,
        "details": (
            "County-based canonical strict system instrument"
        ),
    })

    main_exact_mapping_missing += int(
        frame[
            "FINAL_CONCEPT_ID"
        ].isna().sum()
    )

    del frame
    gc.collect()


# No concept-key reconciliation check is run here. Concepts are built once,
# from description-stem text, in Phase 1 SQL, and are never post-split by a
# shoppability scheme -- so there is no second concept universe to reconcile
# against. Shoppability is applied downstream as a label, not as a split.

qa_rows.append({
    "check": "main_exact_concept_mapping_missing",
    "status": (
        "PASS"
        if main_exact_mapping_missing == 0
        else "FAIL"
    ),
    "value": main_exact_mapping_missing,
    "expected": 0,
    "details": (
        "Every scoped exact-code row maps to a concept "
        "(ANALYSIS_CONCEPT_ID / FINAL_CONCEPT_ID is never null)"
    ),
})


panel_summary_lookup = (
    panel_run_summary_df
    .set_index("panel")
    ["rows"]
    .astype("int64")
    .to_dict()
)

main_concept_rows = int(
    panel_summary_lookup.get(
        "outpatient_concept",
        0,
    )
    + panel_summary_lookup.get(
        "inpatient_concept",
        0,
    )
)

family_rows = int(
    panel_summary_lookup.get(
        "family",
        0,
    )
)

# No strict-concept reconciliation runs here: there is no strict tier in this
# pipeline, so there is no second row count to reconcile against.
#
# expected_counts is built in Section 2 by reading each phase's QA table at
# load time. Keys are "concept" and "family", matching the export folders.
for label, actual, expected_key in [
    (
        "main_concept_panel",
        main_concept_rows,
        "concept",
    ),
    (
        "family_panel",
        family_rows,
        "family",
    ),
]:
    expected = int(
        expected_counts[
            expected_key
        ]
    )

    qa_rows.append({
        "check": f"{label}_row_reconciliation",
        "status": (
            "PASS"
            if actual == expected
            else "FAIL"
        ),
        "value": actual,
        "expected": expected,
        "details": (
            "Disk-backed Python-enriched rows equal "
            "frozen Snowflake rows"
        ),
    })


qa = pd.DataFrame(
    qa_rows
)

n_failures = int(
    (qa["status"] == "FAIL").sum()
)

qa.to_csv(
    QA_DIR / "HPT_PYTHON_PIPELINE_QA.csv",
    index=False,
)

if n_failures:
    failed = qa.loc[
        qa["status"] == "FAIL"
    ]

    raise RuntimeError(
        "Python pipeline QA contains failures:\n"
        f"{failed.to_string(index=False)}"
    )

# -----------------------------------------------------------------------------
# 15B. Output inventory
# -----------------------------------------------------------------------------

output_files: list[dict[str, object]] = []

for path in sorted(
    OUTPUT_DIR.rglob("*")
):
    if path.is_file():
        output_files.append({
            "file_name": str(
                path.relative_to(OUTPUT_DIR)
            ),
            "size_bytes": path.stat().st_size,
            "size_mb": (
                path.stat().st_size
                / 1024**2
            ),
        })

pd.DataFrame(
    output_files
).to_csv(
    QA_DIR / "HPT_PYTHON_OUTPUT_INVENTORY.csv",
    index=False,
)

# -----------------------------------------------------------------------------
# 15C. Variable dictionary
# -----------------------------------------------------------------------------

variable_dictionary_rows = [
    (
        "N_PRIOR_POSTERS_COUNTY_EXACT",
        "Primary exact-code treatment: distinct hospitals in the same county and exact code entering strictly before the focal posting month.",
    ),
    (
        "N_PRIOR_POSTERS_CITY_EXACT",
        "City robustness treatment: distinct hospitals in the same provider city/state and exact code entering strictly before the focal posting month.",
    ),
    (
        "N_PRIOR_POSTERS_CBSA_EXACT",
        "CBSA robustness treatment: distinct hospitals in the same CBSA and exact code entering strictly before the focal posting month.",
    ),
    (
        "N_PRIOR_POSTERS_COUNTY_CONCEPT",
        "Distinct hospitals in the same county and final reviewed Phase 4 concept entering strictly before the focal posting month.",
    ),
    (
        "N_PRIOR_POSTERS_COUNTY_FAMILY",
        "Distinct hospitals in the same county and service family entering strictly before the focal posting month.",
    ),
    (
        "COUNTY_N_PRIOR_POSTERS",
        "Snowflake market-wide county context: hospitals whose first observed posting cohort predates the focal hospital's first observed posting cohort.",
    ),
    (
        "COUNTY_N_SAME_COHORT_POSTERS",
        "Hospitals entering the county disclosure sample in the same first-post cohort as the focal hospital.",
    ),
    (
        "COUNTY_N_CUMULATIVE_POSTERS",
        "Market-wide cumulative county posters through the focal hospital's first observed posting cohort.",
    ),
    (
        "CBSA_N_PRIOR_POSTERS",
        "Market-wide CBSA hospitals posting before the focal hospital's first observed posting cohort.",
    ),
    (
        "SAMPLE_COHORT_COUNTY",
        "Hospital first-post exact-code observation with a valid county and price.",
    ),
    (
        "SAMPLE_COHORT_COUNTY_PREFERRED",
        "First-post county exact-code sample with at least 5 payer cells and 3 distinct payers.",
    ),
    (
        "SAMPLE_COHORT_COUNTY_STRONG",
        "First-post county exact-code sample with at least 10 payer cells and 5 distinct payers.",
    ),
    # PREFERRED_CONCEPT_SUPPORT_FLAG / STRONG_CONCEPT_SUPPORT_FLAG removed --
    # these flags do not exist in this pipeline's concept table (see cell 14).
    *[
        (
            column,
            instrument_dictionary.loc[
                instrument_dictionary[
                    "column_name"
                ] == column,
                "definition",
            ].iloc[0],
        )
        for column
        in CANONICAL_INSTRUMENTS.values()
    ],
    (
        "LEGACY_FINE_ROLL_6M",
        "Earlier county-level six-month CMS fine measure retained for comparability; not the preferred health-system instrument.",
    ),
]

variable_dictionary = pd.DataFrame(
    variable_dictionary_rows,
    columns=[
        "variable",
        "definition",
    ],
).drop_duplicates("variable")

variable_dictionary.to_csv(
    QA_DIR / "HPT_ANALYSIS_VARIABLE_DICTIONARY.csv",
    index=False,
)

# Preserve the Snowflake-side QA and reference tables alongside the Python QA
# output, so a completed run carries documentation of both stages. These are
# the tables loaded in Section 2.
phase2_qa.to_csv(
    QA_DIR / "HPT_SOURCE_PHASE2_QA_SUMMARY.csv",
    index=False,
)
phase3_qa.to_csv(
    QA_DIR / "HPT_SOURCE_PHASE3_QA_SUMMARY.csv",
    index=False,
)
phase4_qa.to_csv(
    QA_DIR / "HPT_SOURCE_PHASE4_QA_SUMMARY.csv",
    index=False,
)
codebook.to_csv(
    QA_DIR / "HPT_SOURCE_CODEBOOK.csv",
    index=False,
)
hospital_directory.to_csv(
    QA_DIR / "HPT_SOURCE_HOSPITAL_DIRECTORY.csv",
    index=False,
)

msg("\nPython pipeline completed successfully")
msg(f"  QA checks: {len(qa):,}")
msg(f"  FAIL: {n_failures:,}")
msg(
    "  REVIEW: "
    f"{int((qa['status'] == 'REVIEW').sum()):,}"
)
msg(f"  Output folder: {OUTPUT_DIR}")


In [ ]:
# =============================================================================
# 16. FINAL RUN SUMMARY
# =============================================================================

print(
    "\nThe new Snowflake-output Python pipeline is complete.\n"
    "\nPrimary R-ready files:\n"
    "  - HPT_R_MASTER_OUTPATIENT_EXACT_CODE_ALL_GEOGRAPHIES.parquet\n"
    "  - HPT_R_MASTER_INPATIENT_DRG_ALL_GEOGRAPHIES.parquet\n"
    "  - HPT_R_MASTER_OUTPATIENT_CONCEPT_ALL_GEOGRAPHIES.parquet\n"
    "  - HPT_R_MASTER_INPATIENT_CONCEPT_ALL_GEOGRAPHIES.parquet\n"
    "  - HPT_R_MASTER_SERVICE_FAMILY_ALL_GEOGRAPHIES.parquet\n"
    "\nPreferred analytical unit:\n"
    "  hospital × observed posting cohort × reviewed Phase 4 concept\n"
    "\nPreferred treatment:\n"
    "  N_PRIOR_POSTERS_COUNTY_CONCEPT\n"
    "\nPreferred instrument:\n"
    "  Z_SYS_STRICT_9M_EXCL_CURRENT\n"
    "\nAll outputs are stored under:\n"
    f"  {OUTPUT_DIR}\n"
)
